<div dir="rtl" align="right">

# NB15 — ساخت و ارزیابی Sparse Index فارسی

## هدف Notebook

هدف این Notebook ساختن یک Sparse Index نسخه‌دار، قابل‌اندازه‌گیری، قابل‌بازسازی و قابل‌تحویل برای داده‌های فریز‌شده Phase 2 است.

Dense Retrieval مفاهیم مشابه را به‌خوبی بازیابی می‌کند، اما ممکن است در بازیابی عبارت‌های دقیق، نام اشخاص، اصطلاحات مالی، اعداد، مخفف‌ها و ترکیب‌های کم‌تکرار ضعف داشته باشد. Sparse Retrieval باید این فاصله را با تطبیق واژگانی کنترل‌شده پوشش دهد.

در این Notebook یک Analyzer فارسی بدون انتخاب حدسی طراحی و آزمایش می‌شود. سیاست‌های Normalization، Tokenization، نیم‌فاصله، ارقام، علائم، Stopword و شکل‌های عربی/فارسی باید نسخه‌دار باشند و اثر آن‌ها روی داده واقعی پروژه اندازه‌گیری شود.

## جایگاه در Pipeline

ترتیب مراحل مرتبط Retrieval به این صورت است:

- `NB13`: انتخاب و Freeze مدل Embedding؛
- `NB14`: ساخت و Freeze Dense Index؛
- `NB15`: ساخت و Freeze Sparse Index؛
- `NB16`: ترکیب Dense و Sparse در Hybrid Retrieval؛
- `NB17`: ارزیابی و انتخاب Reranker.

NB15 نباید Dense و Sparse را ترکیب کند و نباید وارد Reranking شود.

## ورودی‌های اصلی

ورودی‌ها باید از Artifactهای واقعی و فریز‌شده مخزن کشف و اعتبارسنجی شوند:

- Canonical Chunk Release؛
- Golden Dataset و Splitهای مجاز ارزیابی؛
- Schema، Lineage و Metadataهای هر Chunk؛
- Taxonomy و Knowledge Authority موردنیاز؛
- Release و Acceptance نهایی NB14 برای کنترل سازگاری نسخه‌های Pipeline.

هیچ Path، Count، Hash یا Status پیش از مشاهده Evidence واقعی پذیرفته نمی‌شود.

## خروجی‌های مورد انتظار

Notebook باید حداقل این خروجی‌ها را تولید کند:

- Persian Analyzer نسخه‌دار؛
- سیاست رسمی Tokenization و Normalization واژگانی؛
- Candidateهای قابل‌مقایسه Sparse/BM25؛
- Sparse Index منتخب؛
- Vocabulary و Document Frequency report؛
- Benchmark مستقل Sparse؛
- نتایج Sliceهای Website، Book و Transcript؛
- آزمون نام‌ها، اصطلاحات دقیق و اعداد؛
- Error Analysis و Failure Taxonomy؛
- ADR انتخاب Analyzer و تنظیمات BM25؛
- Rebuild و Idempotency evidence؛
- Sparse Index Manifest؛
- Acceptance Report؛
- Handoff نسخه‌دار برای NB16.

## Scope

این Notebook شامل طراحی Analyzer، ساخت Sparse Index، Benchmark مستقل، تحلیل خطا، انتخاب Config، Validation، Rebuild، Freeze و Handoff است.

## خارج از Scope

موارد زیر عمداً در NB15 انجام نمی‌شوند:

- Dense/Sparse Fusion؛
- RRF یا Weighted Fusion؛
- Hybrid Retrieval؛
- Reranking؛
- Context Building؛
- Generation؛
- Citation و Answerability.

## سیاست Golden Dataset

- تنظیم Analyzer و BM25 فقط با بخش مجاز Development انجام می‌شود.
- Test نهایی نباید برای تنظیم دستی یا انتخاب Candidate استفاده شود.
- Gold locatorها و Evidenceها باید پیش از Benchmark اعتبارسنجی شوند.
- هیچ نتیجه‌ای بدون Metric واقعی و Evidence ثبت‌شده، PASS اعلام نمی‌شود.

## سیاست هزینه و اجرا

این Notebook نباید API پولی اجرا کند. عملیات باید محلی، Artifact-first، Cache-first، Checkpoint-first و قابل اجرای مجدد باشند.

## معیار موفقیت نهایی

NB15 فقط زمانی موفق است که:

- تمام ورودی‌های Canonical با Hash و Status معتبر شناسایی شوند؛
- همه Chunkهای واجد شرایط وارد Sparse Index شوند؛
- Analyzer فارسی با Benchmark انتخاب شود؛
- Sparse Retrieval روی Golden Dataset ارزیابی شود؛
- Recall، MRR، nDCG، Hit Rate و latency ثبت شوند؛
- نام‌ها، اصطلاحات، اعداد و هر سه Source type تحلیل شوند؛
- Failureها مستند و طبقه‌بندی شوند؛
- Config منتخب با ADR ثبت شود؛
- Rebuild و Idempotency پاس شوند؛
- Manifest، Acceptance و Handoff ساخته شوند؛
- `Restart Kernel → Run All` بدون State پنهان موفق شود؛
- وضعیت نهایی برابر `PASS_COMPLETE_FROZEN` باشد.

</div>

<div dir="rtl" align="right">

## مرحله ۱ — Bootstrap و کشف وضعیت واقعی پروژه

NB14 با وضعیت `PASS_COMPLETE_FROZEN` پایان یافته و Dense Index آن نباید در NB15 تغییر کند.

نقطه شروع NB15، کشف و اعتبارسنجی Artifactهای واقعی مخزن است. پیش از طراحی Analyzer یا ساخت Sparse Index باید مشخص شود:

- Repository Root واقعی کجاست؛
- Notebook در مسیر رسمی قرار دارد یا نه؛
- چه Releaseها، Handoffها، Manifestها و Acceptanceهایی موجودند؛
- Chunk Release و Golden Dataset از چه مسیرهایی قابل دریافت‌اند؛
- خروجی‌های NB14 واقعاً در مخزن حضور دارند یا نه؛
- نسخه‌های Python و Dependencyهای پایه چیست؛
- آیا Artifactهای لازم برای ادامه وجود دارند.

در این مرحله هیچ Dataset پردازش نمی‌شود، هیچ Index ساخته نمی‌شود و هیچ Artifact موجودی تغییر نمی‌کند.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- Repository Root بدون حدس پیدا شود؛
- مسیر رسمی NB15 وجود داشته باشد؛
- پوشه‌های اصلی Phase 2 شناسایی شوند؛
- Artifactهای مرتبط با Chunk، Golden Dataset، Dense Index و Handoff فهرست شوند؛
- Hash و اندازه فایل‌های کشف‌شده ثبت شود؛
- هیچ API پولی اجرا نشود.

خروجی این مرحله مبنای Input Contract رسمی NB15 خواهد بود.

</div>

In [1]:
# Code Cell 01 — Centralized imports and bootstrap dependencies

from __future__ import annotations

import ast
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import statistics
import sys
import time
import unicodedata
import uuid

from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Iterator, Sequence

import numpy as np
import pandas as pd

from scipy import sparse

In [2]:
# Code Cell 02 — Repository discovery and read-only artifact preflight

NOTEBOOK_RELATIVE_PATH = Path(
    "notebooks/phase2/15_sparse_index.ipynb"
)

SPARSE_INDEX_RELEASE_ID = (
    "rag_finance_phase2_sparse_v1"
)

REPO_ROOT_OVERRIDE = None


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def discover_repo_root() -> tuple[Path, list[str]]:
    candidates: list[Path] = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(
            Path(REPO_ROOT_OVERRIDE).expanduser()
        )

    environment_root = os.environ.get(
        "RAG_FINANCE_REPO_ROOT"
    )

    if environment_root:
        candidates.append(
            Path(environment_root).expanduser()
        )

    current_path = Path.cwd().resolve()

    candidates.extend(
        [
            current_path,
            *current_path.parents,
            Path("/content/RAG_finance"),
            Path(
                "/content/drive/MyDrive/RAG_finance"
            ),
        ]
    )

    my_drive = Path("/content/drive/MyDrive")

    if my_drive.is_dir():
        candidates.extend(
            sorted(
                path
                for path in my_drive.iterdir()
                if (
                    path.is_dir()
                    and "rag" in path.name.lower()
                )
            )
        )

    unique_candidates: list[Path] = []
    checked_candidates: list[str] = []

    for candidate in candidates:
        try:
            resolved_candidate = candidate.resolve()
        except OSError:
            continue

        if resolved_candidate in unique_candidates:
            continue

        unique_candidates.append(
            resolved_candidate
        )

    for candidate in unique_candidates:
        checked_candidates.append(str(candidate))

        expected_notebook = (
            candidate / NOTEBOOK_RELATIVE_PATH
        )

        if expected_notebook.is_file():
            return candidate, checked_candidates

    raise FileNotFoundError(
        "Repository root was not found. "
        "Set REPO_ROOT_OVERRIDE to the exact "
        "repository path and rerun Code Cell 02. "
        f"Checked candidates: {checked_candidates}"
    )


repo_root, checked_repo_candidates = (
    discover_repo_root()
)

notebook_path = (
    repo_root / NOTEBOOK_RELATIVE_PATH
)

artifacts_phase2_root = (
    repo_root / "artifacts/phase2"
)
data_phase2_root = (
    repo_root / "data/phase2"
)
reports_phase2_root = (
    repo_root / "reports/phase2"
)
schemas_root = repo_root / "schemas"

assert notebook_path.is_file(), notebook_path
assert artifacts_phase2_root.is_dir(), (
    artifacts_phase2_root
)
assert data_phase2_root.is_dir(), (
    data_phase2_root
)
assert reports_phase2_root.is_dir(), (
    reports_phase2_root
)
assert sys.version_info >= (3, 10)

relevant_name_tokens = {
    "acceptance",
    "chunk",
    "dataset",
    "dense",
    "embedding",
    "golden",
    "handoff",
    "index",
    "manifest",
    "release",
    "schema",
    "split",
    "taxonomy",
}

allowed_suffixes = {
    ".json",
    ".jsonl",
    ".yaml",
    ".yml",
}

search_roots = [
    artifacts_phase2_root,
    data_phase2_root,
    reports_phase2_root,
]

if schemas_root.is_dir():
    search_roots.append(schemas_root)

discovered_artifacts = []

for search_root in search_roots:
    for artifact_path in search_root.rglob("*"):
        if not artifact_path.is_file():
            continue

        if artifact_path.suffix.lower() not in (
            allowed_suffixes
        ):
            continue

        normalized_name = (
            artifact_path.name.lower()
        )

        if not any(
            token in normalized_name
            for token in relevant_name_tokens
        ):
            continue

        discovered_artifacts.append(
            {
                "path": str(
                    artifact_path.relative_to(
                        repo_root
                    )
                ),
                "suffix": (
                    artifact_path.suffix.lower()
                ),
                "size_bytes": (
                    artifact_path.stat().st_size
                ),
                "sha256": sha256_file(
                    artifact_path
                ),
            }
        )

discovered_artifacts = sorted(
    {
        item["path"]: item
        for item in discovered_artifacts
    }.values(),
    key=lambda item: item["path"],
)

assert discovered_artifacts, (
    "No relevant Phase 2 artifacts were found."
)

nb14_required_names = {
    "dense_index_manifest.json",
    "dense_index_release_v1.json",
    "dense_index_acceptance_v1.json",
    "nb16_dense_index_handoff_v1.json",
    "index_lifecycle_evidence.json",
}

discovered_file_names = {
    Path(item["path"]).name
    for item in discovered_artifacts
}

missing_nb14_artifacts = sorted(
    nb14_required_names
    - discovered_file_names
)

chunk_candidates = [
    item
    for item in discovered_artifacts
    if "chunk" in item["path"].lower()
]

golden_candidates = [
    item
    for item in discovered_artifacts
    if "golden" in item["path"].lower()
]

release_candidates = [
    item
    for item in discovered_artifacts
    if "release" in item["path"].lower()
]

handoff_candidates = [
    item
    for item in discovered_artifacts
    if "handoff" in item["path"].lower()
]

environment_packages = {}

for package_name in (
    "numpy",
    "pandas",
    "scipy",
):
    environment_packages[package_name] = (
        importlib.metadata.version(
            package_name
        )
    )

preflight_status = (
    "PASS"
    if (
        not missing_nb14_artifacts
        and chunk_candidates
        and golden_candidates
        and release_candidates
        and handoff_candidates
    )
    else "REVIEW_REQUIRED"
)

preflight_inventory = {
    "notebook": str(
        notebook_path.relative_to(repo_root)
    ),
    "repo_root": str(repo_root),
    "checked_repo_candidates": (
        checked_repo_candidates
    ),
    "python_version": (
        platform.python_version()
    ),
    "platform": platform.platform(),
    "packages": environment_packages,
    "sparse_index_release_id": (
        SPARSE_INDEX_RELEASE_ID
    ),
    "discovered_artifact_count": len(
        discovered_artifacts
    ),
    "chunk_candidate_count": len(
        chunk_candidates
    ),
    "golden_candidate_count": len(
        golden_candidates
    ),
    "release_candidate_count": len(
        release_candidates
    ),
    "handoff_candidate_count": len(
        handoff_candidates
    ),
    "missing_required_nb14_artifacts": (
        missing_nb14_artifacts
    ),
    "discovered_artifacts": (
        discovered_artifacts
    ),
    "paid_api_calls": 0,
    "files_created": 0,
    "repository_mutated": False,
    "preflight_status": preflight_status,
}

print(
    json.dumps(
        preflight_inventory,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "notebook": "notebooks/phase2/15_sparse_index.ipynb",
  "repo_root": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance",
  "checked_repo_candidates": [
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2",
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks",
    "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
  ],
  "python_version": "3.11.3",
  "platform": "macOS-12.7.6-x86_64-i386-64bit",
  "packages": {
    "numpy": "1.24.3",
    "pandas": "1.5.3",
    "scipy": "1.13.1"
  },
  "sparse_index_release_id": "rag_finance_phase2_sparse_v1",
  "discovered_artifact_count": 8238,
  "chunk_candidate_count": 6,
  "golden_candidate_count": 8189,
  "release_candidate_count": 2,
  "handoff_candidate_count": 5,
  "missing_required_nb14_artifacts": [],
  "discovered_artifacts": [
    {
      "path": "artifacts/phase2/chunking/chunk_contract_evaluation_freeze_v1.

<div dir="rtl" align="right">

## مرحله ۲ — تعیین Canonical Input Contract

کشف اولیه نشان داد مخزن علاوه بر Artifactهای نهایی، تعداد زیادی فایل میانی، Pilot، Candidate، Repair و Work Directory دارد. بنابراین انتخاب ورودی براساس شباهت نام فایل می‌تواند باعث استفاده از Dataset تاریخی یا غیرنهایی شود.

در این مرحله فقط Artifactهای مرجع و فریز‌شده بارگذاری می‌شوند:

- Chunk Contract نهایی؛
- Golden Dataset Manifest و Acceptance نهایی؛
- Embedding Release و Handoff رسمی NB14؛
- Dense Index Manifest، Release و Acceptance نهایی؛
- Handoff آماده‌شده برای NB16؛
- Lifecycle Evidence مربوط به NB14.

ساختار واقعی هر Artifact، Statusها، Countها، Hashها و ارجاع‌های داخلی استخراج می‌شوند. سپس هر SHA ارجاع‌شده در Artifactهای موجود مخزن جست‌وجو می‌شود تا مشخص شود کدام فایل دقیقاً ورودی Canonical NB15 است.

در این مرحله هنوز هیچ Path یا Schema داخلی حدس زده نمی‌شود و هیچ فایل جدیدی ساخته نخواهد شد.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- تمام Artifactهای مرجع وجود داشته و JSON معتبر باشند؛
- وضعیت نهایی NB14 مستقیماً از فایل‌های ذخیره‌شده تأیید شود؛
- SHA مربوط به Chunk Release و Golden Dataset از قراردادهای فریز‌شده استخراج شود؛
- هر SHA Canonical به یک فایل واقعی در مخزن Resolve شود؛
- هیچ Artifact تاریخی یا Work-in-progress به‌عنوان ورودی نهایی انتخاب نشود؛
- Input Contract اولیه بدون تغییر مخزن ساخته شود.

</div>

In [3]:
# Code Cell 03 — Resolve and validate the canonical NB15 input contract

canonical_reference_paths = {
    "chunk_contract": (
        repo_root
        / "artifacts/phase2/chunking/"
        "chunk_contract_evaluation_freeze_v1.json"
    ),
    "golden_manifest": (
        repo_root
        / "artifacts/phase2/golden_dataset_v2/"
        "golden_dataset_v2_manifest.json"
    ),
    "golden_final_report": (
        repo_root
        / "artifacts/phase2/golden_dataset_v2/"
        "golden_dataset_v2_final_report.json"
    ),
    "golden_adjudicated_experiment_acceptance": (
        repo_root
        / "reports/phase2/"
        "golden_dataset_final_adjudicated_acceptance_v1.json"
    ),
    "embedding_release": (
        repo_root
        / "artifacts/phase2/embedding/"
        "embedding_release_v1.json"
    ),
    "nb14_embedding_handoff": (
        repo_root
        / "artifacts/phase2/embedding/"
        "nb14_dense_index_handoff_v1.json"
    ),
    "dense_manifest": (
        repo_root
        / "data/phase2/index/dense_index_v1/"
        "dense_index_manifest.json"
    ),
    "dense_lifecycle_evidence": (
        repo_root
        / "data/phase2/index/dense_index_v1/"
        "index_lifecycle_evidence.json"
    ),
    "dense_release": (
        repo_root
        / "artifacts/phase2/dense_index/"
        "dense_index_release_v1.json"
    ),
    "dense_acceptance": (
        repo_root
        / "reports/phase2/"
        "dense_index_acceptance_v1.json"
    ),
    "nb16_dense_handoff": (
        repo_root
        / "artifacts/phase2/dense_index/"
        "nb16_dense_index_handoff_v1.json"
    ),
}

missing_reference_paths = {
    name: str(path)
    for name, path
    in canonical_reference_paths.items()
    if not path.is_file()
}

assert not missing_reference_paths, (
    missing_reference_paths
)


def load_json_object(
    path: Path,
) -> dict[str, Any]:
    payload = json.loads(
        path.read_text(encoding="utf-8")
    )

    assert isinstance(payload, dict), (
        f"Expected JSON object: {path}"
    )

    return payload


reference_payloads = {
    name: load_json_object(path)
    for name, path
    in canonical_reference_paths.items()
}

chunk_contract = reference_payloads[
    "chunk_contract"
]
golden_manifest = reference_payloads[
    "golden_manifest"
]
golden_final_report = reference_payloads[
    "golden_final_report"
]
golden_adjudicated_acceptance = (
    reference_payloads[
        "golden_adjudicated_experiment_acceptance"
    ]
)
embedding_release = reference_payloads[
    "embedding_release"
]
nb14_embedding_handoff = reference_payloads[
    "nb14_embedding_handoff"
]
dense_manifest = reference_payloads[
    "dense_manifest"
]
dense_lifecycle_evidence = reference_payloads[
    "dense_lifecycle_evidence"
]
dense_release = reference_payloads[
    "dense_release"
]
dense_acceptance = reference_payloads[
    "dense_acceptance"
]
nb16_dense_handoff = reference_payloads[
    "nb16_dense_handoff"
]

# Resolve the exact canonical Chunk file through
# the frozen embedding release and NB14 handoff.
release_chunk_contract = (
    embedding_release["chunking"]
)
handoff_chunk_contract = (
    nb14_embedding_handoff["chunk_input"]
)

assert (
    release_chunk_contract["path"]
    == handoff_chunk_contract["path"]
)
assert (
    release_chunk_contract["sha256"]
    == handoff_chunk_contract["sha256"]
)
assert (
    release_chunk_contract["record_count"]
    == handoff_chunk_contract["record_count"]
)

canonical_chunk_relative_path = Path(
    release_chunk_contract["path"]
)
canonical_chunk_path = (
    repo_root / canonical_chunk_relative_path
)

assert canonical_chunk_path.is_file(), (
    canonical_chunk_path
)

canonical_chunk_file_sha256 = sha256_file(
    canonical_chunk_path
)

assert canonical_chunk_file_sha256 == (
    release_chunk_contract["sha256"]
)

canonical_chunk_count_contract = int(
    release_chunk_contract["record_count"]
)

# Resolve the exact frozen Golden Dataset through
# its final report, manifest and downstream handoffs.
canonical_golden_relative_path = Path(
    golden_final_report["dataset_path"]
)
canonical_golden_path = (
    repo_root / canonical_golden_relative_path
)

assert canonical_golden_path.is_file(), (
    canonical_golden_path
)

canonical_golden_file_sha256 = sha256_file(
    canonical_golden_path
)

assert canonical_golden_file_sha256 == (
    golden_manifest["dataset_file_sha256"]
)
assert canonical_golden_file_sha256 == (
    golden_final_report["hashes"][
        "dataset_file_sha256"
    ]
)
assert canonical_golden_file_sha256 == (
    nb14_embedding_handoff[
        "golden_dataset_sha256"
    ]
)
assert canonical_golden_file_sha256 == (
    dense_manifest["inputs"][
        "golden_dataset_sha256"
    ]
)
assert canonical_golden_file_sha256 == (
    dense_release["inputs"][
        "golden_dataset_sha256"
    ]
)
assert canonical_golden_file_sha256 == (
    nb16_dense_handoff[
        "golden_dataset_sha256"
    ]
)

canonical_golden_record_count_contract = int(
    golden_manifest["record_count"]
)

assert canonical_golden_record_count_contract == int(
    golden_final_report["record_count"]
)
assert canonical_golden_record_count_contract == int(
    embedding_release["golden_dataset"][
        "record_count"
    ]
)

# Validate downstream Chunk binding.
assert canonical_chunk_file_sha256 == (
    dense_manifest["inputs"]["chunk_sha256"]
)
assert canonical_chunk_file_sha256 == (
    dense_release["inputs"]["chunk_sha256"]
)
assert canonical_chunk_file_sha256 == (
    nb16_dense_handoff["chunk_sha256"]
)

assert canonical_chunk_count_contract == int(
    dense_manifest["inputs"]["chunk_count"]
)
assert canonical_chunk_count_contract == int(
    dense_manifest["index_content"][
        "point_count"
    ]
)
assert canonical_chunk_count_contract == int(
    dense_release["point_count"]
)
assert canonical_chunk_count_contract == int(
    nb16_dense_handoff["point_count"]
)

# Validate persisted NB14 completion.
assert dense_release["status"] == (
    "FROZEN_FOR_DOWNSTREAM_RETRIEVAL"
)
assert dense_release["final_status"] == (
    "PASS_COMPLETE_FROZEN"
)
assert dense_acceptance["status"] == "PASS"
assert dense_acceptance["final_status"] == (
    "PASS_COMPLETE_FROZEN"
)
assert dense_acceptance["objective"][
    "modular_nb14_scope_achieved"
] is True
assert dense_acceptance["objective"][
    "p2_09_index_lifecycle_achieved"
] is True
assert dense_lifecycle_evidence["status"] == (
    "PASS"
)
assert nb16_dense_handoff["status"] == (
    "READY_FOR_NB16"
)

# Validate Golden Dataset freeze.
assert golden_manifest["freeze_status"] == (
    "FROZEN"
)
assert golden_final_report["freeze_status"] == (
    "FROZEN"
)
assert golden_final_report["status"] == "PASS"

# The 46-record adjudicated artifact is an accepted
# experiment, not the active dataset used by NB13/NB14.
assert golden_adjudicated_acceptance[
    "status"
] == "ACCEPTED_EXPERIMENT_NOT_ACTIVE_DATASET"
assert golden_adjudicated_acceptance[
    "active_dataset_created"
] is False
assert int(
    golden_adjudicated_acceptance[
        "record_count"
    ]
) != canonical_golden_record_count_contract

# These are content/inventory digests, not standalone
# file hashes, so they are recorded but not resolved
# as physical files.
internal_digest_contracts = {
    "golden_dataset_payload_sha256": (
        golden_manifest[
            "dataset_payload_sha256"
        ]
    ),
    "embedding_cache_inventory_sha256": (
        embedding_release["cache"][
            "inventory_sha256"
        ]
    ),
    "dense_storage_inventory_sha256": (
        dense_release[
            "storage_inventory_sha256"
        ]
    ),
}

assert all(
    isinstance(value, str)
    and re.fullmatch(r"[0-9a-f]{64}", value)
    for value in internal_digest_contracts.values()
)

canonical_input_contract = {
    "artifact_type": (
        "NB15_CANONICAL_INPUT_CONTRACT"
    ),
    "schema_version": "1.0",
    "sparse_index_release_id": (
        SPARSE_INDEX_RELEASE_ID
    ),
    "chunk_input": {
        "path": str(
            canonical_chunk_relative_path
        ),
        "sha256": (
            canonical_chunk_file_sha256
        ),
        "record_count": (
            canonical_chunk_count_contract
        ),
        "selected_by": [
            "embedding_release_v1",
            "nb14_dense_index_handoff_v1",
            "dense_index_release_v1",
        ],
    },
    "golden_dataset": {
        "path": str(
            canonical_golden_relative_path
        ),
        "file_sha256": (
            canonical_golden_file_sha256
        ),
        "payload_sha256": (
            golden_manifest[
                "dataset_payload_sha256"
            ]
        ),
        "record_count": (
            canonical_golden_record_count_contract
        ),
        "freeze_status": (
            golden_manifest["freeze_status"]
        ),
        "selected_by": [
            "golden_dataset_v2_manifest",
            "golden_dataset_v2_final_report",
            "embedding_release_v1",
            "dense_index_release_v1",
        ],
    },
    "dense_index_compatibility": {
        "release_path": str(
            canonical_reference_paths[
                "dense_release"
            ].relative_to(repo_root)
        ),
        "release_sha256": sha256_file(
            canonical_reference_paths[
                "dense_release"
            ]
        ),
        "release_status": (
            dense_release["status"]
        ),
        "final_status": (
            dense_release["final_status"]
        ),
        "point_count": (
            dense_release["point_count"]
        ),
        "logical_index_sha256": (
            dense_release[
                "logical_index_sha256"
            ]
        ),
        "mutation_allowed": False,
    },
    "excluded_noncanonical_input": {
        "path": str(
            canonical_reference_paths[
                "golden_adjudicated_experiment_acceptance"
            ].relative_to(repo_root)
        ),
        "status": (
            golden_adjudicated_acceptance[
                "status"
            ]
        ),
        "record_count": (
            golden_adjudicated_acceptance[
                "record_count"
            ]
        ),
        "reason": (
            "Accepted experiment explicitly marked "
            "as not being the active dataset."
        ),
    },
    "internal_digest_contracts": (
        internal_digest_contracts
    ),
    "paid_api_calls": 0,
    "files_created": 0,
    "repository_mutated": False,
    "input_contract_status": "PASS",
}

print(
    json.dumps(
        canonical_input_contract,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "artifact_type": "NB15_CANONICAL_INPUT_CONTRACT",
  "schema_version": "1.0",
  "sparse_index_release_id": "rag_finance_phase2_sparse_v1",
  "chunk_input": {
    "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
    "sha256": "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
    "record_count": 3488,
    "selected_by": [
      "embedding_release_v1",
      "nb14_dense_index_handoff_v1",
      "dense_index_release_v1"
    ]
  },
  "golden_dataset": {
    "path": "artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl",
    "file_sha256": "b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2",
    "payload_sha256": "539635b4c89e8f4eda41dde8106e04307c063f3fe0f5db402ef0440802233a03",
    "record_count": 3813,
    "freeze_status": "FROZEN",
    "selected_by": [
      "golden_dataset_v2_manifest",
      "golden_dataset_v2_final_report",
      "embedding_release_v1",
      "dense_index_release_v1"
    ]
  },
  "dense_index_

<div dir="rtl" align="right">

## مرحله ۳ — بارگذاری و ممیزی ساختاری Chunkها و Golden Dataset

پس از تثبیت Input Contract، باید ساختار واقعی دو ورودی اصلی بررسی شود. طراحی Analyzer، Benchmark و معیارهای بازیابی بدون شناخت دقیق Schema می‌تواند باعث انتخاب فیلد اشتباه، Leakage یا ارزیابی نادرست شود.

در این مرحله:

- تمام Chunkهای Canonical بارگذاری می‌شوند؛
- تمام رکوردهای Golden Dataset بارگذاری می‌شوند؛
- تعداد واقعی رکوردها با Contract تطبیق داده می‌شود؛
- کلیدهای موجود و میزان پوشش هر کلید محاسبه می‌شود؛
- فیلدهای احتمالی ID، متن، Query، Split، Source و Evidence شناسایی می‌شوند؛
- نوع داده فیلدها بررسی می‌شود؛
- توزیع Source type و Split، در صورت وجود، گزارش می‌شود؛
- تهی‌بودن، Duplicate و ناسازگاری اولیه شناسه‌ها بررسی می‌شود؛
- فقط ساختار و آمار نمایش داده می‌شود و متن کامل داده‌ها چاپ نخواهد شد.

هنوز هیچ فیلدی به‌صورت حدسی برای Index یا Benchmark انتخاب نمی‌شود. انتخاب رسمی فیلدها در مرحله بعد و براساس خروجی واقعی این ممیزی انجام خواهد شد.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- Hash فایل‌ها همچنان با Input Contract برابر باشد؛
- تعداد واقعی Chunkها و Golden recordها دقیقاً مطابق Contract باشد؛
- تمام خطوط JSONL معتبر و از نوع Object باشند؛
- Schema واقعی و Coverage فیلدها ثبت شود؛
- Candidate fieldهای لازم برای قرارداد بعدی قابل شناسایی باشند؛
- هیچ فایل یا Artifact جدیدی ساخته نشود.

</div>

In [4]:
# Code Cell 04 — Load and audit canonical Chunk and Golden JSONL schemas

def load_jsonl_objects(
    path: Path,
) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, raw_line in enumerate(
            handle,
            start=1,
        ):
            stripped_line = raw_line.strip()

            if not stripped_line:
                continue

            try:
                record = json.loads(
                    stripped_line
                )
            except json.JSONDecodeError as exc:
                raise AssertionError(
                    f"Invalid JSONL at {path}, "
                    f"line {line_number}: {exc}"
                ) from exc

            assert isinstance(record, dict), (
                f"Expected object at {path}, "
                f"line {line_number}"
            )

            records.append(record)

    return records


def value_type_name(value: Any) -> str:
    if value is None:
        return "null"

    if isinstance(value, bool):
        return "boolean"

    if isinstance(value, int):
        return "integer"

    if isinstance(value, float):
        return "number"

    if isinstance(value, str):
        return "string"

    if isinstance(value, list):
        return "array"

    if isinstance(value, dict):
        return "object"

    return type(value).__name__


def build_schema_profile(
    records: Sequence[dict[str, Any]],
) -> dict[str, Any]:
    key_presence: Counter[str] = Counter()
    key_non_null: Counter[str] = Counter()
    key_non_empty: Counter[str] = Counter()
    key_types: dict[str, Counter[str]] = (
        defaultdict(Counter)
    )

    for record in records:
        for key, value in record.items():
            key_presence[key] += 1
            key_types[key][
                value_type_name(value)
            ] += 1

            if value is not None:
                key_non_null[key] += 1

            if (
                value is not None
                and value != ""
                and value != []
                and value != {}
            ):
                key_non_empty[key] += 1

    total_records = len(records)

    return {
        "record_count": total_records,
        "fields": {
            key: {
                "present_count": (
                    key_presence[key]
                ),
                "present_rate": (
                    key_presence[key]
                    / total_records
                ),
                "non_null_count": (
                    key_non_null[key]
                ),
                "non_empty_count": (
                    key_non_empty[key]
                ),
                "types": dict(
                    sorted(
                        key_types[key].items()
                    )
                ),
            }
            for key in sorted(key_presence)
        },
    }


def nested_object_key_profile(
    records: Sequence[dict[str, Any]],
) -> dict[str, Any]:
    nested_profile: dict[
        str,
        Counter[str],
    ] = defaultdict(Counter)

    for record in records:
        for parent_key, value in record.items():
            if not isinstance(value, dict):
                continue

            for child_key in value:
                nested_profile[parent_key][
                    child_key
                ] += 1

    return {
        parent_key: dict(
            sorted(child_counts.items())
        )
        for parent_key, child_counts
        in sorted(nested_profile.items())
    }


def candidate_field_names(
    records: Sequence[dict[str, Any]],
) -> dict[str, list[str]]:
    all_keys = sorted(
        {
            key
            for record in records
            for key in record
        }
    )

    return {
        "id_candidates": [
            key
            for key in all_keys
            if (
                key == "id"
                or key.endswith("_id")
                or key.endswith("_ids")
            )
        ],
        "text_candidates": [
            key
            for key in all_keys
            if any(
                token in key.lower()
                for token in (
                    "text",
                    "content",
                    "question",
                    "query",
                    "answer",
                    "claim",
                    "quote",
                )
            )
        ],
        "split_candidates": [
            key
            for key in all_keys
            if any(
                token in key.lower()
                for token in (
                    "split",
                    "fold",
                    "partition",
                    "holdout",
                )
            )
        ],
        "source_candidates": [
            key
            for key in all_keys
            if any(
                token in key.lower()
                for token in (
                    "source",
                    "document",
                    "speaker",
                    "author",
                )
            )
        ],
        "evidence_candidates": [
            key
            for key in all_keys
            if any(
                token in key.lower()
                for token in (
                    "evidence",
                    "locator",
                    "relevant",
                    "gold",
                    "chunk",
                )
            )
        ],
    }


def safe_distribution(
    records: Sequence[dict[str, Any]],
    field_name: str,
) -> dict[str, int]:
    distribution: Counter[str] = Counter()

    for record in records:
        value = record.get(field_name)

        if value is None:
            continue

        if isinstance(
            value,
            (str, int, float, bool),
        ):
            distribution[str(value)] += 1

    return dict(
        sorted(distribution.items())
    )


def identifier_profile(
    records: Sequence[dict[str, Any]],
    candidate_fields: Sequence[str],
) -> dict[str, Any]:
    profiles = {}

    for field_name in candidate_fields:
        values = [
            record.get(field_name)
            for record in records
            if record.get(field_name) is not None
        ]

        scalar_values = [
            value
            for value in values
            if isinstance(
                value,
                (str, int, float, bool),
            )
        ]

        if len(scalar_values) != len(values):
            continue

        normalized_values = [
            str(value)
            for value in scalar_values
        ]

        profiles[field_name] = {
            "value_count": len(
                normalized_values
            ),
            "unique_count": len(
                set(normalized_values)
            ),
            "duplicate_count": (
                len(normalized_values)
                - len(set(normalized_values))
            ),
            "empty_string_count": sum(
                value == ""
                for value in normalized_values
            ),
        }

    return profiles


assert sha256_file(
    canonical_chunk_path
) == canonical_input_contract[
    "chunk_input"
]["sha256"]

assert sha256_file(
    canonical_golden_path
) == canonical_input_contract[
    "golden_dataset"
]["file_sha256"]

chunk_records = load_jsonl_objects(
    canonical_chunk_path
)
golden_records = load_jsonl_objects(
    canonical_golden_path
)

assert len(chunk_records) == (
    canonical_input_contract[
        "chunk_input"
    ]["record_count"]
)

assert len(golden_records) == (
    canonical_input_contract[
        "golden_dataset"
    ]["record_count"]
)

chunk_schema_profile = build_schema_profile(
    chunk_records
)
golden_schema_profile = build_schema_profile(
    golden_records
)

chunk_candidate_fields = (
    candidate_field_names(
        chunk_records
    )
)
golden_candidate_fields = (
    candidate_field_names(
        golden_records
    )
)

chunk_identifier_profiles = (
    identifier_profile(
        chunk_records,
        chunk_candidate_fields[
            "id_candidates"
        ],
    )
)

golden_identifier_profiles = (
    identifier_profile(
        golden_records,
        golden_candidate_fields[
            "id_candidates"
        ],
    )
)

common_distribution_field_names = (
    "source_type",
    "split",
    "dataset_split",
    "evaluation_split",
    "intent",
    "difficulty",
    "query_type",
    "status",
)

chunk_distributions = {
    field_name: safe_distribution(
        chunk_records,
        field_name,
    )
    for field_name
    in common_distribution_field_names
    if any(
        field_name in record
        for record in chunk_records
    )
}

golden_distributions = {
    field_name: safe_distribution(
        golden_records,
        field_name,
    )
    for field_name
    in common_distribution_field_names
    if any(
        field_name in record
        for record in golden_records
    )
}

schema_audit = {
    "chunk_dataset": {
        "path": str(
            canonical_chunk_relative_path
        ),
        "sha256": sha256_file(
            canonical_chunk_path
        ),
        "record_count": len(
            chunk_records
        ),
        "schema": chunk_schema_profile,
        "nested_object_keys": (
            nested_object_key_profile(
                chunk_records
            )
        ),
        "candidate_fields": (
            chunk_candidate_fields
        ),
        "identifier_profiles": (
            chunk_identifier_profiles
        ),
        "distributions": (
            chunk_distributions
        ),
    },
    "golden_dataset": {
        "path": str(
            canonical_golden_relative_path
        ),
        "sha256": sha256_file(
            canonical_golden_path
        ),
        "record_count": len(
            golden_records
        ),
        "schema": golden_schema_profile,
        "nested_object_keys": (
            nested_object_key_profile(
                golden_records
            )
        ),
        "candidate_fields": (
            golden_candidate_fields
        ),
        "identifier_profiles": (
            golden_identifier_profiles
        ),
        "distributions": (
            golden_distributions
        ),
    },
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "schema_audit_status": "PASS",
}

print(
    json.dumps(
        schema_audit,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "chunk_dataset": {
    "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
    "sha256": "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
    "record_count": 3488,
    "schema": {
      "record_count": 3488,
      "fields": {
        "attribution_context": {
          "present_count": 3488,
          "present_rate": 1.0,
          "non_null_count": 3488,
          "non_empty_count": 3488,
          "types": {
            "object": 3488
          }
        },
        "boundary_metadata": {
          "present_count": 3488,
          "present_rate": 1.0,
          "non_null_count": 3488,
          "non_empty_count": 3488,
          "types": {
            "object": 3488
          }
        },
        "char_count": {
          "present_count": 3488,
          "present_rate": 1.0,
          "non_null_count": 3488,
          "non_empty_count": 3488,
          "types": {
            "integer": 3488
          }
        },
        "chunk_cont

<div dir="rtl" align="right">

## مرحله ۴ — تثبیت قرارداد Query، Evidence و Split

ساختار واقعی داده نشان داد:

- هر ۳۴۸۸ Chunk دارای `chunk_id` یکتا و `content` معتبر است؛
- Source typeها شامل Website، Book و Transcript هستند؛
- هر ۳۸۱۳ رکورد Golden دارای `candidate_id` و `claim_id` یکتا است؛
- Query واقعی در `evaluation_content.question` قرار دارد؛
- پاسخ مورد انتظار در `evaluation_content.expected_answer` قرار دارد؛
- فیلدهای سطح بالای `question`، `expected_answer` و `question_intent` خالی هستند؛
- Golden Dataset دارای Split مستقیم در سطح رکورد نیست.

پیش از ساخت Analyzer باید مشخص شود Gold relevance چگونه به Chunkهای Retrieval متصل می‌شود و NB13 برای Benchmark از چه قرارداد فعال، Mapping و Split استفاده کرده است.

در این مرحله:

- Query و Expected Answer از محل صحیح استخراج و اعتبارسنجی می‌شوند؛
- ساختار `evidence_ids`، `evidence_locators` و `evidence_quotes` بررسی می‌شود؛
- میزان اتصال مستقیم Evidence IDها به Chunk IDها، Document IDها و Source IDها اندازه‌گیری می‌شود؛
- وضعیت Validation و Generation تمام رکوردها بررسی می‌شود؛
- Root رسمی Benchmark منتخب NB13 از Embedding Release استخراج می‌شود؛
- Artifactهای Query profile، Metrics، Split و Handoff همان Benchmark شناسایی می‌شوند؛
- هیچ Split جدید یا Gold mapping حدسی ساخته نمی‌شود.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- تمام Queryهای واقعی و پاسخ‌های مورد انتظار معتبر باشند؛
- هیچ رکورد نامعتبر یا بدون Evidence وجود نداشته باشد؛
- نوع و ساختار Locatorها مشخص شود؛
- روش اتصال Golden Evidence به Retrieval Corpus قابل تعیین باشد؛
- Benchmark root رسمی NB13 بدون انتخاب دستی شناسایی شود؛
- Artifactهای لازم برای بازاستفاده از Split و Gold mapping مشخص شوند.

</div>

In [5]:
# Code Cell 05 — Audit Query/Evidence contracts and discover NB13 benchmark mapping

chunk_ids = [
    record["chunk_id"]
    for record in chunk_records
]

chunk_id_set = set(chunk_ids)

assert len(chunk_ids) == len(chunk_id_set)
assert all(
    isinstance(chunk_id, str)
    and chunk_id.strip()
    for chunk_id in chunk_ids
)

chunk_document_id_set = {
    document_id
    for record in chunk_records
    for document_id in (
        record.get("member_document_ids")
        or []
    )
    if isinstance(document_id, str)
    and document_id
}

chunk_source_id_set = {
    record["source_id"]
    for record in chunk_records
    if isinstance(
        record.get("source_id"),
        str,
    )
    and record["source_id"]
}

golden_queries = []
golden_expected_answers = []
golden_candidate_ids = []
golden_claim_ids = []

evidence_id_values = []
evidence_locator_values = []
evidence_quote_values = []

golden_source_type_counts: Counter[str] = (
    Counter()
)
golden_generation_status_counts: Counter[str] = (
    Counter()
)
golden_validation_status_counts: Counter[str] = (
    Counter()
)

invalid_evaluation_content = []
invalid_evidence_records = []
invalid_validation_records = []

evidence_locator_type_counts: Counter[str] = (
    Counter()
)
evidence_locator_key_counts: Counter[str] = (
    Counter()
)

for record_index, record in enumerate(
    golden_records
):
    evaluation_content = record.get(
        "evaluation_content"
    )

    if not isinstance(
        evaluation_content,
        dict,
    ):
        invalid_evaluation_content.append(
            {
                "record_index": record_index,
                "reason": (
                    "evaluation_content_not_object"
                ),
            }
        )
        continue

    question = evaluation_content.get(
        "question"
    )
    expected_answer = evaluation_content.get(
        "expected_answer"
    )

    if (
        not isinstance(question, str)
        or not question.strip()
        or not isinstance(
            expected_answer,
            str,
        )
        or not expected_answer.strip()
    ):
        invalid_evaluation_content.append(
            {
                "record_index": record_index,
                "candidate_id": record.get(
                    "candidate_id"
                ),
                "question_valid": (
                    isinstance(question, str)
                    and bool(question.strip())
                ),
                "expected_answer_valid": (
                    isinstance(
                        expected_answer,
                        str,
                    )
                    and bool(
                        expected_answer.strip()
                    )
                ),
            }
        )
        continue

    golden_queries.append(question)
    golden_expected_answers.append(
        expected_answer
    )
    golden_candidate_ids.append(
        record["candidate_id"]
    )
    golden_claim_ids.append(
        record["claim_id"]
    )

    evidence_ids = record.get(
        "evidence_ids"
    )
    evidence_locators = record.get(
        "evidence_locators"
    )
    evidence_quotes = record.get(
        "evidence_quotes"
    )

    if (
        not isinstance(evidence_ids, list)
        or not evidence_ids
        or not isinstance(
            evidence_locators,
            list,
        )
        or not evidence_locators
        or not isinstance(
            evidence_quotes,
            list,
        )
        or not evidence_quotes
    ):
        invalid_evidence_records.append(
            {
                "record_index": record_index,
                "candidate_id": record.get(
                    "candidate_id"
                ),
                "evidence_ids_valid": (
                    isinstance(
                        evidence_ids,
                        list,
                    )
                    and bool(evidence_ids)
                ),
                "evidence_locators_valid": (
                    isinstance(
                        evidence_locators,
                        list,
                    )
                    and bool(evidence_locators)
                ),
                "evidence_quotes_valid": (
                    isinstance(
                        evidence_quotes,
                        list,
                    )
                    and bool(evidence_quotes)
                ),
            }
        )
        continue

    evidence_id_values.extend(
        value
        for value in evidence_ids
        if isinstance(value, str)
        and value
    )

    evidence_locator_values.extend(
        evidence_locators
    )

    evidence_quote_values.extend(
        value
        for value in evidence_quotes
        if isinstance(value, str)
        and value.strip()
    )

    for locator in evidence_locators:
        locator_type = value_type_name(
            locator
        )
        evidence_locator_type_counts[
            locator_type
        ] += 1

        if isinstance(locator, dict):
            for key in locator:
                evidence_locator_key_counts[
                    key
                ] += 1

    for source_type in (
        record.get("source_types")
        or []
    ):
        if isinstance(source_type, str):
            golden_source_type_counts[
                source_type
            ] += 1

    generation = record.get("generation")
    validation = record.get("validation")

    if isinstance(generation, dict):
        golden_generation_status_counts[
            str(generation.get("status"))
        ] += 1
    else:
        golden_generation_status_counts[
            "MISSING_OR_INVALID"
        ] += 1

    if isinstance(validation, dict):
        validation_status = validation.get(
            "status"
        )

        golden_validation_status_counts[
            str(validation_status)
        ] += 1

        failure_reasons = validation.get(
            "failure_reasons"
        )

        if (
            validation_status not in (
                "PASS",
                "VALID",
            )
            or (
                isinstance(
                    failure_reasons,
                    list,
                )
                and failure_reasons
            )
        ):
            invalid_validation_records.append(
                {
                    "record_index": (
                        record_index
                    ),
                    "candidate_id": (
                        record.get(
                            "candidate_id"
                        )
                    ),
                    "status": (
                        validation_status
                    ),
                    "failure_reasons": (
                        failure_reasons
                    ),
                }
            )
    else:
        invalid_validation_records.append(
            {
                "record_index": record_index,
                "candidate_id": record.get(
                    "candidate_id"
                ),
                "status": (
                    "MISSING_OR_INVALID"
                ),
            }
        )

assert not invalid_evaluation_content, (
    invalid_evaluation_content[:20]
)
assert not invalid_evidence_records, (
    invalid_evidence_records[:20]
)

assert len(golden_queries) == len(
    golden_records
)
assert len(golden_expected_answers) == len(
    golden_records
)
assert len(golden_candidate_ids) == len(
    golden_records
)
assert len(set(golden_candidate_ids)) == len(
    golden_records
)
assert len(set(golden_claim_ids)) == len(
    golden_records
)

evidence_id_set = set(
    evidence_id_values
)

direct_chunk_id_overlap = (
    evidence_id_set & chunk_id_set
)
document_id_overlap = (
    evidence_id_set & chunk_document_id_set
)
source_id_overlap = (
    evidence_id_set & chunk_source_id_set
)

roadmap_metrics_relative_path = Path(
    embedding_release["evidence"][
        "roadmap_metrics_path"
    ]
)

roadmap_metrics_path = (
    repo_root / roadmap_metrics_relative_path
)

assert roadmap_metrics_path.is_file(), (
    roadmap_metrics_path
)

selected_benchmark_root = (
    roadmap_metrics_path.parent
)

assert selected_benchmark_root.is_dir()

benchmark_artifact_tokens = (
    "active",
    "benchmark",
    "golden",
    "handoff",
    "metric",
    "profile",
    "query",
    "split",
    "test",
    "dev",
)

selected_benchmark_artifacts = []

for artifact_path in sorted(
    selected_benchmark_root.iterdir()
):
    if not artifact_path.is_file():
        continue

    if artifact_path.suffix.lower() not in {
        ".json",
        ".jsonl",
    }:
        continue

    if not any(
        token in artifact_path.name.lower()
        for token in benchmark_artifact_tokens
    ):
        continue

    selected_benchmark_artifacts.append(
        {
            "path": str(
                artifact_path.relative_to(
                    repo_root
                )
            ),
            "size_bytes": (
                artifact_path.stat().st_size
            ),
            "sha256": sha256_file(
                artifact_path
            ),
        }
    )

assert selected_benchmark_artifacts

benchmark_json_structures = {}

for artifact in selected_benchmark_artifacts:
    artifact_path = (
        repo_root / artifact["path"]
    )

    if artifact_path.suffix.lower() != ".json":
        continue

    try:
        payload = json.loads(
            artifact_path.read_text(
                encoding="utf-8"
            )
        )
    except json.JSONDecodeError:
        continue

    if isinstance(payload, dict):
        benchmark_json_structures[
            artifact["path"]
        ] = {
            "root_type": "object",
            "top_level_keys": sorted(
                payload.keys()
            ),
        }
    elif isinstance(payload, list):
        benchmark_json_structures[
            artifact["path"]
        ] = {
            "root_type": "array",
            "item_count": len(payload),
            "first_item_keys": (
                sorted(payload[0].keys())
                if (
                    payload
                    and isinstance(
                        payload[0],
                        dict,
                    )
                )
                else []
            ),
        }

query_evidence_contract_audit = {
    "query_contract": {
        "field": (
            "evaluation_content.question"
        ),
        "record_count": len(
            golden_queries
        ),
        "non_empty_count": sum(
            bool(query.strip())
            for query in golden_queries
        ),
        "unique_query_count": len(
            set(golden_queries)
        ),
        "top_level_question_is_null_count": (
            sum(
                record.get("question") is None
                for record in golden_records
            )
        ),
    },
    "expected_answer_contract": {
        "field": (
            "evaluation_content.expected_answer"
        ),
        "record_count": len(
            golden_expected_answers
        ),
        "non_empty_count": sum(
            bool(answer.strip())
            for answer
            in golden_expected_answers
        ),
        "top_level_expected_answer_is_null_count": (
            sum(
                record.get(
                    "expected_answer"
                )
                is None
                for record in golden_records
            )
        ),
    },
    "evidence_contract": {
        "evidence_id_count": len(
            evidence_id_values
        ),
        "unique_evidence_id_count": len(
            evidence_id_set
        ),
        "evidence_locator_count": len(
            evidence_locator_values
        ),
        "evidence_quote_count": len(
            evidence_quote_values
        ),
        "locator_type_counts": dict(
            sorted(
                evidence_locator_type_counts.items()
            )
        ),
        "locator_key_counts": dict(
            sorted(
                evidence_locator_key_counts.items()
            )
        ),
        "direct_chunk_id_overlap_count": len(
            direct_chunk_id_overlap
        ),
        "document_id_overlap_count": len(
            document_id_overlap
        ),
        "source_id_overlap_count": len(
            source_id_overlap
        ),
    },
    "golden_source_type_counts": dict(
        sorted(
            golden_source_type_counts.items()
        )
    ),
    "generation_status_counts": dict(
        sorted(
            golden_generation_status_counts.items()
        )
    ),
    "validation_status_counts": dict(
        sorted(
            golden_validation_status_counts.items()
        )
    ),
    "invalid_validation_record_count": len(
        invalid_validation_records
    ),
    "invalid_validation_examples": (
        invalid_validation_records[:10]
    ),
    "selected_nb13_benchmark_root": str(
        selected_benchmark_root.relative_to(
            repo_root
        )
    ),
    "selected_nb13_benchmark_artifacts": (
        selected_benchmark_artifacts
    ),
    "benchmark_json_structures": (
        benchmark_json_structures
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "query_evidence_contract_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        query_evidence_contract_audit,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "query_contract": {
    "field": "evaluation_content.question",
    "record_count": 3813,
    "non_empty_count": 3813,
    "unique_query_count": 3735,
    "top_level_question_is_null_count": 3813
  },
  "expected_answer_contract": {
    "field": "evaluation_content.expected_answer",
    "record_count": 3813,
    "non_empty_count": 3813,
    "top_level_expected_answer_is_null_count": 3813
  },
  "evidence_contract": {
    "evidence_id_count": 3924,
    "unique_evidence_id_count": 3924,
    "evidence_locator_count": 3924,
    "evidence_quote_count": 3924,
    "locator_type_counts": {
      "object": 3924
    },
    "locator_key_counts": {
      "source_locator": 3924,
      "text_char_end": 3924,
      "text_char_start": 3924
    },
    "direct_chunk_id_overlap_count": 0,
    "document_id_overlap_count": 0,
    "source_id_overlap_count": 0
  },
  "golden_source_type_counts": {
    "book": 395,
    "transcript": 2089,
    "website": 1329
  },
  "generation_status_counts": {
    "PASS"

<div dir="rtl" align="right">

## مرحله ۵ — ممیزی Gold Mapping و قرارداد ارزیابی NB13

Evidenceهای Golden Dataset در سطح Claim و Source locator ثبت شده‌اند و مستقیماً با `chunk_id` برابر نیستند. NB13 برای محاسبه Metricهای Retrieval ناچار بوده است این Evidenceها را به Chunkهای قابل‌بازیابی نگاشت کند.

در این مرحله Artifact رسمی `active_golden_query_information_profile` و خروجی‌های Benchmark منتخب NB13 بررسی می‌شوند تا مشخص شود:

- هر Query با چه شناسه‌ای ثبت شده است؛
- Gold relevant Chunkها در کدام فیلد قرار دارند؛
- Queryهای قابل ارزیابی و غیرقابل ارزیابی چگونه مشخص شده‌اند؛
- چه سیاست IDF یا Information profile استفاده شده است؛
- آیا Split توسعه و تست در Artifactهای NB13 وجود دارد؛
- Queryهای تکراری چگونه نمایش داده شده‌اند؛
- Metricهای NB13 از چه Query count و Eligibility contract استفاده کرده‌اند؛
- چه محدودیت‌هایی در Handoff رسمی ثبت شده است.

این مرحله فقط قرارداد قبلی را ممیزی می‌کند. هنوز Split جدید، Analyzer یا Sparse Index ساخته نمی‌شود.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- SHA Golden Dataset در تمام Artifactهای NB13 با ورودی Canonical برابر باشد؛
- ساختار تمام Rowهای Query profile مشخص شود؛
- Gold chunk mapping رسمی شناسایی شود؛
- تعداد Queryهای Eligible و Ineligible معلوم شود؛
- وجود یا نبود Split رسمی با Evidence تعیین شود؛
- قرارداد قابل‌بازتولید Metricهای NB13 برای استفاده در NB15 مشخص شود.

</div>

In [6]:
# Code Cell 06 — Audit NB13 query profile, Gold mapping and evaluation contract

query_profile_path = (
    selected_benchmark_root
    / "active_golden_query_information_profile.json"
)

benchmark_manifest_path = (
    selected_benchmark_root
    / "benchmark_manifest.json"
)

roadmap_metrics_path = (
    selected_benchmark_root
    / "embedding_benchmark_roadmap_metrics.json"
)

reproducibility_handoff_path = (
    selected_benchmark_root
    / "nb13_reproducibility_handoff.json"
)

required_nb13_paths = {
    "query_profile": query_profile_path,
    "benchmark_manifest": (
        benchmark_manifest_path
    ),
    "roadmap_metrics": roadmap_metrics_path,
    "reproducibility_handoff": (
        reproducibility_handoff_path
    ),
}

missing_nb13_paths = {
    name: str(path)
    for name, path
    in required_nb13_paths.items()
    if not path.is_file()
}

assert not missing_nb13_paths, (
    missing_nb13_paths
)

query_profile = load_json_object(
    query_profile_path
)
benchmark_manifest = load_json_object(
    benchmark_manifest_path
)
roadmap_metrics = load_json_object(
    roadmap_metrics_path
)
reproducibility_handoff = load_json_object(
    reproducibility_handoff_path
)

assert query_profile[
    "active_golden_dataset_sha256"
] == canonical_golden_file_sha256

assert reproducibility_handoff[
    "active_golden_dataset_sha256"
] == canonical_golden_file_sha256

profile_rows = query_profile.get("rows")

assert isinstance(profile_rows, list)
assert profile_rows
assert all(
    isinstance(row, dict)
    for row in profile_rows
)

profile_row_schema = build_schema_profile(
    profile_rows
)

profile_row_candidates = (
    candidate_field_names(
        profile_rows
    )
)

profile_nested_keys = (
    nested_object_key_profile(
        profile_rows
    )
)

profile_identifier_profiles = (
    identifier_profile(
        profile_rows,
        profile_row_candidates[
            "id_candidates"
        ],
    )
)

profile_split_candidates = (
    profile_row_candidates[
        "split_candidates"
    ]
)

profile_split_distributions = {
    field_name: safe_distribution(
        profile_rows,
        field_name,
    )
    for field_name
    in profile_split_candidates
}

candidate_id_fields = [
    field_name
    for field_name in (
        "candidate_id",
        "claim_id",
        "query_id",
        "source_seed_slot_id",
    )
    if any(
        field_name in row
        for row in profile_rows
    )
]

candidate_id_alignment = {}

golden_candidate_id_set = set(
    golden_candidate_ids
)
golden_claim_id_set = set(
    golden_claim_ids
)

for field_name in candidate_id_fields:
    field_values = {
        str(row[field_name])
        for row in profile_rows
        if row.get(field_name) is not None
    }

    candidate_id_alignment[field_name] = {
        "value_count": sum(
            row.get(field_name) is not None
            for row in profile_rows
        ),
        "unique_count": len(
            field_values
        ),
        "overlap_with_golden_candidate_ids": (
            len(
                field_values
                & golden_candidate_id_set
            )
        ),
        "overlap_with_golden_claim_ids": (
            len(
                field_values
                & golden_claim_id_set
            )
        ),
    }

mapping_field_tokens = (
    "chunk",
    "relevant",
    "gold",
    "positive",
    "eligible",
    "match",
    "retriev",
    "seed",
)

mapping_candidate_fields = sorted(
    {
        key
        for row in profile_rows
        for key in row
        if any(
            token in key.lower()
            for token in mapping_field_tokens
        )
    }
)

mapping_field_profiles = {}

for field_name in mapping_candidate_fields:
    type_counts: Counter[str] = Counter()
    present_count = 0
    non_empty_count = 0
    total_array_items = 0
    unique_scalar_values = set()

    for row in profile_rows:
        if field_name not in row:
            continue

        present_count += 1
        value = row[field_name]
        type_counts[
            value_type_name(value)
        ] += 1

        if (
            value is not None
            and value != ""
            and value != []
            and value != {}
        ):
            non_empty_count += 1

        if isinstance(value, list):
            total_array_items += len(value)

        if isinstance(
            value,
            (str, int, float, bool),
        ):
            unique_scalar_values.add(
                str(value)
            )

    mapping_field_profiles[field_name] = {
        "present_count": present_count,
        "non_empty_count": non_empty_count,
        "types": dict(
            sorted(type_counts.items())
        ),
        "total_array_items": (
            total_array_items
        ),
        "unique_scalar_value_count": (
            len(unique_scalar_values)
        ),
        "scalar_value_sample": sorted(
            unique_scalar_values
        )[:20],
    }

query_text_candidate_fields = [
    field_name
    for field_name in (
        "question",
        "query",
        "query_text",
        "text",
    )
    if any(
        field_name in row
        for row in profile_rows
    )
]

query_text_alignment = {}

for field_name in query_text_candidate_fields:
    profile_query_values = [
        row.get(field_name)
        for row in profile_rows
        if isinstance(
            row.get(field_name),
            str,
        )
        and row[field_name].strip()
    ]

    query_text_alignment[field_name] = {
        "value_count": len(
            profile_query_values
        ),
        "unique_count": len(
            set(profile_query_values)
        ),
        "overlap_with_canonical_queries": (
            len(
                set(profile_query_values)
                & set(golden_queries)
            )
        ),
    }


def summarize_nested_structure(
    value: Any,
    depth: int = 0,
    maximum_depth: int = 3,
) -> Any:
    if depth >= maximum_depth:
        if isinstance(value, list):
            return {
                "type": "array",
                "length": len(value),
            }

        if isinstance(value, dict):
            return {
                "type": "object",
                "key_count": len(value),
            }

        return {
            "type": value_type_name(value)
        }

    if isinstance(value, dict):
        return {
            key: summarize_nested_structure(
                child,
                depth + 1,
                maximum_depth,
            )
            for key, child in sorted(
                value.items()
            )
        }

    if isinstance(value, list):
        return {
            "type": "array",
            "length": len(value),
            "first_item": (
                summarize_nested_structure(
                    value[0],
                    depth + 1,
                    maximum_depth,
                )
                if value
                else None
            ),
        }

    return {
        "type": value_type_name(value),
        "value": (
            value
            if (
                isinstance(
                    value,
                    (bool, int, float),
                )
                or value is None
            )
            else None
        ),
    }


roadmap_metric_model_summaries = {
    model_name: summarize_nested_structure(
        model_payload,
        maximum_depth=4,
    )
    for model_name, model_payload
    in sorted(roadmap_metrics.items())
}

benchmark_manifest_summary = (
    summarize_nested_structure(
        benchmark_manifest,
        maximum_depth=4,
    )
)

reproducibility_handoff_summary = (
    summarize_nested_structure(
        reproducibility_handoff,
        maximum_depth=4,
    )
)

query_profile_summary = query_profile.get(
    "summary"
)
query_profile_configuration = (
    query_profile.get("configuration")
)
query_profile_idf_policy = (
    query_profile.get("idf_policy")
)
query_profile_candidate = (
    query_profile.get("candidate")
)

nb13_gold_mapping_audit = {
    "active_golden_dataset_sha256": (
        canonical_golden_file_sha256
    ),
    "canonical_golden_query_count": len(
        golden_records
    ),
    "profile_row_count": len(
        profile_rows
    ),
    "profile_row_schema": (
        profile_row_schema
    ),
    "profile_nested_keys": (
        profile_nested_keys
    ),
    "profile_candidate_fields": (
        profile_row_candidates
    ),
    "profile_identifier_profiles": (
        profile_identifier_profiles
    ),
    "candidate_id_alignment": (
        candidate_id_alignment
    ),
    "query_text_alignment": (
        query_text_alignment
    ),
    "mapping_candidate_fields": (
        mapping_candidate_fields
    ),
    "mapping_field_profiles": (
        mapping_field_profiles
    ),
    "split_candidate_fields": (
        profile_split_candidates
    ),
    "split_distributions": (
        profile_split_distributions
    ),
    "query_profile_candidate": (
        query_profile_candidate
    ),
    "query_profile_configuration": (
        query_profile_configuration
    ),
    "query_profile_idf_policy": (
        query_profile_idf_policy
    ),
    "query_profile_summary": (
        query_profile_summary
    ),
    "benchmark_manifest_structure": (
        benchmark_manifest_summary
    ),
    "roadmap_metric_model_structures": (
        roadmap_metric_model_summaries
    ),
    "reproducibility_handoff_structure": (
        reproducibility_handoff_summary
    ),
    "artifact_hashes": {
        name: sha256_file(path)
        for name, path
        in required_nb13_paths.items()
    },
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "nb13_gold_mapping_audit_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        nb13_gold_mapping_audit,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "active_golden_dataset_sha256": "b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2",
  "canonical_golden_query_count": 3813,
  "profile_row_count": 3813,
  "profile_row_schema": {
    "record_count": 3813,
    "fields": {
      "candidate_id": {
        "present_count": 3813,
        "present_rate": 1.0,
        "non_null_count": 3813,
        "non_empty_count": 3813,
        "types": {
          "string": 3813
        }
      },
      "first_positive_rank": {
        "present_count": 3813,
        "present_rate": 1.0,
        "non_null_count": 3813,
        "non_empty_count": 3813,
        "types": {
          "integer": 3813
        }
      },
      "high_idf_feature_count": {
        "present_count": 3813,
        "present_rate": 1.0,
        "non_null_count": 3813,
        "non_empty_count": 3813,
        "types": {
          "integer": 3813
        }
      },
      "hit_at_10": {
        "present_count": 3813,
        "present_rate": 1.0,
        "non_null_count

<div dir="rtl" align="right">

## مرحله ۶ — تطبیق Benchmark تاریخی با Corpus رسمی NB15

Artifact اطلاعاتی NB13 که بررسی شد، برای تحلیل یک Candidate از Chunking به نام `parent_child_children` ساخته شده است. Corpus رسمی Dense Index و ورودی NB15، Candidate دیگری با نام `baseline_phase1_char_safe` و ۳۴۸۸ Chunk است.

استفاده مستقیم از Rankها یا IDFهای Candidate قبلی برای Sparse Benchmark فعلی نادرست خواهد بود، زیرا:

- تعداد Chunkها متفاوت است؛
- مرز Chunkها متفاوت است؛
- Vocabulary و Document Frequency متفاوت است؛
- Rank مثبت‌ها به همان Corpus قبلی وابسته است؛
- NB16 باید Dense و Sparse را روی مجموعه Chunk یکسان ترکیب کند.

در این مرحله Manifest کامل Benchmark، تصمیم نهایی Embedding Release و تمام Artifactهای Root منتخب بررسی می‌شوند تا:

- Candidate نهایی Chunking و دلیل انتخاب آن تأیید شود؛
- Artifactهای مربوط به `baseline_phase1_char_safe` شناسایی شوند؛
- وجود Gold mapping، relevance projection یا per-query result برای Corpus رسمی بررسی شود؛
- Artifactهای صرفاً آزمایشی Parent-child از ورودی رسمی NB15 جدا شوند؛
- مشخص شود Mapping رسمی قابل‌بازاستفاده وجود دارد یا باید از Locatorهای فریز‌شده به‌صورت قابل‌بازتولید بازسازی شود.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- Corpus رسمی ۳۴۸۸ Chunk بدون ابهام تأیید شود؛
- تمام Artifactهای Root منتخب فهرست شوند؛
- ساختار نتایج Candidateهای Chunking مشخص شود؛
- وجود یا نبود Gold mapping سازگار با Corpus رسمی با Evidence تعیین شود؛
- هیچ Rank یا Mapping متعلق به Candidate دیگر وارد Benchmark NB15 نشود.

</div>

In [7]:
# Code Cell 07 — Reconcile NB13 benchmark artifacts with the canonical 3488-Chunk corpus

selected_benchmark_full_inventory = []

for artifact_path in sorted(
    selected_benchmark_root.iterdir()
):
    if not artifact_path.is_file():
        continue

    selected_benchmark_full_inventory.append(
        {
            "name": artifact_path.name,
            "path": str(
                artifact_path.relative_to(
                    repo_root
                )
            ),
            "suffix": (
                artifact_path.suffix.lower()
            ),
            "size_bytes": (
                artifact_path.stat().st_size
            ),
            "sha256": sha256_file(
                artifact_path
            ),
        }
    )

assert selected_benchmark_full_inventory


def compact_structure_with_scalars(
    value: Any,
    depth: int = 0,
    maximum_depth: int = 5,
) -> Any:
    if isinstance(value, dict):
        if depth >= maximum_depth:
            return {
                "type": "object",
                "key_count": len(value),
                "keys": sorted(value.keys()),
            }

        return {
            key: compact_structure_with_scalars(
                child,
                depth + 1,
                maximum_depth,
            )
            for key, child in sorted(
                value.items()
            )
        }

    if isinstance(value, list):
        return {
            "type": "array",
            "length": len(value),
            "first_item_structure": (
                compact_structure_with_scalars(
                    value[0],
                    depth + 1,
                    maximum_depth,
                )
                if value
                else None
            ),
        }

    if isinstance(
        value,
        (bool, int, float),
    ) or value is None:
        return value

    if isinstance(value, str):
        if len(value) <= 200:
            return value

        return {
            "type": "string",
            "length": len(value),
        }

    return {
        "type": type(value).__name__
    }


manifest_active_golden = benchmark_manifest.get(
    "active_golden_dataset"
)
manifest_chunking_candidates = (
    benchmark_manifest.get(
        "chunking_candidates"
    )
)
manifest_workload = benchmark_manifest.get(
    "workload"
)

assert isinstance(
    manifest_active_golden,
    dict,
)
assert isinstance(
    manifest_chunking_candidates,
    dict,
)

assert (
    manifest_active_golden.get("sha256")
    == canonical_golden_file_sha256
)
assert int(
    manifest_active_golden.get(
        "record_count"
    )
) == len(golden_records)

assert (
    "baseline_phase1_char_safe"
    in manifest_chunking_candidates
)

baseline_manifest_contract = (
    manifest_chunking_candidates[
        "baseline_phase1_char_safe"
    ]
)

assert int(
    baseline_manifest_contract["records"]
) == len(chunk_records)

baseline_manifest_path = (
    repo_root
    / Path(
        baseline_manifest_contract["path"]
    )
)

assert baseline_manifest_path.resolve() == (
    canonical_chunk_path.resolve()
)
assert sha256_file(
    baseline_manifest_path
) == canonical_chunk_file_sha256

embedding_release_benchmark = (
    embedding_release.get("benchmark")
)
embedding_release_chunking = (
    embedding_release.get("chunking")
)
embedding_release_closure = (
    embedding_release.get("closure")
)

candidate_result_files = []

for inventory_item in (
    selected_benchmark_full_inventory
):
    normalized_name = (
        inventory_item["name"].lower()
    )

    if (
        inventory_item["suffix"] == ".json"
        and any(
            token in normalized_name
            for token in (
                "retrieval_benchmark",
                "roadmap_metrics",
                "information_profile",
                "failure_profile",
                "tradeoff",
                "decision",
                "adr",
            )
        )
    ):
        candidate_result_files.append(
            inventory_item
        )

candidate_result_structures = {}

for inventory_item in candidate_result_files:
    artifact_path = (
        repo_root
        / inventory_item["path"]
    )

    try:
        payload = json.loads(
            artifact_path.read_text(
                encoding="utf-8"
            )
        )
    except json.JSONDecodeError:
        continue

    candidate_result_structures[
        inventory_item["path"]
    ] = compact_structure_with_scalars(
        payload,
        maximum_depth=6,
    )

mapping_filename_tokens = (
    "mapping",
    "relevance",
    "relevant",
    "positive",
    "projection",
    "locator",
    "ground",
    "gold",
    "alignment",
)

possible_mapping_artifacts = [
    inventory_item
    for inventory_item
    in selected_benchmark_full_inventory
    if any(
        token in inventory_item[
            "name"
        ].lower()
        for token in mapping_filename_tokens
    )
]

baseline_named_artifacts = [
    inventory_item
    for inventory_item
    in selected_benchmark_full_inventory
    if any(
        token in inventory_item[
            "name"
        ].lower()
        for token in (
            "baseline",
            "phase1_char_safe",
            "char_safe",
        )
    )
]

parent_child_named_artifacts = [
    inventory_item
    for inventory_item
    in selected_benchmark_full_inventory
    if "parent_child" in (
        inventory_item["name"].lower()
    )
]

selected_embedding_model_id = (
    embedding_release["embedding"][
        "model_id"
    ]
)

selected_embedding_revision = (
    embedding_release["embedding"][
        "revision"
    ]
)

canonical_candidate_reconciliation = {
    "canonical_corpus": {
        "candidate": (
            "baseline_phase1_char_safe"
        ),
        "path": str(
            canonical_chunk_relative_path
        ),
        "sha256": (
            canonical_chunk_file_sha256
        ),
        "record_count": len(
            chunk_records
        ),
    },
    "noncanonical_information_profile": {
        "candidate": (
            query_profile_candidate
        ),
        "configuration": (
            query_profile_configuration
        ),
        "profile_row_count": len(
            profile_rows
        ),
        "safe_for_direct_nb15_gold_mapping": (
            False
        ),
        "reason": (
            "The profile belongs to a different "
            "Chunking candidate and corpus size."
        ),
    },
    "selected_embedding": {
        "model_id": (
            selected_embedding_model_id
        ),
        "revision": (
            selected_embedding_revision
        ),
    },
    "embedding_release_benchmark": (
        embedding_release_benchmark
    ),
    "embedding_release_chunking": (
        embedding_release_chunking
    ),
    "embedding_release_closure": (
        embedding_release_closure
    ),
    "benchmark_manifest": (
        benchmark_manifest
    ),
    "manifest_workload": (
        manifest_workload
    ),
    "full_benchmark_root_inventory": (
        selected_benchmark_full_inventory
    ),
    "baseline_named_artifacts": (
        baseline_named_artifacts
    ),
    "parent_child_named_artifacts": (
        parent_child_named_artifacts
    ),
    "possible_mapping_artifacts": (
        possible_mapping_artifacts
    ),
    "candidate_result_structures": (
        candidate_result_structures
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "candidate_reconciliation_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        canonical_candidate_reconciliation,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "canonical_corpus": {
    "candidate": "baseline_phase1_char_safe",
    "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
    "sha256": "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
    "record_count": 3488
  },
  "noncanonical_information_profile": {
    "candidate": "parent_child_children",
    "configuration": "arvan_text_embedding_3_large_plus_persian_tfidf_rrf",
    "profile_row_count": 3813,
    "safe_for_direct_nb15_gold_mapping": false,
    "reason": "The profile belongs to a different Chunking candidate and corpus size."
  },
  "selected_embedding": {
    "model_id": "intfloat/multilingual-e5-small",
    "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3"
  },
  "embedding_release_benchmark": {
    "mrr": 0.4112586794908517,
    "ndcg_at_10": 0.3735597782433693,
    "ndcg_at_5": 0.3435425881236073,
    "query_count": 3813,
    "recall_at_10": 0.6113296616837136,
    "recall_at_5": 0.5229478101232625
  },
  "embeddin

<div dir="rtl" align="right">

## مرحله ۷ — ممیزی Locatorها برای بازسازی Gold Chunk Mapping

برای Benchmark معتبر Sparse، هر Query باید به مجموعه‌ای از Chunkهای مرتبط متصل شود. Golden Dataset این ارتباط را با `source_locator` و بازه‌های `text_char_start` و `text_char_end` ثبت کرده است. Chunkها نیز Locatorهای شروع، پایان و Member locatorهای خود را دارند.

پیش از نوشتن الگوریتم Mapping باید شکل واقعی Locatorها بررسی شود:

- نوع و ساختار `source_locator` در Golden Dataset؛
- نوع و ساختار `member_locators` در Chunkها؛
- تفاوت Locatorهای Website، Book و Transcript؛
- فیلدهای موقعیت، Offset، Page، Timestamp یا Anchor؛
- امکان اتصال دقیق Golden locator به Source member؛
- امکان سنجش هم‌پوشانی بازه Evidence با محدوده Chunk؛
- موارد چندمنبعی یا چندLocatorی؛
- مواردی که Mapping مبهم یا ناممکن است.

در این مرحله هنوز Mapping نهایی ساخته نمی‌شود. ابتدا قرارداد Locator براساس داده واقعی استخراج خواهد شد.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- نوع تمام Locatorها مشخص شود؛
- کلیدهای داخلی Locatorها به تفکیک Source type ثبت شوند؛
- نمونه ساختاری کنترل‌شده از هر Source type تهیه شود؛
- فیلدهای قابل‌استفاده برای اتصال Source و محاسبه overlap شناسایی شوند؛
- موارد فاقد Locator یا دارای ساختار ناسازگار شمارش شوند؛
- طراحی الگوریتم Mapping بدون حدس ممکن شود.

</div>

In [8]:
# Code Cell 08 — Audit Golden and Chunk locator structures before mapping

def recursive_structure(
    value: Any,
    depth: int = 0,
    maximum_depth: int = 6,
) -> Any:
    if depth >= maximum_depth:
        if isinstance(value, dict):
            return {
                "type": "object",
                "keys": sorted(value.keys()),
            }

        if isinstance(value, list):
            return {
                "type": "array",
                "length": len(value),
            }

        return {
            "type": value_type_name(value)
        }

    if isinstance(value, dict):
        return {
            key: recursive_structure(
                child,
                depth + 1,
                maximum_depth,
            )
            for key, child in sorted(
                value.items()
            )
        }

    if isinstance(value, list):
        return {
            "type": "array",
            "length": len(value),
            "first_item": (
                recursive_structure(
                    value[0],
                    depth + 1,
                    maximum_depth,
                )
                if value
                else None
            ),
        }

    if isinstance(
        value,
        (bool, int, float),
    ) or value is None:
        return {
            "type": value_type_name(value),
            "value": value,
        }

    if isinstance(value, str):
        return {
            "type": "string",
            "length": len(value),
            "sample": value[:200],
        }

    return {
        "type": type(value).__name__
    }


def flatten_key_paths(
    value: Any,
    prefix: str = "",
) -> Counter[str]:
    paths: Counter[str] = Counter()

    if isinstance(value, dict):
        for key, child in value.items():
            child_path = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            paths[child_path] += 1
            paths.update(
                flatten_key_paths(
                    child,
                    child_path,
                )
            )

    elif isinstance(value, list):
        list_path = (
            f"{prefix}[]"
            if prefix
            else "[]"
        )

        for child in value:
            paths.update(
                flatten_key_paths(
                    child,
                    list_path,
                )
            )

    return paths


golden_locator_type_counts: Counter[str] = (
    Counter()
)
golden_source_locator_type_counts: Counter[str] = (
    Counter()
)
golden_locator_key_paths: Counter[str] = (
    Counter()
)
golden_invalid_locator_count = 0

golden_locator_samples_by_source: dict[
    str,
    list[dict[str, Any]],
] = defaultdict(list)

for record in golden_records:
    record_source_types = (
        record.get("source_types")
        or ["UNKNOWN"]
    )

    locators = record.get(
        "evidence_locators"
    )

    if not isinstance(locators, list):
        golden_invalid_locator_count += 1
        continue

    for locator in locators:
        golden_locator_type_counts[
            value_type_name(locator)
        ] += 1

        if not isinstance(locator, dict):
            golden_invalid_locator_count += 1
            continue

        golden_locator_key_paths.update(
            flatten_key_paths(locator)
        )

        source_locator = locator.get(
            "source_locator"
        )

        golden_source_locator_type_counts[
            value_type_name(source_locator)
        ] += 1

        for source_type in record_source_types:
            normalized_source_type = str(
                source_type
            )

            if (
                len(
                    golden_locator_samples_by_source[
                        normalized_source_type
                    ]
                )
                < 3
            ):
                golden_locator_samples_by_source[
                    normalized_source_type
                ].append(
                    {
                        "candidate_id": (
                            record[
                                "candidate_id"
                            ]
                        ),
                        "locator_structure": (
                            recursive_structure(
                                locator
                            )
                        ),
                    }
                )

chunk_member_locator_type_counts: Counter[str] = (
    Counter()
)
chunk_member_item_type_counts: Counter[str] = (
    Counter()
)
chunk_member_locator_key_paths: Counter[str] = (
    Counter()
)
chunk_start_locator_key_paths: Counter[str] = (
    Counter()
)
chunk_end_locator_key_paths: Counter[str] = (
    Counter()
)

chunk_invalid_member_locator_count = 0

chunk_locator_samples_by_source: dict[
    str,
    list[dict[str, Any]],
] = defaultdict(list)

for record in chunk_records:
    source_type = str(
        record["source_type"]
    )

    member_locators = record.get(
        "member_locators"
    )

    chunk_member_locator_type_counts[
        value_type_name(member_locators)
    ] += 1

    if not isinstance(
        member_locators,
        list,
    ):
        chunk_invalid_member_locator_count += 1
        continue

    for member_locator in member_locators:
        chunk_member_item_type_counts[
            value_type_name(member_locator)
        ] += 1

        if isinstance(
            member_locator,
            dict,
        ):
            chunk_member_locator_key_paths.update(
                flatten_key_paths(
                    member_locator
                )
            )
        else:
            chunk_invalid_member_locator_count += 1

    locator_start = record.get(
        "locator_start"
    )
    locator_end = record.get(
        "locator_end"
    )

    if isinstance(locator_start, dict):
        chunk_start_locator_key_paths.update(
            flatten_key_paths(
                locator_start
            )
        )

    if isinstance(locator_end, dict):
        chunk_end_locator_key_paths.update(
            flatten_key_paths(
                locator_end
            )
        )

    if (
        len(
            chunk_locator_samples_by_source[
                source_type
            ]
        )
        < 3
    ):
        chunk_locator_samples_by_source[
            source_type
        ].append(
            {
                "chunk_id": record[
                    "chunk_id"
                ],
                "source_id": record[
                    "source_id"
                ],
                "member_document_ids": (
                    record.get(
                        "member_document_ids"
                    )
                ),
                "member_locators_structure": (
                    recursive_structure(
                        member_locators
                    )
                ),
                "locator_start_structure": (
                    recursive_structure(
                        locator_start
                    )
                ),
                "locator_end_structure": (
                    recursive_structure(
                        locator_end
                    )
                ),
            }
        )

golden_char_range_valid_count = 0
golden_char_range_invalid_count = 0

for locator in evidence_locator_values:
    if not isinstance(locator, dict):
        golden_char_range_invalid_count += 1
        continue

    char_start = locator.get(
        "text_char_start"
    )
    char_end = locator.get(
        "text_char_end"
    )

    if (
        isinstance(char_start, int)
        and isinstance(char_end, int)
        and 0 <= char_start < char_end
    ):
        golden_char_range_valid_count += 1
    else:
        golden_char_range_invalid_count += 1

locator_contract_audit = {
    "golden_locators": {
        "locator_count": len(
            evidence_locator_values
        ),
        "locator_type_counts": dict(
            sorted(
                golden_locator_type_counts.items()
            )
        ),
        "source_locator_type_counts": dict(
            sorted(
                golden_source_locator_type_counts.items()
            )
        ),
        "key_paths": dict(
            sorted(
                golden_locator_key_paths.items()
            )
        ),
        "valid_text_char_range_count": (
            golden_char_range_valid_count
        ),
        "invalid_text_char_range_count": (
            golden_char_range_invalid_count
        ),
        "invalid_locator_count": (
            golden_invalid_locator_count
        ),
        "samples_by_source_type": (
            dict(
                golden_locator_samples_by_source
            )
        ),
    },
    "chunk_locators": {
        "chunk_count": len(
            chunk_records
        ),
        "member_locators_type_counts": dict(
            sorted(
                chunk_member_locator_type_counts.items()
            )
        ),
        "member_item_type_counts": dict(
            sorted(
                chunk_member_item_type_counts.items()
            )
        ),
        "member_locator_key_paths": dict(
            sorted(
                chunk_member_locator_key_paths.items()
            )
        ),
        "locator_start_key_paths": dict(
            sorted(
                chunk_start_locator_key_paths.items()
            )
        ),
        "locator_end_key_paths": dict(
            sorted(
                chunk_end_locator_key_paths.items()
            )
        ),
        "invalid_member_locator_count": (
            chunk_invalid_member_locator_count
        ),
        "samples_by_source_type": (
            dict(
                chunk_locator_samples_by_source
            )
        ),
    },
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "locator_contract_audit_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        locator_contract_audit,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "golden_locators": {
    "locator_count": 3924,
    "locator_type_counts": {
      "object": 3924
    },
    "source_locator_type_counts": {
      "object": 3924
    },
    "key_paths": {
      "source_locator": 3924,
      "source_locator.canonical_url": 1365,
      "source_locator.content_id": 9,
      "source_locator.edition_id": 399,
      "source_locator.episode_number": 2160,
      "source_locator.heading_path": 390,
      "source_locator.locator_id": 2160,
      "source_locator.paragraph_id": 390,
      "source_locator.paragraph_number": 390,
      "source_locator.parent_paragraph_id": 9,
      "source_locator.physical_file_id": 2160,
      "source_locator.program_id": 2160,
      "source_locator.season_number": 2160,
      "source_locator.source_line_end": 2160,
      "source_locator.source_line_start": 2160,
      "source_locator.source_locator": 9,
      "source_locator.source_locator.note_number": 9,
      "source_locator.url": 1365,
      "source_locator.work_id": 399,


<div dir="rtl" align="right">

## مرحله ۸ — بازسازی قطعی Golden Query → Relevant Chunk Mapping

هر Chunk شامل مجموعه‌ای از Member locatorهاست و هر Evidence در Golden Dataset یک `source_locator` متناظر دارد. Mapping با برابری Canonical و کامل Locator انجام می‌شود؛ نه با تطبیق تقریبی متن، URL ناقص یا حدس براساس Source ID.

روش Mapping:

1. هر `member_locators[].locator` به JSON مرتب‌شده و Canonical تبدیل می‌شود.
2. برای هر Locator، تمام Chunkهایی که آن Member را پوشش می‌دهند ثبت می‌شوند.
3. هر `evidence_locators[].source_locator` با همان Canonicalization تبدیل می‌شود.
4. تمام Chunkهای متناظر با Evidence به مجموعه Relevantهای Query افزوده می‌شوند.
5. در Chunkهای Overlapدار، چند Relevant Chunk برای یک Evidence مجاز و صحیح است.
6. هیچ Evidence بدون Mapping نباید وارد Benchmark شود.
7. هیچ Mapping تقریبی یا مبتنی بر شباهت متن در این مرحله مجاز نیست.

این قرارداد باید برای Website، Book و Transcript به‌صورت مستقل پوشش کامل داشته باشد.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- تمام ۳۹۲۴ Evidence locator به حداقل یک Chunk متصل شوند؛
- تمام ۳۸۱۳ Query حداقل یک Relevant Chunk داشته باشند؛
- تمام Relevant IDها عضو مجموعه ۳۴۸۸ Chunk رسمی باشند؛
- Coverage هر سه Source type کامل باشد؛
- موارد Ambiguous و چندChunkی اندازه‌گیری شوند؛
- Mapping قطعی و قابل‌بازتولید باشد؛
- هیچ Artifactی هنوز نوشته نشود.

</div>

In [9]:
# Code Cell 09 — Build exact Golden Query to relevant Chunk mapping

def canonical_locator_key(
    locator: dict[str, Any],
) -> str:
    assert isinstance(locator, dict)

    return json.dumps(
        locator,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


locator_to_chunk_ids: dict[
    str,
    set[str],
] = defaultdict(set)

locator_to_source_types: dict[
    str,
    set[str],
] = defaultdict(set)

chunk_locator_pair_count = 0

for chunk_record in chunk_records:
    chunk_id = chunk_record["chunk_id"]
    source_type = chunk_record[
        "source_type"
    ]
    member_locators = chunk_record[
        "member_locators"
    ]

    assert isinstance(member_locators, list)
    assert member_locators

    for member_locator in member_locators:
        assert isinstance(
            member_locator,
            dict,
        )

        locator = member_locator.get(
            "locator"
        )

        assert isinstance(locator, dict)
        assert locator

        locator_key = canonical_locator_key(
            locator
        )

        locator_to_chunk_ids[
            locator_key
        ].add(chunk_id)

        locator_to_source_types[
            locator_key
        ].add(source_type)

        chunk_locator_pair_count += 1

assert locator_to_chunk_ids
assert all(
    chunk_ids_for_locator
    for chunk_ids_for_locator
    in locator_to_chunk_ids.values()
)

gold_mapping_by_candidate_id: dict[
    str,
    set[str],
] = {}

gold_mapping_rows = []
unmapped_evidence = []
mapped_evidence_count = 0

evidence_chunk_multiplicity: Counter[int] = (
    Counter()
)
query_relevant_chunk_multiplicity: Counter[int] = (
    Counter()
)
mapped_query_source_type_counts: Counter[str] = (
    Counter()
)

for golden_record in golden_records:
    candidate_id = golden_record[
        "candidate_id"
    ]
    evidence_locators = golden_record[
        "evidence_locators"
    ]
    record_source_types = {
        str(source_type)
        for source_type in golden_record[
            "source_types"
        ]
    }

    relevant_chunk_ids: set[str] = set()
    evidence_mapping_details = []

    for evidence_index, evidence_locator in enumerate(
        evidence_locators
    ):
        assert isinstance(
            evidence_locator,
            dict,
        )

        source_locator = (
            evidence_locator.get(
                "source_locator"
            )
        )

        assert isinstance(
            source_locator,
            dict,
        )
        assert source_locator

        locator_key = canonical_locator_key(
            source_locator
        )

        matching_chunk_ids = set(
            locator_to_chunk_ids.get(
                locator_key,
                set(),
            )
        )

        if not matching_chunk_ids:
            unmapped_evidence.append(
                {
                    "candidate_id": (
                        candidate_id
                    ),
                    "evidence_index": (
                        evidence_index
                    ),
                    "source_types": sorted(
                        record_source_types
                    ),
                    "source_locator": (
                        source_locator
                    ),
                }
            )
            continue

        assert matching_chunk_ids <= (
            chunk_id_set
        )

        matching_source_types = set(
            locator_to_source_types[
                locator_key
            ]
        )

        assert matching_source_types
        assert (
            matching_source_types
            & record_source_types
        ), {
            "candidate_id": candidate_id,
            "record_source_types": sorted(
                record_source_types
            ),
            "matching_source_types": sorted(
                matching_source_types
            ),
        }

        relevant_chunk_ids.update(
            matching_chunk_ids
        )

        mapped_evidence_count += 1

        evidence_chunk_multiplicity[
            len(matching_chunk_ids)
        ] += 1

        evidence_mapping_details.append(
            {
                "evidence_index": (
                    evidence_index
                ),
                "matching_chunk_count": (
                    len(matching_chunk_ids)
                ),
                "matching_chunk_ids": sorted(
                    matching_chunk_ids
                ),
            }
        )

    gold_mapping_by_candidate_id[
        candidate_id
    ] = relevant_chunk_ids

    query_relevant_chunk_multiplicity[
        len(relevant_chunk_ids)
    ] += 1

    for source_type in record_source_types:
        mapped_query_source_type_counts[
            source_type
        ] += 1

    gold_mapping_rows.append(
        {
            "candidate_id": candidate_id,
            "claim_id": golden_record[
                "claim_id"
            ],
            "question": (
                golden_record[
                    "evaluation_content"
                ]["question"]
            ),
            "source_types": sorted(
                record_source_types
            ),
            "evidence_count": len(
                evidence_locators
            ),
            "relevant_chunk_count": len(
                relevant_chunk_ids
            ),
            "relevant_chunk_ids": sorted(
                relevant_chunk_ids
            ),
            "evidence_mapping_details": (
                evidence_mapping_details
            ),
        }
    )

unmapped_query_ids = sorted(
    candidate_id
    for candidate_id, relevant_ids
    in gold_mapping_by_candidate_id.items()
    if not relevant_ids
)

mapped_relevant_chunk_id_set = {
    chunk_id
    for relevant_ids
    in gold_mapping_by_candidate_id.values()
    for chunk_id in relevant_ids
}

assert not unmapped_evidence, (
    unmapped_evidence[:20]
)
assert not unmapped_query_ids, (
    unmapped_query_ids[:20]
)
assert mapped_evidence_count == len(
    evidence_locator_values
)
assert len(gold_mapping_by_candidate_id) == (
    len(golden_records)
)
assert mapped_relevant_chunk_id_set <= (
    chunk_id_set
)
assert all(
    gold_mapping_by_candidate_id[
        candidate_id
    ]
    for candidate_id in golden_candidate_ids
)

gold_mapping_digest = hashlib.sha256()

for row in sorted(
    gold_mapping_rows,
    key=lambda item: item[
        "candidate_id"
    ],
):
    mapping_bytes = json.dumps(
        {
            "candidate_id": (
                row["candidate_id"]
            ),
            "relevant_chunk_ids": (
                row["relevant_chunk_ids"]
            ),
        },
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

    gold_mapping_digest.update(
        len(mapping_bytes).to_bytes(
            8,
            byteorder="big",
            signed=False,
        )
    )
    gold_mapping_digest.update(
        mapping_bytes
    )

gold_mapping_sha256 = (
    gold_mapping_digest.hexdigest()
)

gold_mapping_audit = {
    "mapping_method": (
        "EXACT_CANONICAL_SOURCE_LOCATOR_IDENTITY"
    ),
    "canonicalization": (
        "json.dumps("
        "ensure_ascii=False,"
        "sort_keys=True,"
        "separators=(',', ':')"
        ")"
    ),
    "chunk_count": len(
        chunk_records
    ),
    "chunk_member_locator_pair_count": (
        chunk_locator_pair_count
    ),
    "unique_chunk_locator_count": len(
        locator_to_chunk_ids
    ),
    "golden_query_count": len(
        golden_records
    ),
    "evidence_locator_count": len(
        evidence_locator_values
    ),
    "mapped_evidence_count": (
        mapped_evidence_count
    ),
    "unmapped_evidence_count": len(
        unmapped_evidence
    ),
    "mapped_query_count": sum(
        bool(relevant_ids)
        for relevant_ids
        in gold_mapping_by_candidate_id.values()
    ),
    "unmapped_query_count": len(
        unmapped_query_ids
    ),
    "unique_relevant_chunk_count": len(
        mapped_relevant_chunk_id_set
    ),
    "evidence_to_chunk_multiplicity": {
        str(key): value
        for key, value
        in sorted(
            evidence_chunk_multiplicity.items()
        )
    },
    "query_relevant_chunk_multiplicity": {
        str(key): value
        for key, value
        in sorted(
            query_relevant_chunk_multiplicity.items()
        )
    },
    "mapped_query_source_type_counts": dict(
        sorted(
            mapped_query_source_type_counts.items()
        )
    ),
    "gold_mapping_sha256": (
        gold_mapping_sha256
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "gold_mapping_status": "PASS",
}

print(
    json.dumps(
        gold_mapping_audit,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "mapping_method": "EXACT_CANONICAL_SOURCE_LOCATOR_IDENTITY",
  "canonicalization": "json.dumps(ensure_ascii=False,sort_keys=True,separators=(',', ':'))",
  "chunk_count": 3488,
  "chunk_member_locator_pair_count": 25120,
  "unique_chunk_locator_count": 19456,
  "golden_query_count": 3813,
  "evidence_locator_count": 3924,
  "mapped_evidence_count": 3924,
  "unmapped_evidence_count": 0,
  "mapped_query_count": 3813,
  "unmapped_query_count": 0,
  "unique_relevant_chunk_count": 2483,
  "evidence_to_chunk_multiplicity": {
    "1": 1543,
    "2": 1787,
    "3": 371,
    "4": 121,
    "5": 19,
    "6": 30,
    "7": 30,
    "8": 12,
    "10": 11
  },
  "query_relevant_chunk_multiplicity": {
    "1": 1447,
    "2": 1776,
    "3": 374,
    "4": 118,
    "5": 18,
    "6": 26,
    "7": 29,
    "8": 12,
    "9": 1,
    "10": 12
  },
  "mapped_query_source_type_counts": {
    "book": 395,
    "transcript": 2089,
    "website": 1329
  },
  "gold_mapping_sha256": "42216f67665cfc50697789dffe4eb55

<div dir="rtl" align="right">

## مرحله ۹ — ساخت Split قطعی Development/Test بدون Leakage

Golden Dataset فعال Split آماده ندارد. برای انتخاب علمی Analyzer و تنظیمات BM25 باید پیش از هر Benchmark یک Split ثابت ساخته شود.

سیاست Split:

- ۷۰٪ رکوردها برای Development؛
- ۳۰٪ رکوردها برای Test نهایی؛
- Queryهای تکراری یا معادل دقیق نباید بین دو Split پخش شوند؛
- گروه‌بندی Query مستقل از Analyzerهای آینده انجام می‌شود؛
- Source typeها تا حد ممکن در هر دو Split متوازن می‌مانند؛
- Candidate ID و Gold mapping تغییر نمی‌کنند؛
- تخصیص با Hash و Seed نسخه‌دار، قطعی و قابل‌بازتولید است؛
- تنظیم Analyzer، BM25 و Threshold فقط روی Development انجام می‌شود؛
- Test تا پس از انتخاب و Freeze تنظیمات استفاده نخواهد شد.

برای جلوگیری از اثرگذاری Analyzerهای Candidate بر Split، Canonicalization سؤال فقط شامل Unicode NFC و یکسان‌سازی فاصله‌های سفید است. تبدیل حروف عربی/فارسی، حذف علائم، Stopword یا Stemming در Split استفاده نمی‌شود.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- تمام ۳۸۱۳ رکورد دقیقاً در یکی از Splitها قرار گیرند؛
- هیچ Candidate ID تکراری یا گم‌شده وجود نداشته باشد؛
- هیچ Query group میان Development و Test مشترک نباشد؛
- نسبت Test نزدیک ۳۰٪ باشد؛
- Website، Book و Transcript در هر دو Split حضور داشته باشند؛
- تمام رکوردهای هر دو Split Gold mapping معتبر داشته باشند؛
- Split fingerprint قطعی ثبت شود؛
- هنوز هیچ Artifactی نوشته نشود.

</div>

In [10]:
# Code Cell 10 — Build deterministic leakage-safe Development/Test split

SPLIT_POLICY_VERSION = (
    "nb15_grouped_query_hash_split_v1"
)
SPLIT_SEED = (
    "rag_finance_phase2_nb15_sparse_v1"
)
TARGET_TEST_RATIO = 0.30


def canonical_split_question(
    question: str,
) -> str:
    normalized = unicodedata.normalize(
        "NFC",
        question,
    )

    return " ".join(
        normalized.split()
    )


golden_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in golden_records
}

assert len(
    golden_record_by_candidate_id
) == len(golden_records)

query_groups: dict[
    str,
    list[str],
] = defaultdict(list)

for record in golden_records:
    candidate_id = record[
        "candidate_id"
    ]
    question = record[
        "evaluation_content"
    ]["question"]

    question_group_key = (
        canonical_split_question(
            question
        )
    )

    assert question_group_key

    query_groups[
        question_group_key
    ].append(candidate_id)

assert len(query_groups) == len(
    set(
        canonical_split_question(
            question
        )
        for question in golden_queries
    )
)

group_records = []

for question_group_key, candidate_ids in (
    query_groups.items()
):
    group_source_types = sorted(
        {
            str(source_type)
            for candidate_id in candidate_ids
            for source_type in (
                golden_record_by_candidate_id[
                    candidate_id
                ]["source_types"]
            )
        }
    )

    assert group_source_types

    source_signature = "|".join(
        group_source_types
    )

    group_hash = hashlib.sha256(
        (
            SPLIT_SEED
            + "\x1f"
            + question_group_key
        ).encode("utf-8")
    ).hexdigest()

    group_records.append(
        {
            "question_group_key": (
                question_group_key
            ),
            "candidate_ids": sorted(
                candidate_ids
            ),
            "record_count": len(
                candidate_ids
            ),
            "source_types": (
                group_source_types
            ),
            "source_signature": (
                source_signature
            ),
            "group_hash": group_hash,
        }
    )

groups_by_source_signature: dict[
    str,
    list[dict[str, Any]],
] = defaultdict(list)

for group_record in group_records:
    groups_by_source_signature[
        group_record["source_signature"]
    ].append(group_record)

split_by_candidate_id: dict[str, str] = {}
split_by_question_group: dict[str, str] = {}
stratum_summaries = {}

for source_signature, stratum_groups in sorted(
    groups_by_source_signature.items()
):
    ordered_groups = sorted(
        stratum_groups,
        key=lambda item: (
            item["group_hash"],
            item["question_group_key"],
        ),
    )

    stratum_record_count = sum(
        group["record_count"]
        for group in ordered_groups
    )

    target_test_count = round(
        stratum_record_count
        * TARGET_TEST_RATIO
    )

    assigned_test_count = 0
    test_group_keys = set()

    for group in ordered_groups:
        if assigned_test_count >= (
            target_test_count
        ):
            break

        test_group_keys.add(
            group["question_group_key"]
        )
        assigned_test_count += (
            group["record_count"]
        )

    for group in ordered_groups:
        split_name = (
            "test"
            if group["question_group_key"]
            in test_group_keys
            else "development"
        )

        split_by_question_group[
            group["question_group_key"]
        ] = split_name

        for candidate_id in group[
            "candidate_ids"
        ]:
            assert (
                candidate_id
                not in split_by_candidate_id
            )

            split_by_candidate_id[
                candidate_id
            ] = split_name

    stratum_summaries[
        source_signature
    ] = {
        "group_count": len(
            ordered_groups
        ),
        "record_count": (
            stratum_record_count
        ),
        "target_test_count": (
            target_test_count
        ),
        "actual_test_count": (
            assigned_test_count
        ),
        "actual_test_ratio": (
            assigned_test_count
            / stratum_record_count
        ),
    }

assert set(
    split_by_candidate_id
) == set(golden_candidate_ids)

development_candidate_ids = {
    candidate_id
    for candidate_id, split_name
    in split_by_candidate_id.items()
    if split_name == "development"
}

test_candidate_ids = {
    candidate_id
    for candidate_id, split_name
    in split_by_candidate_id.items()
    if split_name == "test"
}

assert development_candidate_ids
assert test_candidate_ids
assert (
    development_candidate_ids
    & test_candidate_ids
) == set()

assert (
    development_candidate_ids
    | test_candidate_ids
) == set(golden_candidate_ids)

development_question_groups = {
    canonical_split_question(
        golden_record_by_candidate_id[
            candidate_id
        ]["evaluation_content"][
            "question"
        ]
    )
    for candidate_id
    in development_candidate_ids
}

test_question_groups = {
    canonical_split_question(
        golden_record_by_candidate_id[
            candidate_id
        ]["evaluation_content"][
            "question"
        ]
    )
    for candidate_id in test_candidate_ids
}

assert (
    development_question_groups
    & test_question_groups
) == set()

assert all(
    gold_mapping_by_candidate_id[
        candidate_id
    ]
    for candidate_id
    in development_candidate_ids
)

assert all(
    gold_mapping_by_candidate_id[
        candidate_id
    ]
    for candidate_id
    in test_candidate_ids
)

split_source_type_counts: dict[
    str,
    Counter[str],
] = {
    "development": Counter(),
    "test": Counter(),
}

split_relevant_chunk_counts: dict[
    str,
    Counter[int],
] = {
    "development": Counter(),
    "test": Counter(),
}

for candidate_id, split_name in (
    split_by_candidate_id.items()
):
    record = golden_record_by_candidate_id[
        candidate_id
    ]

    for source_type in record[
        "source_types"
    ]:
        split_source_type_counts[
            split_name
        ][str(source_type)] += 1

    split_relevant_chunk_counts[
        split_name
    ][
        len(
            gold_mapping_by_candidate_id[
                candidate_id
            ]
        )
    ] += 1

all_source_types = set(
    split_source_type_counts[
        "development"
    ]
) | set(
    split_source_type_counts["test"]
)

assert all_source_types == {
    "book",
    "transcript",
    "website",
}

assert all(
    split_source_type_counts[
        "development"
    ][source_type] > 0
    and split_source_type_counts[
        "test"
    ][source_type] > 0
    for source_type in all_source_types
)

actual_test_ratio = (
    len(test_candidate_ids)
    / len(golden_records)
)

assert 0.28 <= actual_test_ratio <= 0.32

split_digest = hashlib.sha256()

for candidate_id in sorted(
    split_by_candidate_id
):
    split_component = (
        candidate_id
        + "\x1f"
        + split_by_candidate_id[
            candidate_id
        ]
    ).encode("utf-8")

    split_digest.update(
        len(split_component).to_bytes(
            8,
            byteorder="big",
            signed=False,
        )
    )
    split_digest.update(split_component)

split_sha256 = split_digest.hexdigest()

split_contract = {
    "policy_version": (
        SPLIT_POLICY_VERSION
    ),
    "seed": SPLIT_SEED,
    "target_test_ratio": (
        TARGET_TEST_RATIO
    ),
    "grouping": (
        "Unicode NFC + whitespace collapse "
        "on evaluation_content.question"
    ),
    "stratification": (
        "exact source-type signature"
    ),
    "record_count": len(
        golden_records
    ),
    "unique_question_group_count": len(
        query_groups
    ),
    "development": {
        "record_count": len(
            development_candidate_ids
        ),
        "question_group_count": len(
            development_question_groups
        ),
        "source_type_counts": dict(
            sorted(
                split_source_type_counts[
                    "development"
                ].items()
            )
        ),
        "relevant_chunk_multiplicity": {
            str(key): value
            for key, value
            in sorted(
                split_relevant_chunk_counts[
                    "development"
                ].items()
            )
        },
    },
    "test": {
        "record_count": len(
            test_candidate_ids
        ),
        "question_group_count": len(
            test_question_groups
        ),
        "actual_ratio": (
            actual_test_ratio
        ),
        "source_type_counts": dict(
            sorted(
                split_source_type_counts[
                    "test"
                ].items()
            )
        ),
        "relevant_chunk_multiplicity": {
            str(key): value
            for key, value
            in sorted(
                split_relevant_chunk_counts[
                    "test"
                ].items()
            )
        },
    },
    "strata": stratum_summaries,
    "cross_split_question_group_overlap": 0,
    "gold_mapping_coverage": "PASS",
    "split_sha256": split_sha256,
    "test_access_policy": (
        "LOCKED_UNTIL_ANALYZER_AND_BM25_"
        "CONFIG_SELECTION_IS_FROZEN"
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "split_status": "PASS",
}

print(
    json.dumps(
        split_contract,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "policy_version": "nb15_grouped_query_hash_split_v1",
  "seed": "rag_finance_phase2_nb15_sparse_v1",
  "target_test_ratio": 0.3,
  "grouping": "Unicode NFC + whitespace collapse on evaluation_content.question",
  "stratification": "exact source-type signature",
  "record_count": 3813,
  "unique_question_group_count": 3735,
  "development": {
    "record_count": 2664,
    "question_group_count": 2620,
    "source_type_counts": {
      "book": 277,
      "transcript": 1459,
      "website": 928
    },
    "relevant_chunk_multiplicity": {
      "1": 997,
      "2": 1250,
      "3": 260,
      "4": 87,
      "5": 14,
      "6": 19,
      "7": 20,
      "8": 7,
      "9": 1,
      "10": 9
    }
  },
  "test": {
    "record_count": 1149,
    "question_group_count": 1115,
    "actual_ratio": 0.3013375295043273,
    "source_type_counts": {
      "book": 118,
      "transcript": 630,
      "website": 401
    },
    "relevant_chunk_multiplicity": {
      "1": 450,
      "2": 526,
      "3": 

<div dir="rtl" align="right">

## مرحله ۱۰ — ممیزی نویسه‌ها و نیازهای واقعی Persian Analyzer

Analyzer فارسی نباید صرفاً مجموعه‌ای از Replaceهای رایج باشد. هر تبدیل می‌تواند باعث افزایش Recall یا ازبین‌رفتن اطلاعات دقیق شود؛ به‌ویژه برای نام‌ها، اعداد، درصدها، مخفف‌ها و اصطلاحات مالی.

در این مرحله Corpus کامل و فقط Queryهای Development از نظر موارد زیر بررسی می‌شوند:

- شکل‌های فارسی و عربی «ی» و «ک»؛
- انواع همزه، الف، ه و تاء مربوطه؛
- نیم‌فاصله و سایر نویسه‌های Zero-width؛
- ارقام فارسی، عربی و لاتین؛
- جداکننده‌های عددی، درصد و ممیز؛
- حروف لاتین و مخفف‌ها؛
- نشانه‌گذاری فارسی و لاتین؛
- فاصله‌های تکراری، Tab و Newline؛
- اثر احتمالی NFC و NFKC؛
- طول متن و Query؛
- Tokenهای دارای نیم‌فاصله یا عدد.

Test queryها در این ممیزی استفاده نمی‌شوند تا سیاست Analyzer تحت تأثیر Test قرار نگیرد.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- تمام نویسه‌های مهم فارسی و عربی شمارش شوند؛
- سیاست ارقام و نیم‌فاصله براساس Evidence قابل طراحی باشد؛
- میزان اثر NFC و NFKC مشخص شود؛
- الگوهای عددی و لاتین Corpus و Development ثبت شوند؛
- هیچ تبدیل یا Tokenization هنوز انتخاب نشود؛
- هیچ Artifact جدیدی ساخته نشود.

</div>

In [11]:
# Code Cell 11 — Audit Persian characters, digits and tokenization requirements

development_records = [
    golden_record_by_candidate_id[
        candidate_id
    ]
    for candidate_id
    in sorted(
        development_candidate_ids
    )
]

development_queries = [
    record["evaluation_content"][
        "question"
    ]
    for record in development_records
]

assert len(development_queries) == (
    split_contract["development"][
        "record_count"
    ]
)

corpus_texts = [
    record["content"]
    for record in chunk_records
]

assert all(
    isinstance(text, str)
    and text.strip()
    for text in corpus_texts
)

assert all(
    isinstance(query, str)
    and query.strip()
    for query in development_queries
)

target_characters = {
    "persian_yeh": "ی",
    "arabic_yeh": "ي",
    "alef_maksura": "ى",
    "persian_kaf": "ک",
    "arabic_kaf": "ك",
    "heh": "ه",
    "heh_with_yeh_above": "ۀ",
    "teh_marbuta": "ة",
    "hamza": "ء",
    "waw_with_hamza": "ؤ",
    "yeh_with_hamza": "ئ",
    "alef_with_madda": "آ",
    "alef_with_hamza_above": "أ",
    "alef_with_hamza_below": "إ",
    "zwnj": "\u200c",
    "zwj": "\u200d",
    "word_joiner": "\u2060",
    "zero_width_space": "\u200b",
    "rtl_mark": "\u200f",
    "ltr_mark": "\u200e",
    "persian_comma": "،",
    "arabic_semicolon": "؛",
    "arabic_question_mark": "؟",
    "percent_ascii": "%",
    "percent_arabic": "٪",
    "decimal_separator_arabic": "٫",
    "thousands_separator_arabic": "٬",
}


def text_collection_audit(
    texts: Sequence[str],
) -> dict[str, Any]:
    character_counts: Counter[str] = (
        Counter()
    )
    unicode_category_counts: Counter[str] = (
        Counter()
    )

    nfc_changed_count = 0
    nfkc_changed_count = 0
    repeated_whitespace_count = 0
    newline_text_count = 0
    tab_text_count = 0
    zwnj_text_count = 0

    persian_digit_count = 0
    arabic_digit_count = 0
    ascii_digit_count = 0

    numeric_token_count = 0
    latin_token_count = 0
    zwnj_token_count = 0

    character_lengths = []
    whitespace_token_lengths = []

    numeric_pattern = re.compile(
        r"[0-9۰-۹٠-٩]+"
        r"(?:[.,٫٬][0-9۰-۹٠-٩]+)*"
        r"(?:[%٪])?"
    )
    latin_pattern = re.compile(
        r"[A-Za-z]+(?:[-_/][A-Za-z0-9]+)*"
    )
    zwnj_token_pattern = re.compile(
        r"\S*\u200c\S*"
    )

    for text in texts:
        character_lengths.append(len(text))
        whitespace_token_lengths.append(
            len(text.split())
        )

        if unicodedata.normalize(
            "NFC",
            text,
        ) != text:
            nfc_changed_count += 1

        if unicodedata.normalize(
            "NFKC",
            text,
        ) != text:
            nfkc_changed_count += 1

        if re.search(r"\s{2,}", text):
            repeated_whitespace_count += 1

        if "\n" in text:
            newline_text_count += 1

        if "\t" in text:
            tab_text_count += 1

        if "\u200c" in text:
            zwnj_text_count += 1

        numeric_token_count += len(
            numeric_pattern.findall(text)
        )
        latin_token_count += len(
            latin_pattern.findall(text)
        )
        zwnj_token_count += len(
            zwnj_token_pattern.findall(text)
        )

        for character in text:
            character_counts[character] += 1
            unicode_category_counts[
                unicodedata.category(
                    character
                )
            ] += 1

            if "۰" <= character <= "۹":
                persian_digit_count += 1
            elif "٠" <= character <= "٩":
                arabic_digit_count += 1
            elif "0" <= character <= "9":
                ascii_digit_count += 1

    length_array = np.asarray(
        character_lengths,
        dtype=float,
    )
    token_length_array = np.asarray(
        whitespace_token_lengths,
        dtype=float,
    )

    return {
        "text_count": len(texts),
        "target_character_counts": {
            name: character_counts[
                character
            ]
            for name, character
            in target_characters.items()
        },
        "digit_character_counts": {
            "persian": persian_digit_count,
            "arabic": arabic_digit_count,
            "ascii": ascii_digit_count,
        },
        "pattern_counts": {
            "numeric_tokens": (
                numeric_token_count
            ),
            "latin_tokens": (
                latin_token_count
            ),
            "zwnj_tokens": (
                zwnj_token_count
            ),
        },
        "normalization_impact": {
            "nfc_changed_text_count": (
                nfc_changed_count
            ),
            "nfkc_changed_text_count": (
                nfkc_changed_count
            ),
        },
        "whitespace": {
            "repeated_whitespace_text_count": (
                repeated_whitespace_count
            ),
            "newline_text_count": (
                newline_text_count
            ),
            "tab_text_count": (
                tab_text_count
            ),
            "zwnj_text_count": (
                zwnj_text_count
            ),
        },
        "unicode_category_counts": dict(
            sorted(
                unicode_category_counts.items()
            )
        ),
        "character_length": {
            "minimum": int(
                length_array.min()
            ),
            "median": float(
                np.percentile(
                    length_array,
                    50,
                )
            ),
            "p95": float(
                np.percentile(
                    length_array,
                    95,
                )
            ),
            "maximum": int(
                length_array.max()
            ),
        },
        "whitespace_token_length": {
            "minimum": int(
                token_length_array.min()
            ),
            "median": float(
                np.percentile(
                    token_length_array,
                    50,
                )
            ),
            "p95": float(
                np.percentile(
                    token_length_array,
                    95,
                )
            ),
            "maximum": int(
                token_length_array.max()
            ),
        },
    }


corpus_character_audit = (
    text_collection_audit(
        corpus_texts
    )
)

development_query_character_audit = (
    text_collection_audit(
        development_queries
    )
)

assert len(corpus_texts) == 3488
assert len(development_queries) == 2664

character_audit = {
    "scope": {
        "corpus": (
            "ALL_CANONICAL_CHUNKS"
        ),
        "query": (
            "DEVELOPMENT_ONLY"
        ),
        "test_queries_accessed": False,
    },
    "corpus": corpus_character_audit,
    "development_queries": (
        development_query_character_audit
    ),
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "character_audit_status": "PASS",
}

print(
    json.dumps(
        character_audit,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "corpus": "ALL_CANONICAL_CHUNKS",
    "query": "DEVELOPMENT_ONLY",
    "test_queries_accessed": false
  },
  "corpus": {
    "text_count": 3488,
    "target_character_counts": {
      "persian_yeh": 309110,
      "arabic_yeh": 149,
      "alef_maksura": 75,
      "persian_kaf": 87159,
      "arabic_kaf": 48,
      "heh": 188493,
      "heh_with_yeh_above": 2161,
      "teh_marbuta": 47,
      "hamza": 148,
      "waw_with_hamza": 877,
      "yeh_with_hamza": 1952,
      "alef_with_madda": 18527,
      "alef_with_hamza_above": 1291,
      "alef_with_hamza_below": 29,
      "zwnj": 69002,
      "zwj": 0,
      "word_joiner": 0,
      "zero_width_space": 0,
      "rtl_mark": 0,
      "ltr_mark": 0,
      "persian_comma": 30420,
      "arabic_semicolon": 4879,
      "arabic_question_mark": 3430,
      "percent_ascii": 37,
      "percent_arabic": 12,
      "decimal_separator_arabic": 0,
      "thousands_separator_arabic": 0
    },
    "digit_character_counts": {
      "pe

<div dir="rtl" align="right">

## مرحله ۱۱ — تعریف Candidateهای نسخه‌دار Persian Analyzer

براساس ممیزی واقعی داده، چهار Candidate کنترل‌شده تعریف می‌شوند. تفاوت اصلی آن‌ها در سیاست نیم‌فاصله و Stopword است.

قواعد مشترک Candidateها:

- Unicode NFC؛
- تبدیل «ي» و «ى» به «ی»؛
- تبدیل «ك» به «ک»؛
- تبدیل «ة» به «ه»؛
- تبدیل ارقام فارسی و عربی به لاتین؛
- حذف اعراب و Combining markها؛
- Lowercase حروف لاتین؛
- یکسان‌سازی جداکننده هزارگان و ممیز؛
- حفظ عدد و Percent marker؛
- تبدیل نشانه‌گذاری به مرز Token؛
- حذف Token خالی؛
- بدون Stemming حدسی.

Candidateها:

1. حفظ نیم‌فاصله داخل Token؛
2. اتصال دو سوی نیم‌فاصله؛
3. تبدیل نیم‌فاصله به مرز Token؛
4. اتصال نیم‌فاصله همراه با حذف Stopwordهای محدود و نسخه‌دار.

Stopword Candidate فقط روی Development ارزیابی می‌شود و در صورت افت Metric رد خواهد شد.

در این مرحله Analyzerها تعریف و از نظر فنی بررسی می‌شوند، اما هنوز BM25 Benchmark اجرا نمی‌شود.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- تمام Candidateها خروجی قطعی تولید کنند؛
- هیچ Chunk یا Development Query به Token خالی تبدیل نشود؛
- ارقام و حروف عربی طبق قرارداد Normalize شوند؛
- رفتار نیم‌فاصله در هر Candidate قابل اثبات باشد؛
- Vocabulary، Document Frequency و Collision اولیه ثبت شود؛
- Test queryها استفاده نشوند؛
- هیچ Artifactی نوشته نشود.

</div>

In [12]:
# Code Cell 12 — Define and audit versioned Persian Analyzer candidates

PERSIAN_STOPWORDS_V1 = frozenset(
    {
        "از",
        "اگر",
        "اما",
        "آن",
        "این",
        "با",
        "برای",
        "بر",
        "به",
        "بود",
        "باشد",
        "تا",
        "را",
        "در",
        "که",
        "و",
        "یا",
        "یک",
        "می",
        "شود",
        "شده",
        "است",
        "هست",
        "چه",
        "چرا",
        "چگونه",
        "کدام",
        "آیا",
    }
)

ARABIC_TO_PERSIAN_CHARACTER_MAP = str.maketrans(
    {
        "ي": "ی",
        "ى": "ی",
        "ك": "ک",
        "ة": "ه",
    }
)

DIGIT_TO_ASCII_MAP = str.maketrans(
    {
        "۰": "0",
        "۱": "1",
        "۲": "2",
        "۳": "3",
        "۴": "4",
        "۵": "5",
        "۶": "6",
        "۷": "7",
        "۸": "8",
        "۹": "9",
        "٠": "0",
        "١": "1",
        "٢": "2",
        "٣": "3",
        "٤": "4",
        "٥": "5",
        "٦": "6",
        "٧": "7",
        "٨": "8",
        "٩": "9",
    }
)


@dataclass(frozen=True)
class AnalyzerConfig:
    analyzer_id: str
    unicode_form: str
    zwnj_policy: str
    normalize_arabic_letters: bool
    normalize_digits: bool
    remove_diacritics: bool
    lowercase_latin: bool
    remove_stopwords: bool
    stopword_version: str | None


ANALYZER_CANDIDATES = (
    AnalyzerConfig(
        analyzer_id=(
            "fa_surface_preserve_zwnj_v1"
        ),
        unicode_form="NFC",
        zwnj_policy="preserve",
        normalize_arabic_letters=True,
        normalize_digits=True,
        remove_diacritics=True,
        lowercase_latin=True,
        remove_stopwords=False,
        stopword_version=None,
    ),
    AnalyzerConfig(
        analyzer_id=(
            "fa_surface_join_zwnj_v1"
        ),
        unicode_form="NFC",
        zwnj_policy="join",
        normalize_arabic_letters=True,
        normalize_digits=True,
        remove_diacritics=True,
        lowercase_latin=True,
        remove_stopwords=False,
        stopword_version=None,
    ),
    AnalyzerConfig(
        analyzer_id=(
            "fa_surface_split_zwnj_v1"
        ),
        unicode_form="NFC",
        zwnj_policy="split",
        normalize_arabic_letters=True,
        normalize_digits=True,
        remove_diacritics=True,
        lowercase_latin=True,
        remove_stopwords=False,
        stopword_version=None,
    ),
    AnalyzerConfig(
        analyzer_id=(
            "fa_join_zwnj_stopwords_v1"
        ),
        unicode_form="NFC",
        zwnj_policy="join",
        normalize_arabic_letters=True,
        normalize_digits=True,
        remove_diacritics=True,
        lowercase_latin=True,
        remove_stopwords=True,
        stopword_version=(
            "PERSIAN_STOPWORDS_V1"
        ),
    ),
)

assert len(
    {
        config.analyzer_id
        for config in ANALYZER_CANDIDATES
    }
) == len(ANALYZER_CANDIDATES)


def normalize_numeric_notation(
    text: str,
) -> str:
    text = re.sub(
        r"(?<=\d)[,٬](?=\d)",
        "",
        text,
    )

    text = re.sub(
        r"(?<=\d)٫(?=\d)",
        ".",
        text,
    )

    text = text.replace(
        "٪",
        " __percent__ ",
    )
    text = text.replace(
        "%",
        " __percent__ ",
    )

    return text


def analyze_persian_text(
    text: str,
    config: AnalyzerConfig,
) -> list[str]:
    assert isinstance(text, str)

    normalized = unicodedata.normalize(
        config.unicode_form,
        text,
    )

    if config.normalize_arabic_letters:
        normalized = normalized.translate(
            ARABIC_TO_PERSIAN_CHARACTER_MAP
        )

    if config.normalize_digits:
        normalized = normalized.translate(
            DIGIT_TO_ASCII_MAP
        )

    if config.lowercase_latin:
        normalized = normalized.lower()

    normalized = normalize_numeric_notation(
        normalized
    )

    if config.zwnj_policy == "preserve":
        pass
    elif config.zwnj_policy == "join":
        normalized = normalized.replace(
            "\u200c",
            "",
        )
    elif config.zwnj_policy == "split":
        normalized = normalized.replace(
            "\u200c",
            " ",
        )
    else:
        raise AssertionError(
            f"Unknown ZWNJ policy: "
            f"{config.zwnj_policy}"
        )

    output_characters = []

    for character in normalized:
        if (
            config.remove_diacritics
            and unicodedata.category(
                character
            ).startswith("M")
        ):
            continue

        if character == "\u200c":
            output_characters.append(
                character
            )
            continue

        if character == ".":
            output_characters.append(
                character
            )
            continue

        if (
            character == "_"
            or character.isalpha()
            or character.isdigit()
        ):
            output_characters.append(
                character
            )
        else:
            output_characters.append(" ")

    normalized_for_tokens = "".join(
        output_characters
    )

    raw_tokens = (
        normalized_for_tokens.split()
    )

    tokens = []

    for token in raw_tokens:
        token = token.strip("._")

        if not token:
            continue

        if (
            config.remove_stopwords
            and token in PERSIAN_STOPWORDS_V1
        ):
            continue

        tokens.append(token)

    return tokens


analyzer_unit_cases = [
    {
        "text": "مي‌خواهم كسب‌وكار را بررسی کنم.",
        "expected": {
            "fa_surface_preserve_zwnj_v1": [
                "می‌خواهم",
                "کسب‌وکار",
                "را",
                "بررسی",
                "کنم",
            ],
            "fa_surface_join_zwnj_v1": [
                "میخواهم",
                "کسبوکار",
                "را",
                "بررسی",
                "کنم",
            ],
            "fa_surface_split_zwnj_v1": [
                "می",
                "خواهم",
                "کسب",
                "وکار",
                "را",
                "بررسی",
                "کنم",
            ],
            "fa_join_zwnj_stopwords_v1": [
                "میخواهم",
                "کسبوکار",
                "بررسی",
                "کنم",
            ],
        },
    },
    {
        "text": "بازدهی ۱۲٫۵٪ و مبلغ 1,250 تومان",
        "expected": {
            "fa_surface_preserve_zwnj_v1": [
                "بازدهی",
                "12.5",
                "percent",
                "و",
                "مبلغ",
                "1250",
                "تومان",
            ],
            "fa_surface_join_zwnj_v1": [
                "بازدهی",
                "12.5",
                "percent",
                "و",
                "مبلغ",
                "1250",
                "تومان",
            ],
            "fa_surface_split_zwnj_v1": [
                "بازدهی",
                "12.5",
                "percent",
                "و",
                "مبلغ",
                "1250",
                "تومان",
            ],
            "fa_join_zwnj_stopwords_v1": [
                "بازدهی",
                "12.5",
                "percent",
                "مبلغ",
                "1250",
                "تومان",
            ],
        },
    },
    {
        "text": "ETF، P/E و بهرهٔ مرکب چیست؟",
        "expected": {
            "fa_surface_preserve_zwnj_v1": [
                "etf",
                "p",
                "e",
                "و",
                "بهره",
                "مرکب",
                "چیست",
            ],
            "fa_surface_join_zwnj_v1": [
                "etf",
                "p",
                "e",
                "و",
                "بهره",
                "مرکب",
                "چیست",
            ],
            "fa_surface_split_zwnj_v1": [
                "etf",
                "p",
                "e",
                "و",
                "بهره",
                "مرکب",
                "چیست",
            ],
            "fa_join_zwnj_stopwords_v1": [
                "etf",
                "p",
                "e",
                "بهره",
                "مرکب",
                "چیست",
            ],
        },
    },
]

analyzer_by_id = {
    config.analyzer_id: config
    for config in ANALYZER_CANDIDATES
}

unit_test_results = []

for unit_case in analyzer_unit_cases:
    for analyzer_id, expected_tokens in (
        unit_case["expected"].items()
    ):
        actual_tokens = analyze_persian_text(
            unit_case["text"],
            analyzer_by_id[analyzer_id],
        )

        assert actual_tokens == expected_tokens, {
            "analyzer_id": analyzer_id,
            "text": unit_case["text"],
            "expected": expected_tokens,
            "actual": actual_tokens,
        }

        unit_test_results.append(
            {
                "analyzer_id": analyzer_id,
                "status": "PASS",
            }
        )

analyzer_profiles = {}

for config in ANALYZER_CANDIDATES:
    started_at = time.perf_counter()

    document_token_lists = [
        analyze_persian_text(
            text,
            config,
        )
        for text in corpus_texts
    ]

    development_query_token_lists = [
        analyze_persian_text(
            query,
            config,
        )
        for query in development_queries
    ]

    elapsed_seconds = (
        time.perf_counter() - started_at
    )

    empty_document_indexes = [
        index
        for index, tokens
        in enumerate(
            document_token_lists
        )
        if not tokens
    ]

    empty_query_indexes = [
        index
        for index, tokens
        in enumerate(
            development_query_token_lists
        )
        if not tokens
    ]

    assert not empty_document_indexes
    assert not empty_query_indexes

    term_frequency: Counter[str] = Counter()
    document_frequency: Counter[str] = (
        Counter()
    )

    for tokens in document_token_lists:
        term_frequency.update(tokens)
        document_frequency.update(
            set(tokens)
        )

    query_signatures = [
        "\x1f".join(tokens)
        for tokens
        in development_query_token_lists
    ]

    raw_query_group_count = len(
        {
            canonical_split_question(
                query
            )
            for query
            in development_queries
        }
    )

    analyzed_query_group_count = len(
        set(query_signatures)
    )

    token_lengths = np.asarray(
        [
            len(tokens)
            for tokens
            in document_token_lists
        ],
        dtype=float,
    )

    query_token_lengths = np.asarray(
        [
            len(tokens)
            for tokens
            in development_query_token_lists
        ],
        dtype=float,
    )

    analyzer_profiles[
        config.analyzer_id
    ] = {
        "config": asdict(config),
        "document_count": len(
            document_token_lists
        ),
        "development_query_count": len(
            development_query_token_lists
        ),
        "vocabulary_size": len(
            document_frequency
        ),
        "total_document_tokens": sum(
            term_frequency.values()
        ),
        "empty_document_count": len(
            empty_document_indexes
        ),
        "empty_development_query_count": len(
            empty_query_indexes
        ),
        "document_token_length": {
            "minimum": int(
                token_lengths.min()
            ),
            "median": float(
                np.percentile(
                    token_lengths,
                    50,
                )
            ),
            "p95": float(
                np.percentile(
                    token_lengths,
                    95,
                )
            ),
            "maximum": int(
                token_lengths.max()
            ),
        },
        "development_query_token_length": {
            "minimum": int(
                query_token_lengths.min()
            ),
            "median": float(
                np.percentile(
                    query_token_lengths,
                    50,
                )
            ),
            "p95": float(
                np.percentile(
                    query_token_lengths,
                    95,
                )
            ),
            "maximum": int(
                query_token_lengths.max()
            ),
        },
        "raw_query_group_count": (
            raw_query_group_count
        ),
        "analyzed_query_group_count": (
            analyzed_query_group_count
        ),
        "additional_query_collisions": (
            raw_query_group_count
            - analyzed_query_group_count
        ),
        "top_terms_by_document_frequency": [
            {
                "term": term,
                "document_frequency": count,
            }
            for term, count
            in document_frequency.most_common(
                20
            )
        ],
        "elapsed_seconds": (
            elapsed_seconds
        ),
        "status": "PASS",
    }

analyzer_audit = {
    "candidate_count": len(
        ANALYZER_CANDIDATES
    ),
    "candidate_ids": [
        config.analyzer_id
        for config in ANALYZER_CANDIDATES
    ],
    "unit_test_count": len(
        unit_test_results
    ),
    "unit_tests_passed": all(
        result["status"] == "PASS"
        for result in unit_test_results
    ),
    "profiles": analyzer_profiles,
    "scope": {
        "corpus": (
            "ALL_CANONICAL_CHUNKS"
        ),
        "queries": (
            "DEVELOPMENT_ONLY"
        ),
        "test_queries_accessed": False,
    },
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "analyzer_audit_status": "PASS",
}

print(
    json.dumps(
        analyzer_audit,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "candidate_count": 4,
  "candidate_ids": [
    "fa_surface_preserve_zwnj_v1",
    "fa_surface_join_zwnj_v1",
    "fa_surface_split_zwnj_v1",
    "fa_join_zwnj_stopwords_v1"
  ],
  "unit_test_count": 12,
  "unit_tests_passed": true,
  "profiles": {
    "fa_surface_preserve_zwnj_v1": {
      "config": {
        "analyzer_id": "fa_surface_preserve_zwnj_v1",
        "unicode_form": "NFC",
        "zwnj_policy": "preserve",
        "normalize_arabic_letters": true,
        "normalize_digits": true,
        "remove_diacritics": true,
        "lowercase_latin": true,
        "remove_stopwords": false,
        "stopword_version": null
      },
      "document_count": 3488,
      "development_query_count": 2664,
      "vocabulary_size": 28464,
      "total_document_tokens": 694599,
      "empty_document_count": 0,
      "empty_development_query_count": 0,
      "document_token_length": {
        "minimum": 1,
        "median": 223.0,
        "p95": 249.0,
        "maximum": 271
      },
   

<div dir="rtl" align="right">

## مرحله ۱۲ — ساخت BM25 و اجرای Pilot فنی روی Development

در این مرحله یک پیاده‌سازی محلی و نسخه‌دار BM25 ساخته می‌شود. هدف Pilot انتخاب نهایی نیست؛ هدف کشف زودهنگام خطاهای Indexing، Scoring، Ranking و Metric است.

فرمول BM25:

- IDF از نوع Robertson با فرم مثبت؛
- Term Frequency saturation با پارامتر `k1`؛
- Document length normalization با پارامتر `b`؛
- فقط Tokenهای Query موجود در Vocabulary امتیاز می‌گیرند؛
- Result دارای Score صفر، بازیابی معتبر محسوب نمی‌شود؛
- Tieها با ترتیب ثابت Document index شکسته می‌شوند.

Metricهای Pilot:

- Hit@5 و Hit@10؛
- Recall@5 و Recall@10؛
- Precision@5 و Precision@10؛
- MRR؛
- nDCG@5 و nDCG@10؛
- Zero-match query rate؛
- latency محلی p50 و p95.

Pilot فقط از Development ساخته می‌شود و Queryهای Test همچنان قفل باقی می‌مانند. برای مقایسه اولیه Analyzerها، پارامتر استاندارد `k1=1.2` و `b=0.75` استفاده می‌شود. انتخاب نهایی Analyzer یا پارامتر در این مرحله انجام نمی‌شود.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- برای هر چهار Analyzer یک Index مستقل ساخته شود؛
- Document count و شناسه‌ها ثابت بمانند؛
- BM25 Scoreها متناهی و نامنفی باشند؛
- Metricها در بازه معتبر قرار گیرند؛
- Pilot شامل هر سه Source type باشد؛
- Test queryها استفاده نشوند؛
- هیچ API یا Artifact جدیدی ایجاد نشود.

</div>

In [13]:
# Code Cell 13 — Build BM25 indexes and run a Development-only technical pilot

BM25_IMPLEMENTATION_VERSION = (
    "local_robertson_bm25_v1"
)
PILOT_K1 = 1.2
PILOT_B = 0.75
PILOT_PER_SOURCE = 50
PILOT_TOP_K_VALUES = (5, 10)


@dataclass
class BM25Index:
    analyzer_id: str
    document_ids: list[str]
    document_lengths: np.ndarray
    average_document_length: float
    postings: dict[
        str,
        tuple[np.ndarray, np.ndarray],
    ]
    document_frequency: dict[str, int]
    vocabulary_size: int
    total_token_count: int
    build_seconds: float


def build_bm25_index(
    texts: Sequence[str],
    document_ids: Sequence[str],
    config: AnalyzerConfig,
) -> BM25Index:
    assert len(texts) == len(document_ids)
    assert len(set(document_ids)) == len(
        document_ids
    )

    started_at = time.perf_counter()

    posting_builders: dict[
        str,
        list[tuple[int, int]],
    ] = defaultdict(list)

    document_lengths = np.zeros(
        len(texts),
        dtype=np.float32,
    )

    total_token_count = 0

    for document_index, text in enumerate(
        texts
    ):
        tokens = analyze_persian_text(
            text,
            config,
        )

        assert tokens

        token_counts = Counter(tokens)

        document_lengths[
            document_index
        ] = len(tokens)

        total_token_count += len(tokens)

        for term, term_frequency in (
            token_counts.items()
        ):
            posting_builders[term].append(
                (
                    document_index,
                    term_frequency,
                )
            )

    postings = {}
    document_frequency = {}

    for term, posting_rows in (
        posting_builders.items()
    ):
        document_indexes = np.asarray(
            [
                row[0]
                for row in posting_rows
            ],
            dtype=np.int32,
        )

        term_frequencies = np.asarray(
            [
                row[1]
                for row in posting_rows
            ],
            dtype=np.float32,
        )

        postings[term] = (
            document_indexes,
            term_frequencies,
        )

        document_frequency[term] = len(
            document_indexes
        )

    elapsed_seconds = (
        time.perf_counter() - started_at
    )

    average_document_length = float(
        document_lengths.mean()
    )

    assert average_document_length > 0
    assert postings

    return BM25Index(
        analyzer_id=config.analyzer_id,
        document_ids=list(document_ids),
        document_lengths=document_lengths,
        average_document_length=(
            average_document_length
        ),
        postings=postings,
        document_frequency=(
            document_frequency
        ),
        vocabulary_size=len(postings),
        total_token_count=(
            total_token_count
        ),
        build_seconds=elapsed_seconds,
    )


def bm25_score_query(
    index: BM25Index,
    query_tokens: Sequence[str],
    k1: float,
    b: float,
) -> np.ndarray:
    assert k1 > 0
    assert 0 <= b <= 1

    scores = np.zeros(
        len(index.document_ids),
        dtype=np.float64,
    )

    query_term_counts = Counter(
        query_tokens
    )

    document_count = len(
        index.document_ids
    )

    for term, query_term_frequency in (
        query_term_counts.items()
    ):
        posting = index.postings.get(term)

        if posting is None:
            continue

        document_indexes, term_frequencies = (
            posting
        )

        document_frequency = len(
            document_indexes
        )

        inverse_document_frequency = math.log(
            1.0
            + (
                (
                    document_count
                    - document_frequency
                    + 0.5
                )
                / (
                    document_frequency
                    + 0.5
                )
            )
        )

        length_normalization = (
            1.0
            - b
            + b
            * (
                index.document_lengths[
                    document_indexes
                ]
                / index.average_document_length
            )
        )

        term_score = (
            inverse_document_frequency
            * (
                term_frequencies
                * (k1 + 1.0)
            )
            / (
                term_frequencies
                + k1
                * length_normalization
            )
        )

        scores[
            document_indexes
        ] += (
            term_score
            * query_term_frequency
        )

    assert np.all(np.isfinite(scores))
    assert np.all(scores >= 0)

    return scores


def ranked_positive_document_indexes(
    scores: np.ndarray,
) -> np.ndarray:
    positive_indexes = np.flatnonzero(
        scores > 0
    )

    if len(positive_indexes) == 0:
        return np.asarray(
            [],
            dtype=np.int32,
        )

    ordering = np.lexsort(
        (
            positive_indexes,
            -scores[positive_indexes],
        )
    )

    return positive_indexes[
        ordering
    ]


def evaluate_ranked_result(
    ranked_document_ids: Sequence[str],
    relevant_document_ids: set[str],
    k_values: Sequence[int] = (5, 10),
) -> dict[str, float]:
    assert relevant_document_ids

    relevant_flags = np.asarray(
        [
            document_id
            in relevant_document_ids
            for document_id
            in ranked_document_ids
        ],
        dtype=bool,
    )

    result: dict[str, float] = {}

    first_relevant_positions = np.flatnonzero(
        relevant_flags
    )

    reciprocal_rank = (
        1.0
        / (
            int(
                first_relevant_positions[0]
            )
            + 1
        )
        if len(first_relevant_positions)
        else 0.0
    )

    result["reciprocal_rank"] = (
        reciprocal_rank
    )

    for k in k_values:
        top_flags = relevant_flags[:k]

        relevant_retrieved = int(
            top_flags.sum()
        )

        result[f"hit_at_{k}"] = float(
            relevant_retrieved > 0
        )

        result[f"recall_at_{k}"] = (
            relevant_retrieved
            / len(relevant_document_ids)
        )

        result[f"precision_at_{k}"] = (
            relevant_retrieved / k
        )

        discounts = (
            1.0
            / np.log2(
                np.arange(
                    2,
                    2 + len(top_flags),
                    dtype=float,
                )
            )
        )

        discounted_gain = float(
            np.sum(
                top_flags.astype(float)
                * discounts
            )
        )

        ideal_length = min(
            len(relevant_document_ids),
            k,
        )

        ideal_discounted_gain = float(
            np.sum(
                1.0
                / np.log2(
                    np.arange(
                        2,
                        2 + ideal_length,
                        dtype=float,
                    )
                )
            )
        )

        result[f"ndcg_at_{k}"] = (
            discounted_gain
            / ideal_discounted_gain
            if ideal_discounted_gain > 0
            else 0.0
        )

    return result


chunk_document_ids = [
    record["chunk_id"]
    for record in chunk_records
]

chunk_index_by_id = {
    chunk_id: index
    for index, chunk_id
    in enumerate(chunk_document_ids)
}

assert len(chunk_index_by_id) == len(
    chunk_document_ids
)

bm25_indexes = {}

for config in ANALYZER_CANDIDATES:
    bm25_indexes[
        config.analyzer_id
    ] = build_bm25_index(
        texts=corpus_texts,
        document_ids=chunk_document_ids,
        config=config,
    )

assert set(bm25_indexes) == {
    config.analyzer_id
    for config in ANALYZER_CANDIDATES
}

development_ids_by_source: dict[
    str,
    list[str],
] = defaultdict(list)

for candidate_id in sorted(
    development_candidate_ids
):
    record = golden_record_by_candidate_id[
        candidate_id
    ]

    for source_type in record[
        "source_types"
    ]:
        development_ids_by_source[
            str(source_type)
        ].append(candidate_id)

pilot_candidate_ids = set()

for source_type in (
    "book",
    "transcript",
    "website",
):
    source_candidate_ids = sorted(
        set(
            development_ids_by_source[
                source_type
            ]
        ),
        key=lambda candidate_id: (
            hashlib.sha256(
                (
                    SPLIT_SEED
                    + "\x1fPILOT\x1f"
                    + source_type
                    + "\x1f"
                    + candidate_id
                ).encode("utf-8")
            ).hexdigest()
        ),
    )

    assert len(source_candidate_ids) >= (
        PILOT_PER_SOURCE
    )

    pilot_candidate_ids.update(
        source_candidate_ids[
            :PILOT_PER_SOURCE
        ]
    )

pilot_candidate_ids = sorted(
    pilot_candidate_ids
)

assert pilot_candidate_ids
assert set(pilot_candidate_ids) <= (
    development_candidate_ids
)
assert not (
    set(pilot_candidate_ids)
    & test_candidate_ids
)

pilot_source_coverage = Counter()

for candidate_id in pilot_candidate_ids:
    for source_type in (
        golden_record_by_candidate_id[
            candidate_id
        ]["source_types"]
    ):
        pilot_source_coverage[
            str(source_type)
        ] += 1

assert all(
    pilot_source_coverage[
        source_type
    ] >= PILOT_PER_SOURCE
    for source_type in (
        "book",
        "transcript",
        "website",
    )
)

pilot_results = {}

for config in ANALYZER_CANDIDATES:
    index = bm25_indexes[
        config.analyzer_id
    ]

    metric_rows = []
    latencies_ms = []
    zero_match_count = 0

    for candidate_id in (
        pilot_candidate_ids
    ):
        record = (
            golden_record_by_candidate_id[
                candidate_id
            ]
        )

        query = record[
            "evaluation_content"
        ]["question"]

        query_tokens = analyze_persian_text(
            query,
            config,
        )

        started_at = time.perf_counter()

        scores = bm25_score_query(
            index=index,
            query_tokens=query_tokens,
            k1=PILOT_K1,
            b=PILOT_B,
        )

        ranked_indexes = (
            ranked_positive_document_indexes(
                scores
            )
        )

        latency_ms = (
            time.perf_counter()
            - started_at
        ) * 1000.0

        latencies_ms.append(
            latency_ms
        )

        if len(ranked_indexes) == 0:
            zero_match_count += 1

        ranked_document_ids = [
            index.document_ids[
                document_index
            ]
            for document_index
            in ranked_indexes
        ]

        query_metrics = (
            evaluate_ranked_result(
                ranked_document_ids=(
                    ranked_document_ids
                ),
                relevant_document_ids=(
                    gold_mapping_by_candidate_id[
                        candidate_id
                    ]
                ),
                k_values=(
                    PILOT_TOP_K_VALUES
                ),
            )
        )

        metric_rows.append(
            query_metrics
        )

    latency_array = np.asarray(
        latencies_ms,
        dtype=float,
    )

    metric_names = sorted(
        metric_rows[0]
    )

    aggregate_metrics = {
        metric_name: float(
            np.mean(
                [
                    row[metric_name]
                    for row in metric_rows
                ]
            )
        )
        for metric_name in metric_names
    }

    assert all(
        0.0 <= value <= 1.0
        for value
        in aggregate_metrics.values()
    )

    pilot_results[
        config.analyzer_id
    ] = {
        "analyzer_id": (
            config.analyzer_id
        ),
        "bm25": {
            "implementation": (
                BM25_IMPLEMENTATION_VERSION
            ),
            "k1": PILOT_K1,
            "b": PILOT_B,
        },
        "index": {
            "document_count": len(
                index.document_ids
            ),
            "vocabulary_size": (
                index.vocabulary_size
            ),
            "total_token_count": (
                index.total_token_count
            ),
            "average_document_length": (
                index.average_document_length
            ),
            "build_seconds": (
                index.build_seconds
            ),
        },
        "query_count": len(
            pilot_candidate_ids
        ),
        "zero_match_query_count": (
            zero_match_count
        ),
        "metrics": aggregate_metrics,
        "latency_ms": {
            "minimum": float(
                latency_array.min()
            ),
            "median_p50": float(
                np.percentile(
                    latency_array,
                    50,
                )
            ),
            "p95": float(
                np.percentile(
                    latency_array,
                    95,
                )
            ),
            "maximum": float(
                latency_array.max()
            ),
            "scope": (
                "LOCAL_PILOT_NOT_PRODUCTION_SLO"
            ),
        },
        "status": "PASS",
    }

bm25_pilot_summary = {
    "pilot_policy": {
        "split": "development",
        "per_source_target": (
            PILOT_PER_SOURCE
        ),
        "unique_query_count": len(
            pilot_candidate_ids
        ),
        "source_coverage": dict(
            sorted(
                pilot_source_coverage.items()
            )
        ),
        "selection": (
            "deterministic SHA256 by source type"
        ),
    },
    "analyzer_count": len(
        ANALYZER_CANDIDATES
    ),
    "results": pilot_results,
    "test_queries_accessed": False,
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "bm25_pilot_status": "PASS",
}

print(
    json.dumps(
        bm25_pilot_summary,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "pilot_policy": {
    "split": "development",
    "per_source_target": 50,
    "unique_query_count": 150,
    "source_coverage": {
      "book": 50,
      "transcript": 50,
      "website": 50
    },
    "selection": "deterministic SHA256 by source type"
  },
  "analyzer_count": 4,
  "results": {
    "fa_surface_preserve_zwnj_v1": {
      "analyzer_id": "fa_surface_preserve_zwnj_v1",
      "bm25": {
        "implementation": "local_robertson_bm25_v1",
        "k1": 1.2,
        "b": 0.75
      },
      "index": {
        "document_count": 3488,
        "vocabulary_size": 28464,
        "total_token_count": 694599,
        "average_document_length": 199.13961791992188,
        "build_seconds": 2.7373440789997403
      },
      "query_count": 150,
      "zero_match_query_count": 0,
      "metrics": {
        "hit_at_10": 0.78,
        "hit_at_5": 0.74,
        "ndcg_at_10": 0.5466469361850839,
        "ndcg_at_5": 0.5280612995431097,
        "precision_at_10": 0.09866666666666667,
  

<div dir="rtl" align="right">

## مرحله ۱۳ — Benchmark کامل Analyzer × BM25 روی Development

در این مرحله تمام Candidateهای Analyzer روی کل Development benchmark می‌شوند. برای جلوگیری از انتخاب دلخواه پس از مشاهده نتایج، Grid و قاعده انتخاب از قبل ثابت می‌شوند.

Grid پارامترها:

- `k1`: برابر ۰٫۸، ۱٫۲ و ۱٫۶؛
- `b`: برابر ۰٫۳ و ۰٫۷۵؛
- چهار Analyzer؛
- در مجموع ۲۴ Configuration.

قاعده رتبه‌بندی Development:

1. بیشترین `nDCG@10`؛
2. سپس بیشترین MRR؛
3. سپس بیشترین `Hit@10`؛
4. سپس بیشترین `Recall@10`؛
5. در تساوی کامل، latency کمتر.

این مرحله فقط Configهای برتر را برای تحلیل عمیق‌تر تعیین می‌کند. انتخاب نهایی پس از Slice analysis، آزمون نام‌ها و اعداد، Error analysis و بررسی پایداری انجام می‌شود.

Test همچنان قفل است.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- هر ۲۴ Configuration روی تمام Development اجرا شود؛
- همه Metricها معتبر و متناهی باشند؛
- Zero-match و latency ثبت شوند؛
- رتبه‌بندی طبق قانون از پیش تعیین‌شده انجام شود؛
- سه Configuration برتر برای تحلیل بعدی مشخص شوند؛
- هیچ Test query استفاده نشود؛
- هیچ Artifactی نوشته نشود.

</div>

In [14]:
# Code Cell 14 — Run the full Analyzer × BM25 grid on Development

BM25_K1_GRID = (0.8, 1.2, 1.6)
BM25_B_GRID = (0.3, 0.75)
GRID_TOP_K_VALUES = (5, 10)
TOP_CONFIGURATION_COUNT = 3

development_candidate_id_list = sorted(
    development_candidate_ids
)

assert len(
    development_candidate_id_list
) == 2664
assert not (
    set(development_candidate_id_list)
    & test_candidate_ids
)

development_query_tokens_by_analyzer = {}

for config in ANALYZER_CANDIDATES:
    development_query_tokens_by_analyzer[
        config.analyzer_id
    ] = {
        candidate_id: analyze_persian_text(
            golden_record_by_candidate_id[
                candidate_id
            ]["evaluation_content"][
                "question"
            ],
            config,
        )
        for candidate_id
        in development_candidate_id_list
    }

    assert all(
        development_query_tokens_by_analyzer[
            config.analyzer_id
        ][candidate_id]
        for candidate_id
        in development_candidate_id_list
    )


def aggregate_metric_rows(
    metric_rows: Sequence[
        dict[str, float]
    ],
) -> dict[str, float]:
    assert metric_rows

    metric_names = sorted(
        metric_rows[0]
    )

    return {
        metric_name: float(
            np.mean(
                [
                    row[metric_name]
                    for row in metric_rows
                ]
            )
        )
        for metric_name in metric_names
    }


development_grid_rows = []

for config in ANALYZER_CANDIDATES:
    analyzer_id = config.analyzer_id
    index = bm25_indexes[analyzer_id]

    for k1 in BM25_K1_GRID:
        for b in BM25_B_GRID:
            configuration_id = (
                f"{analyzer_id}"
                f"__k1_{k1:.2f}"
                f"__b_{b:.2f}"
            )

            metric_rows = []
            latency_values_ms = []
            zero_match_count = 0
            positive_result_counts = []

            configuration_started_at = (
                time.perf_counter()
            )

            for candidate_id in (
                development_candidate_id_list
            ):
                query_tokens = (
                    development_query_tokens_by_analyzer[
                        analyzer_id
                    ][candidate_id]
                )

                query_started_at = (
                    time.perf_counter()
                )

                scores = bm25_score_query(
                    index=index,
                    query_tokens=query_tokens,
                    k1=k1,
                    b=b,
                )

                ranked_indexes = (
                    ranked_positive_document_indexes(
                        scores
                    )
                )

                query_elapsed_ms = (
                    time.perf_counter()
                    - query_started_at
                ) * 1000.0

                latency_values_ms.append(
                    query_elapsed_ms
                )

                positive_result_counts.append(
                    len(ranked_indexes)
                )

                if len(ranked_indexes) == 0:
                    zero_match_count += 1

                ranked_document_ids = [
                    index.document_ids[
                        document_index
                    ]
                    for document_index
                    in ranked_indexes
                ]

                metric_rows.append(
                    evaluate_ranked_result(
                        ranked_document_ids=(
                            ranked_document_ids
                        ),
                        relevant_document_ids=(
                            gold_mapping_by_candidate_id[
                                candidate_id
                            ]
                        ),
                        k_values=(
                            GRID_TOP_K_VALUES
                        ),
                    )
                )

            configuration_elapsed_seconds = (
                time.perf_counter()
                - configuration_started_at
            )

            aggregate_metrics = (
                aggregate_metric_rows(
                    metric_rows
                )
            )

            assert all(
                np.isfinite(value)
                and 0.0 <= value <= 1.0
                for value
                in aggregate_metrics.values()
            )

            latency_array = np.asarray(
                latency_values_ms,
                dtype=float,
            )

            positive_result_array = np.asarray(
                positive_result_counts,
                dtype=float,
            )

            development_grid_rows.append(
                {
                    "configuration_id": (
                        configuration_id
                    ),
                    "analyzer_id": (
                        analyzer_id
                    ),
                    "k1": float(k1),
                    "b": float(b),
                    "query_count": len(
                        development_candidate_id_list
                    ),
                    "zero_match_query_count": (
                        zero_match_count
                    ),
                    "metrics": (
                        aggregate_metrics
                    ),
                    "latency_ms": {
                        "minimum": float(
                            latency_array.min()
                        ),
                        "median_p50": float(
                            np.percentile(
                                latency_array,
                                50,
                            )
                        ),
                        "p95": float(
                            np.percentile(
                                latency_array,
                                95,
                            )
                        ),
                        "maximum": float(
                            latency_array.max()
                        ),
                    },
                    "positive_result_count": {
                        "minimum": int(
                            positive_result_array.min()
                        ),
                        "median": float(
                            np.percentile(
                                positive_result_array,
                                50,
                            )
                        ),
                        "p95": float(
                            np.percentile(
                                positive_result_array,
                                95,
                            )
                        ),
                        "maximum": int(
                            positive_result_array.max()
                        ),
                    },
                    "elapsed_seconds": (
                        configuration_elapsed_seconds
                    ),
                    "status": "PASS",
                }
            )

assert len(development_grid_rows) == (
    len(ANALYZER_CANDIDATES)
    * len(BM25_K1_GRID)
    * len(BM25_B_GRID)
)

ranked_development_grid = sorted(
    development_grid_rows,
    key=lambda row: (
        -row["metrics"]["ndcg_at_10"],
        -row["metrics"]["reciprocal_rank"],
        -row["metrics"]["hit_at_10"],
        -row["metrics"]["recall_at_10"],
        row["latency_ms"]["median_p50"],
        row["configuration_id"],
    ),
)

for rank, row in enumerate(
    ranked_development_grid,
    start=1,
):
    row["development_rank"] = rank

top_development_configurations = (
    ranked_development_grid[
        :TOP_CONFIGURATION_COUNT
    ]
)

assert len(
    top_development_configurations
) == TOP_CONFIGURATION_COUNT

development_grid_summary = {
    "evaluation_scope": (
        "DEVELOPMENT_ONLY"
    ),
    "selection_policy": [
        "maximum_ndcg_at_10",
        "maximum_mrr",
        "maximum_hit_at_10",
        "maximum_recall_at_10",
        "minimum_median_latency_on_exact_tie",
    ],
    "analyzer_count": len(
        ANALYZER_CANDIDATES
    ),
    "k1_values": list(
        BM25_K1_GRID
    ),
    "b_values": list(
        BM25_B_GRID
    ),
    "configuration_count": len(
        development_grid_rows
    ),
    "development_query_count": len(
        development_candidate_id_list
    ),
    "ranked_configurations": (
        ranked_development_grid
    ),
    "top_configurations_for_deeper_analysis": (
        top_development_configurations
    ),
    "test_queries_accessed": False,
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "development_grid_status": "PASS",
}

print(
    json.dumps(
        development_grid_summary,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "evaluation_scope": "DEVELOPMENT_ONLY",
  "selection_policy": [
    "maximum_ndcg_at_10",
    "maximum_mrr",
    "maximum_hit_at_10",
    "maximum_recall_at_10",
    "minimum_median_latency_on_exact_tie"
  ],
  "analyzer_count": 4,
  "k1_values": [
    0.8,
    1.2,
    1.6
  ],
  "b_values": [
    0.3,
    0.75
  ],
  "configuration_count": 24,
  "development_query_count": 2664,
  "ranked_configurations": [
    {
      "configuration_id": "fa_join_zwnj_stopwords_v1__k1_1.20__b_0.30",
      "analyzer_id": "fa_join_zwnj_stopwords_v1",
      "k1": 1.2,
      "b": 0.3,
      "query_count": 2664,
      "zero_match_query_count": 0,
      "metrics": {
        "hit_at_10": 0.743993993993994,
        "hit_at_5": 0.6786786786786787,
        "ndcg_at_10": 0.50936500655395,
        "ndcg_at_5": 0.4855041074074556,
        "precision_at_10": 0.1001876876876877,
        "precision_at_5": 0.17432432432432432,
        "recall_at_10": 0.5796647838314506,
        "recall_at_5": 0.5128710555793888,


<div dir="rtl" align="right">

## مرحله ۱۴ — Slice Analysis و آزمون پایداری Configهای برتر

رتبه‌بندی تجمیعی به‌تنهایی برای انتخاب Config کافی نیست. ممکن است یک تنظیم در میانگین بهتر باشد اما روی Book، اعداد، Queryهای بلند یا عبارت‌های دارای نیم‌فاصله افت جدی داشته باشد.

در این مرحله موارد زیر مقایسه می‌شوند:

- سه Configuration برتر Grid؛
- بهترین Configuration مستقل هر Analyzer؛
- Source typeهای Website، Book و Transcript؛
- Queryهای کوتاه، متوسط و بلند؛
- Queryهای دارای عدد؛
- Queryهای دارای حروف لاتین؛
- Queryهای دارای نیم‌فاصله؛
- Queryهای دارای Term با IDF بالا؛
- Paired bootstrap برای اختلاف `nDCG@10`؛
- تعداد برد، باخت و تساوی QueryبهQuery؛
- Failureهای بدون Hit@10.

Test همچنان قفل باقی می‌ماند.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- بهترین نماینده هر Analyzer مشخص شود؛
- نتایج per-query قابل‌بازتولید ساخته شوند؛
- Sliceهای اصلی حداقل یک Query داشته باشند؛
- اختلاف Config برتر با رقبا به‌صورت جفتی سنجیده شود؛
- Confidence interval و Win/Loss/Tie ثبت شود؛
- Failureهای Development شناسایی شوند؛
- هنوز انتخاب نهایی یا استفاده از Test انجام نشود.

</div>

In [15]:
# Code Cell 15 — Run Development slice analysis and paired stability tests

BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_SEED = 15092026

best_row_by_analyzer = {}

for row in ranked_development_grid:
    analyzer_id = row["analyzer_id"]

    if analyzer_id not in best_row_by_analyzer:
        best_row_by_analyzer[
            analyzer_id
        ] = row

analysis_configuration_by_id = {}

for row in (
    top_development_configurations
    + list(
        best_row_by_analyzer.values()
    )
):
    analysis_configuration_by_id[
        row["configuration_id"]
    ] = row

analysis_configuration_rows = [
    analysis_configuration_by_id[
        configuration_id
    ]
    for configuration_id in sorted(
        analysis_configuration_by_id
    )
]

provisional_best_grid_row = (
    ranked_development_grid[0]
)
provisional_best_configuration_id = (
    provisional_best_grid_row[
        "configuration_id"
    ]
)

assert (
    provisional_best_configuration_id
    in analysis_configuration_by_id
)


def query_source_types(
    candidate_id: str,
) -> set[str]:
    return {
        str(source_type)
        for source_type in (
            golden_record_by_candidate_id[
                candidate_id
            ]["source_types"]
        )
    }


def high_idf_signal(
    query_tokens: Sequence[str],
    index: BM25Index,
    threshold: float,
) -> bool:
    document_count = len(
        index.document_ids
    )

    idf_values = []

    for term in set(query_tokens):
        document_frequency = (
            index.document_frequency.get(
                term
            )
        )

        if document_frequency is None:
            continue

        idf_values.append(
            math.log(
                1.0
                + (
                    (
                        document_count
                        - document_frequency
                        + 0.5
                    )
                    / (
                        document_frequency
                        + 0.5
                    )
                )
            )
        )

    return bool(
        idf_values
        and max(idf_values) >= threshold
    )


selected_analyzer_id = (
    provisional_best_grid_row[
        "analyzer_id"
    ]
)
selected_index_for_slice_definition = (
    bm25_indexes[selected_analyzer_id]
)

development_selected_query_tokens = (
    development_query_tokens_by_analyzer[
        selected_analyzer_id
    ]
)

development_maximum_idf_values = []

for candidate_id in (
    development_candidate_id_list
):
    query_tokens = (
        development_selected_query_tokens[
            candidate_id
        ]
    )

    query_idf_values = []

    for term in set(query_tokens):
        document_frequency = (
            selected_index_for_slice_definition
            .document_frequency.get(term)
        )

        if document_frequency is None:
            continue

        query_idf_values.append(
            math.log(
                1.0
                + (
                    (
                        len(
                            selected_index_for_slice_definition
                            .document_ids
                        )
                        - document_frequency
                        + 0.5
                    )
                    / (
                        document_frequency
                        + 0.5
                    )
                )
            )
        )

    development_maximum_idf_values.append(
        max(query_idf_values)
        if query_idf_values
        else 0.0
    )

high_idf_threshold = float(
    np.percentile(
        np.asarray(
            development_maximum_idf_values,
            dtype=float,
        ),
        75,
    )
)

configuration_query_results = {}

for configuration_row in (
    analysis_configuration_rows
):
    configuration_id = (
        configuration_row[
            "configuration_id"
        ]
    )
    analyzer_id = (
        configuration_row["analyzer_id"]
    )
    k1 = float(
        configuration_row["k1"]
    )
    b = float(
        configuration_row["b"]
    )

    config = analyzer_by_id[
        analyzer_id
    ]
    index = bm25_indexes[
        analyzer_id
    ]

    per_query_rows = []

    for candidate_id in (
        development_candidate_id_list
    ):
        golden_record = (
            golden_record_by_candidate_id[
                candidate_id
            ]
        )
        question = golden_record[
            "evaluation_content"
        ]["question"]

        query_tokens = (
            development_query_tokens_by_analyzer[
                analyzer_id
            ][candidate_id]
        )

        started_at = time.perf_counter()

        scores = bm25_score_query(
            index=index,
            query_tokens=query_tokens,
            k1=k1,
            b=b,
        )

        ranked_indexes = (
            ranked_positive_document_indexes(
                scores
            )
        )

        elapsed_ms = (
            time.perf_counter()
            - started_at
        ) * 1000.0

        ranked_document_ids = [
            index.document_ids[
                document_index
            ]
            for document_index
            in ranked_indexes
        ]

        metrics = evaluate_ranked_result(
            ranked_document_ids=(
                ranked_document_ids
            ),
            relevant_document_ids=(
                gold_mapping_by_candidate_id[
                    candidate_id
                ]
            ),
            k_values=(5, 10),
        )

        query_token_count = len(
            query_tokens
        )

        if query_token_count <= 5:
            query_length_slice = "short"
        elif query_token_count <= 12:
            query_length_slice = "medium"
        else:
            query_length_slice = "long"

        per_query_rows.append(
            {
                "candidate_id": (
                    candidate_id
                ),
                "source_types": sorted(
                    query_source_types(
                        candidate_id
                    )
                ),
                "query_token_count": (
                    query_token_count
                ),
                "query_length_slice": (
                    query_length_slice
                ),
                "contains_digit": bool(
                    re.search(
                        r"[0-9۰-۹٠-٩]",
                        question,
                    )
                ),
                "contains_latin": bool(
                    re.search(
                        r"[A-Za-z]",
                        question,
                    )
                ),
                "contains_zwnj": (
                    "\u200c" in question
                ),
                "has_high_idf_signal": (
                    high_idf_signal(
                        query_tokens,
                        index,
                        high_idf_threshold,
                    )
                ),
                "positive_result_count": len(
                    ranked_indexes
                ),
                "latency_ms": elapsed_ms,
                **metrics,
            }
        )

    assert len(per_query_rows) == len(
        development_candidate_id_list
    )

    configuration_query_results[
        configuration_id
    ] = per_query_rows


def aggregate_slice_rows(
    rows: Sequence[dict[str, Any]],
) -> dict[str, Any]:
    assert rows

    metric_names = (
        "hit_at_5",
        "hit_at_10",
        "recall_at_5",
        "recall_at_10",
        "precision_at_5",
        "precision_at_10",
        "ndcg_at_5",
        "ndcg_at_10",
        "reciprocal_rank",
    )

    latency_array = np.asarray(
        [
            row["latency_ms"]
            for row in rows
        ],
        dtype=float,
    )

    return {
        "query_count": len(rows),
        "metrics": {
            metric_name: float(
                np.mean(
                    [
                        row[metric_name]
                        for row in rows
                    ]
                )
            )
            for metric_name in metric_names
        },
        "zero_hit_at_10_count": sum(
            row["hit_at_10"] == 0.0
            for row in rows
        ),
        "latency_ms": {
            "median_p50": float(
                np.percentile(
                    latency_array,
                    50,
                )
            ),
            "p95": float(
                np.percentile(
                    latency_array,
                    95,
                )
            ),
        },
    }


selected_per_query_rows = (
    configuration_query_results[
        provisional_best_configuration_id
    ]
)

selected_slice_results = {
    "all_development": (
        aggregate_slice_rows(
            selected_per_query_rows
        )
    ),
    "source_type": {},
    "query_length": {},
    "features": {},
}

for source_type in (
    "book",
    "transcript",
    "website",
):
    slice_rows = [
        row
        for row in selected_per_query_rows
        if source_type in row[
            "source_types"
        ]
    ]

    assert slice_rows

    selected_slice_results[
        "source_type"
    ][source_type] = (
        aggregate_slice_rows(
            slice_rows
        )
    )

for query_length_slice in (
    "short",
    "medium",
    "long",
):
    slice_rows = [
        row
        for row in selected_per_query_rows
        if row[
            "query_length_slice"
        ] == query_length_slice
    ]

    assert slice_rows

    selected_slice_results[
        "query_length"
    ][query_length_slice] = (
        aggregate_slice_rows(
            slice_rows
        )
    )

feature_slice_definitions = {
    "contains_digit": (
        "contains_digit"
    ),
    "contains_latin": (
        "contains_latin"
    ),
    "contains_zwnj": (
        "contains_zwnj"
    ),
    "has_high_idf_signal": (
        "has_high_idf_signal"
    ),
}

for slice_name, field_name in (
    feature_slice_definitions.items()
):
    positive_rows = [
        row
        for row in selected_per_query_rows
        if row[field_name] is True
    ]

    negative_rows = [
        row
        for row in selected_per_query_rows
        if row[field_name] is False
    ]

    assert positive_rows
    assert negative_rows

    selected_slice_results[
        "features"
    ][slice_name] = {
        "true": aggregate_slice_rows(
            positive_rows
        ),
        "false": aggregate_slice_rows(
            negative_rows
        ),
    }


def paired_bootstrap_mean_delta(
    selected_values: np.ndarray,
    comparison_values: np.ndarray,
    iterations: int,
    seed: int,
) -> dict[str, float]:
    assert selected_values.shape == (
        comparison_values.shape
    )

    differences = (
        selected_values
        - comparison_values
    )

    random_generator = (
        np.random.default_rng(seed)
    )

    bootstrap_means = np.empty(
        iterations,
        dtype=np.float64,
    )

    sample_size = len(differences)

    for iteration in range(iterations):
        sample_indexes = (
            random_generator.integers(
                0,
                sample_size,
                size=sample_size,
            )
        )

        bootstrap_means[
            iteration
        ] = float(
            differences[
                sample_indexes
            ].mean()
        )

    return {
        "mean_delta": float(
            differences.mean()
        ),
        "ci95_lower": float(
            np.percentile(
                bootstrap_means,
                2.5,
            )
        ),
        "ci95_upper": float(
            np.percentile(
                bootstrap_means,
                97.5,
            )
        ),
        "selected_win_count": int(
            np.sum(differences > 0)
        ),
        "comparison_win_count": int(
            np.sum(differences < 0)
        ),
        "tie_count": int(
            np.sum(differences == 0)
        ),
    }


selected_rows_by_id = {
    row["candidate_id"]: row
    for row in selected_per_query_rows
}

paired_comparisons = {}

for configuration_id, comparison_rows in (
    configuration_query_results.items()
):
    if (
        configuration_id
        == provisional_best_configuration_id
    ):
        continue

    comparison_rows_by_id = {
        row["candidate_id"]: row
        for row in comparison_rows
    }

    assert set(
        comparison_rows_by_id
    ) == set(selected_rows_by_id)

    ordered_candidate_ids = (
        development_candidate_id_list
    )

    selected_ndcg = np.asarray(
        [
            selected_rows_by_id[
                candidate_id
            ]["ndcg_at_10"]
            for candidate_id
            in ordered_candidate_ids
        ],
        dtype=float,
    )

    comparison_ndcg = np.asarray(
        [
            comparison_rows_by_id[
                candidate_id
            ]["ndcg_at_10"]
            for candidate_id
            in ordered_candidate_ids
        ],
        dtype=float,
    )

    selected_mrr = np.asarray(
        [
            selected_rows_by_id[
                candidate_id
            ]["reciprocal_rank"]
            for candidate_id
            in ordered_candidate_ids
        ],
        dtype=float,
    )

    comparison_mrr = np.asarray(
        [
            comparison_rows_by_id[
                candidate_id
            ]["reciprocal_rank"]
            for candidate_id
            in ordered_candidate_ids
        ],
        dtype=float,
    )

    seed_offset = int(
        hashlib.sha256(
            configuration_id.encode(
                "utf-8"
            )
        ).hexdigest()[:8],
        16,
    )

    paired_comparisons[
        configuration_id
    ] = {
        "ndcg_at_10": (
            paired_bootstrap_mean_delta(
                selected_values=(
                    selected_ndcg
                ),
                comparison_values=(
                    comparison_ndcg
                ),
                iterations=(
                    BOOTSTRAP_ITERATIONS
                ),
                seed=(
                    BOOTSTRAP_SEED
                    + seed_offset
                ),
            )
        ),
        "reciprocal_rank": (
            paired_bootstrap_mean_delta(
                selected_values=(
                    selected_mrr
                ),
                comparison_values=(
                    comparison_mrr
                ),
                iterations=(
                    BOOTSTRAP_ITERATIONS
                ),
                seed=(
                    BOOTSTRAP_SEED
                    + seed_offset
                    + 1
                ),
            )
        ),
    }

development_failure_candidate_ids = [
    row["candidate_id"]
    for row in selected_per_query_rows
    if row["hit_at_10"] == 0.0
]

slice_stability_summary = {
    "provisional_best_configuration": (
        provisional_best_grid_row
    ),
    "best_configuration_by_analyzer": {
        analyzer_id: row
        for analyzer_id, row
        in sorted(
            best_row_by_analyzer.items()
        )
    },
    "analyzed_configuration_count": len(
        analysis_configuration_rows
    ),
    "analyzed_configuration_ids": [
        row["configuration_id"]
        for row in analysis_configuration_rows
    ],
    "high_idf_threshold": (
        high_idf_threshold
    ),
    "selected_configuration_slices": (
        selected_slice_results
    ),
    "paired_bootstrap": {
        "iterations": (
            BOOTSTRAP_ITERATIONS
        ),
        "metric_direction": (
            "selected_minus_comparison"
        ),
        "comparisons": (
            paired_comparisons
        ),
    },
    "development_failure_count": len(
        development_failure_candidate_ids
    ),
    "development_failure_candidate_id_sample": (
        development_failure_candidate_ids[
            :30
        ]
    ),
    "test_queries_accessed": False,
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "slice_stability_status": "PASS",
}

print(
    json.dumps(
        slice_stability_summary,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "provisional_best_configuration": {
    "configuration_id": "fa_join_zwnj_stopwords_v1__k1_1.20__b_0.30",
    "analyzer_id": "fa_join_zwnj_stopwords_v1",
    "k1": 1.2,
    "b": 0.3,
    "query_count": 2664,
    "zero_match_query_count": 0,
    "metrics": {
      "hit_at_10": 0.743993993993994,
      "hit_at_5": 0.6786786786786787,
      "ndcg_at_10": 0.50936500655395,
      "ndcg_at_5": 0.4855041074074556,
      "precision_at_10": 0.1001876876876877,
      "precision_at_5": 0.17432432432432432,
      "recall_at_10": 0.5796647838314506,
      "recall_at_5": 0.5128710555793888,
      "reciprocal_rank": 0.5861315247868337
    },
    "latency_ms": {
      "minimum": 0.09437199696549214,
      "median_p50": 0.3868549993057968,
      "p95": 0.7118573994375763,
      "maximum": 4.139079999731621
    },
    "positive_result_count": {
      "minimum": 5,
      "median": 1700.0,
      "p95": 2748.7,
      "maximum": 3236
    },
    "elapsed_seconds": 2.611030498999753,
    "status": "PASS",

<div dir="rtl" align="right">

## مرحله ۱۵ — Document Frequency، اصطلاحات دقیق و Failure Taxonomy

Sparse Retrieval باید مخصوصاً در بازیابی Tokenهای دقیق، نام‌ها، اعداد، مخفف‌ها و اصطلاحات نادر قابل‌اعتماد باشد. در این مرحله Configuration برتر با بهترین Analyzer بدون Stopword مقایسه می‌شود.

تحلیل شامل موارد زیر است:

- توزیع کامل Document Frequency؛
- Tokenهای Hapax و بسیار نادر؛
- OOV در Queryهای Development؛
- Queryهای دارای عدد؛
- Queryهای دارای حروف لاتین؛
- Queryهای دارای نیم‌فاصله؛
- Queryهای دارای Term نادر؛
- Queryهای دارای Term بسیار رایج؛
- میزان هم‌پوشانی Tokenهای Query با Relevant Chunkها؛
- آزمون مستقل بازیابی Tokenهای دقیق از Posting list؛
- طبقه‌بندی Failureهای بدون Hit@10.

Failureها در گروه‌های زیر قرار می‌گیرند:

- نبود هم‌پوشانی واژگانی با Gold Chunk؛
- غلبه Tokenهای OOV؛
- وجود هم‌پوشانی اما Rank پایین‌تر از ۱۰؛
- Query عمومی با Signal ضعیف؛
- Failure باقی‌مانده نیازمند بررسی بعدی.

این تحلیل فقط روی Development انجام می‌شود.

### معیار موفقیت مرحله

این مرحله زمانی PASS است که:

- Document Frequency تمام Vocabulary معتبر باشد؛
- Exact token probes بدون خطا بازیابی شوند؛
- Queryهای عددی، لاتین، نیم‌فاصله‌دار و نادر جداگانه ارزیابی شوند؛
- Failureهای Development طبقه‌بندی شوند؛
- دلیل انتخاب یا رد Stopword policy با Evidence روشن شود؛
- Test همچنان استفاده نشود.

</div>

In [16]:
# Code Cell 16 — Analyze document frequency, exact-term behavior and failures

selected_configuration_row = (
    provisional_best_grid_row
)

selected_configuration_id = (
    selected_configuration_row[
        "configuration_id"
    ]
)
selected_analyzer_id = (
    selected_configuration_row[
        "analyzer_id"
    ]
)
selected_k1 = float(
    selected_configuration_row["k1"]
)
selected_b = float(
    selected_configuration_row["b"]
)

selected_config = analyzer_by_id[
    selected_analyzer_id
]
selected_index = bm25_indexes[
    selected_analyzer_id
]

non_stopword_analyzer_ids = {
    config.analyzer_id
    for config in ANALYZER_CANDIDATES
    if not config.remove_stopwords
}

best_non_stopword_row = next(
    row
    for row in ranked_development_grid
    if row["analyzer_id"]
    in non_stopword_analyzer_ids
)

best_non_stopword_configuration_id = (
    best_non_stopword_row[
        "configuration_id"
    ]
)

assert (
    best_non_stopword_configuration_id
    in configuration_query_results
)

selected_rows = (
    configuration_query_results[
        selected_configuration_id
    ]
)
non_stopword_rows = (
    configuration_query_results[
        best_non_stopword_configuration_id
    ]
)

selected_rows_by_candidate_id = {
    row["candidate_id"]: row
    for row in selected_rows
}

non_stopword_rows_by_candidate_id = {
    row["candidate_id"]: row
    for row in non_stopword_rows
}

assert set(
    selected_rows_by_candidate_id
) == set(
    development_candidate_id_list
)
assert set(
    non_stopword_rows_by_candidate_id
) == set(
    development_candidate_id_list
)

selected_document_tokens = [
    analyze_persian_text(
        text,
        selected_config,
    )
    for text in corpus_texts
]

selected_document_token_sets = [
    set(tokens)
    for tokens in selected_document_tokens
]

assert all(
    selected_document_token_sets
)

document_frequency_values = np.asarray(
    list(
        selected_index
        .document_frequency
        .values()
    ),
    dtype=float,
)

assert np.all(document_frequency_values >= 1)
assert np.all(
    document_frequency_values
    <= len(chunk_records)
)

document_frequency_profile = {
    "vocabulary_size": (
        selected_index.vocabulary_size
    ),
    "minimum": int(
        document_frequency_values.min()
    ),
    "p25": float(
        np.percentile(
            document_frequency_values,
            25,
        )
    ),
    "median_p50": float(
        np.percentile(
            document_frequency_values,
            50,
        )
    ),
    "p75": float(
        np.percentile(
            document_frequency_values,
            75,
        )
    ),
    "p95": float(
        np.percentile(
            document_frequency_values,
            95,
        )
    ),
    "p99": float(
        np.percentile(
            document_frequency_values,
            99,
        )
    ),
    "maximum": int(
        document_frequency_values.max()
    ),
    "hapax_term_count": sum(
        value == 1
        for value in (
            selected_index
            .document_frequency
            .values()
        )
    ),
    "df_le_5_term_count": sum(
        value <= 5
        for value in (
            selected_index
            .document_frequency
            .values()
        )
    ),
    "df_le_20_term_count": sum(
        value <= 20
        for value in (
            selected_index
            .document_frequency
            .values()
        )
    ),
    "df_ge_half_corpus_term_count": sum(
        value
        >= len(chunk_records) / 2
        for value in (
            selected_index
            .document_frequency
            .values()
        )
    ),
}

selected_query_diagnostics = {}

for candidate_id in (
    development_candidate_id_list
):
    query_tokens = (
        development_query_tokens_by_analyzer[
            selected_analyzer_id
        ][candidate_id]
    )

    unique_query_tokens = set(
        query_tokens
    )

    in_vocabulary_tokens = {
        token
        for token in unique_query_tokens
        if token
        in selected_index.document_frequency
    }

    oov_tokens = (
        unique_query_tokens
        - in_vocabulary_tokens
    )

    rare_tokens = {
        token
        for token in in_vocabulary_tokens
        if selected_index.document_frequency[
            token
        ] <= 20
    }

    very_rare_tokens = {
        token
        for token in in_vocabulary_tokens
        if selected_index.document_frequency[
            token
        ] <= 5
    }

    idf_values = {
        token: math.log(
            1.0
            + (
                (
                    len(chunk_records)
                    - selected_index
                    .document_frequency[token]
                    + 0.5
                )
                / (
                    selected_index
                    .document_frequency[token]
                    + 0.5
                )
            )
        )
        for token in in_vocabulary_tokens
    }

    relevant_chunk_ids = (
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    relevant_chunk_indexes = [
        chunk_index_by_id[
            chunk_id
        ]
        for chunk_id
        in relevant_chunk_ids
    ]

    overlap_counts = [
        len(
            unique_query_tokens
            & selected_document_token_sets[
                document_index
            ]
        )
        for document_index
        in relevant_chunk_indexes
    ]

    maximum_gold_overlap = (
        max(overlap_counts)
        if overlap_counts
        else 0
    )

    maximum_idf = (
        max(idf_values.values())
        if idf_values
        else 0.0
    )

    question = (
        golden_record_by_candidate_id[
            candidate_id
        ]["evaluation_content"][
            "question"
        ]
    )

    selected_query_diagnostics[
        candidate_id
    ] = {
        "query_token_count": len(
            query_tokens
        ),
        "unique_query_token_count": len(
            unique_query_tokens
        ),
        "in_vocabulary_token_count": len(
            in_vocabulary_tokens
        ),
        "oov_token_count": len(
            oov_tokens
        ),
        "oov_ratio": (
            len(oov_tokens)
            / len(unique_query_tokens)
            if unique_query_tokens
            else 1.0
        ),
        "rare_token_count_df_le_20": len(
            rare_tokens
        ),
        "very_rare_token_count_df_le_5": (
            len(very_rare_tokens)
        ),
        "maximum_idf": (
            maximum_idf
        ),
        "maximum_gold_token_overlap": (
            maximum_gold_overlap
        ),
        "contains_digit": bool(
            re.search(
                r"[0-9۰-۹٠-٩]",
                question,
            )
        ),
        "contains_latin": bool(
            re.search(
                r"[A-Za-z]",
                question,
            )
        ),
        "contains_zwnj": (
            "\u200c" in question
        ),
    }

slice_predicates = {
    "contains_digit": (
        lambda diagnostic:
        diagnostic["contains_digit"]
    ),
    "contains_latin": (
        lambda diagnostic:
        diagnostic["contains_latin"]
    ),
    "contains_zwnj": (
        lambda diagnostic:
        diagnostic["contains_zwnj"]
    ),
    "has_rare_term_df_le_20": (
        lambda diagnostic:
        diagnostic[
            "rare_token_count_df_le_20"
        ] > 0
    ),
    "has_very_rare_term_df_le_5": (
        lambda diagnostic:
        diagnostic[
            "very_rare_token_count_df_le_5"
        ] > 0
    ),
    "has_any_oov": (
        lambda diagnostic:
        diagnostic[
            "oov_token_count"
        ] > 0
    ),
    "oov_majority": (
        lambda diagnostic:
        diagnostic["oov_ratio"] > 0.5
    ),
    "no_gold_lexical_overlap": (
        lambda diagnostic:
        diagnostic[
            "maximum_gold_token_overlap"
        ] == 0
    ),
}

exact_term_slice_comparison = {}

for slice_name, predicate in (
    slice_predicates.items()
):
    slice_candidate_ids = [
        candidate_id
        for candidate_id, diagnostic
        in selected_query_diagnostics.items()
        if predicate(diagnostic)
    ]

    if not slice_candidate_ids:
        exact_term_slice_comparison[
            slice_name
        ] = {
            "query_count": 0,
            "status": "EMPTY_SLICE",
        }
        continue

    selected_slice_rows = [
        selected_rows_by_candidate_id[
            candidate_id
        ]
        for candidate_id
        in slice_candidate_ids
    ]

    non_stopword_slice_rows = [
        non_stopword_rows_by_candidate_id[
            candidate_id
        ]
        for candidate_id
        in slice_candidate_ids
    ]

    selected_aggregate = (
        aggregate_slice_rows(
            selected_slice_rows
        )
    )
    non_stopword_aggregate = (
        aggregate_slice_rows(
            non_stopword_slice_rows
        )
    )

    exact_term_slice_comparison[
        slice_name
    ] = {
        "query_count": len(
            slice_candidate_ids
        ),
        "selected": selected_aggregate,
        "best_non_stopword": (
            non_stopword_aggregate
        ),
        "delta_selected_minus_non_stopword": {
            metric_name: (
                selected_aggregate[
                    "metrics"
                ][metric_name]
                - non_stopword_aggregate[
                    "metrics"
                ][metric_name]
            )
            for metric_name in (
                "hit_at_10",
                "recall_at_10",
                "ndcg_at_10",
                "reciprocal_rank",
            )
        },
        "status": "PASS",
    }

# Exact posting probes test whether rare Persian,
# numeric and Latin tokens retrieve their posting set.
rare_persian_terms = []
rare_numeric_terms = []
rare_latin_terms = []

for term, document_frequency in sorted(
    selected_index.document_frequency.items(),
    key=lambda item: (
        item[1],
        item[0],
    ),
):
    if document_frequency > 5:
        continue

    if re.fullmatch(
        r"[0-9]+(?:\.[0-9]+)?",
        term,
    ):
        rare_numeric_terms.append(term)
    elif re.search(r"[a-z]", term):
        rare_latin_terms.append(term)
    elif re.search(r"[\u0600-\u06ff]", term):
        rare_persian_terms.append(term)

EXACT_PROBE_LIMIT_PER_TYPE = 100

exact_probe_terms = {
    "persian": rare_persian_terms[
        :EXACT_PROBE_LIMIT_PER_TYPE
    ],
    "numeric": rare_numeric_terms[
        :EXACT_PROBE_LIMIT_PER_TYPE
    ],
    "latin": rare_latin_terms[
        :EXACT_PROBE_LIMIT_PER_TYPE
    ],
}

exact_probe_results = {}

for probe_type, terms in (
    exact_probe_terms.items()
):
    probe_rows = []

    for term in terms:
        expected_document_indexes = set(
            selected_index.postings[
                term
            ][0].tolist()
        )

        scores = bm25_score_query(
            index=selected_index,
            query_tokens=[term],
            k1=selected_k1,
            b=selected_b,
        )

        ranked_indexes = (
            ranked_positive_document_indexes(
                scores
            )
        )

        returned_positive_indexes = set(
            ranked_indexes.tolist()
        )

        assert returned_positive_indexes == (
            expected_document_indexes
        )

        top_indexes = set(
            ranked_indexes[
                :min(
                    10,
                    len(ranked_indexes),
                )
            ].tolist()
        )

        probe_rows.append(
            {
                "term": term,
                "document_frequency": len(
                    expected_document_indexes
                ),
                "all_positive_results_exact": (
                    returned_positive_indexes
                    == expected_document_indexes
                ),
                "top_result_in_posting_set": (
                    bool(
                        top_indexes
                        & expected_document_indexes
                    )
                ),
            }
        )

    exact_probe_results[
        probe_type
    ] = {
        "probe_count": len(
            probe_rows
        ),
        "all_passed": all(
            row[
                "all_positive_results_exact"
            ]
            and row[
                "top_result_in_posting_set"
            ]
            for row in probe_rows
        ),
        "document_frequency_distribution": dict(
            sorted(
                Counter(
                    row[
                        "document_frequency"
                    ]
                    for row in probe_rows
                ).items()
            )
        ),
        "status": (
            "PASS"
            if probe_rows
            and all(
                row[
                    "all_positive_results_exact"
                ]
                and row[
                    "top_result_in_posting_set"
                ]
                for row in probe_rows
            )
            else "NO_PROBES"
        ),
    }

assert exact_probe_results[
    "persian"
]["status"] == "PASS"
assert exact_probe_results[
    "numeric"
]["status"] == "PASS"
assert exact_probe_results[
    "latin"
]["status"] == "PASS"

failure_taxonomy_counts: Counter[str] = (
    Counter()
)
failure_taxonomy_by_source: dict[
    str,
    Counter[str],
] = defaultdict(Counter)
failure_examples: dict[
    str,
    list[str],
] = defaultdict(list)

for candidate_id in (
    development_candidate_id_list
):
    result = selected_rows_by_candidate_id[
        candidate_id
    ]
    diagnostic = (
        selected_query_diagnostics[
            candidate_id
        ]
    )

    if result["hit_at_10"] > 0:
        failure_class = "SUCCESS"
    elif (
        diagnostic[
            "maximum_gold_token_overlap"
        ]
        == 0
    ):
        failure_class = (
            "NO_QUERY_GOLD_LEXICAL_OVERLAP"
        )
    elif diagnostic["oov_ratio"] > 0.5:
        failure_class = "OOV_DOMINANT"
    elif (
        diagnostic[
            "maximum_gold_token_overlap"
        ]
        > 0
    ):
        failure_class = (
            "LEXICAL_OVERLAP_RANK_BELOW_10"
        )
    elif (
        diagnostic["maximum_idf"]
        < high_idf_threshold
    ):
        failure_class = (
            "LOW_INFORMATION_QUERY"
        )
    else:
        failure_class = (
            "UNRESOLVED_RETRIEVAL_FAILURE"
        )

    failure_taxonomy_counts[
        failure_class
    ] += 1

    for source_type in query_source_types(
        candidate_id
    ):
        failure_taxonomy_by_source[
            source_type
        ][failure_class] += 1

    if (
        failure_class != "SUCCESS"
        and len(
            failure_examples[
                failure_class
            ]
        )
        < 20
    ):
        failure_examples[
            failure_class
        ].append(candidate_id)

assert sum(
    failure_taxonomy_counts.values()
) == len(
    development_candidate_id_list
)

document_frequency_and_failure_summary = {
    "selected_configuration": (
        selected_configuration_row
    ),
    "best_non_stopword_configuration": (
        best_non_stopword_row
    ),
    "document_frequency_profile": (
        document_frequency_profile
    ),
    "exact_term_slice_comparison": (
        exact_term_slice_comparison
    ),
    "exact_posting_probes": (
        exact_probe_results
    ),
    "failure_taxonomy": {
        "counts": dict(
            sorted(
                failure_taxonomy_counts.items()
            )
        ),
        "by_source_type": {
            source_type: dict(
                sorted(counts.items())
            )
            for source_type, counts
            in sorted(
                failure_taxonomy_by_source.items()
            )
        },
        "candidate_id_examples": {
            failure_class: examples
            for failure_class, examples
            in sorted(
                failure_examples.items()
            )
        },
    },
    "test_queries_accessed": False,
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "document_frequency_and_failure_status": (
        "PASS"
    ),
}

print(
    json.dumps(
        document_frequency_and_failure_summary,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "selected_configuration": {
    "configuration_id": "fa_join_zwnj_stopwords_v1__k1_1.20__b_0.30",
    "analyzer_id": "fa_join_zwnj_stopwords_v1",
    "k1": 1.2,
    "b": 0.3,
    "query_count": 2664,
    "zero_match_query_count": 0,
    "metrics": {
      "hit_at_10": 0.743993993993994,
      "hit_at_5": 0.6786786786786787,
      "ndcg_at_10": 0.50936500655395,
      "ndcg_at_5": 0.4855041074074556,
      "precision_at_10": 0.1001876876876877,
      "precision_at_5": 0.17432432432432432,
      "recall_at_10": 0.5796647838314506,
      "recall_at_5": 0.5128710555793888,
      "reciprocal_rank": 0.5861315247868337
    },
    "latency_ms": {
      "minimum": 0.09437199696549214,
      "median_p50": 0.3868549993057968,
      "p95": 0.7118573994375763,
      "maximum": 4.139079999731621
    },
    "positive_result_count": {
      "minimum": 5,
      "median": 1700.0,
      "p95": 2748.7,
      "maximum": 3236
    },
    "elapsed_seconds": 2.611030498999753,
    "status": "PASS",
    "de

<div dir="rtl" align="right">

## ارزیابی سیاست محافظه‌کارانه Stopword و تصمیم نهایی Analyzer

نتایج تحلیل خطا نشان داد Analyzer مبتنی بر حذف Stopword در معیارهای کلی Development بهترین عملکرد را دارد، اما در برخی Sliceهای حساس به Exact Term، به‌ویژه Queryهای دارای OOV، عدد و واژه لاتین، افت محدودی نسبت به بهترین Analyzer بدون حذف Stopword مشاهده می‌شود.

در این مرحله یک سیاست محافظه‌کارانه و از پیش تعریف‌شده آزمایش می‌شود:

- فقط واژه‌هایی از فهرست Stopword حذف می‌شوند که در حداقل ۲۵٪ اسناد Corpus حضور داشته باشند؛
- فهرست حاصل فقط از Corpus و بدون مشاهده Test ساخته می‌شود؛
- تمام Queryهای Test همچنان قفل باقی می‌مانند؛
- پیکربندی‌های `k1` و `b` همان Grid قبلی هستند؛
- مقایسه با Analyzer منتخب با معیار اصلی `nDCG@10`، معیار `MRR`، Bootstrap زوجی و Sliceهای Exact-Term انجام می‌شود؛
- نسخه محافظه‌کارانه فقط در صورت بهبود معنادار معیار اصلی و نداشتن افت جدی در Sliceهای حساس می‌تواند جایگزین نسخه فعلی شود.

هدف این سل جلوگیری از حذف بیش‌ازحد واژه‌ها و ثبت یک تصمیم مستند پیش از Freeze پیکربندی Sparse است.

</div>

In [20]:
# Code Cell 17 — Conservative stopword policy evaluation and analyzer decision

import collections
import copy
import re
import time

CONSERVATIVE_ANALYZER_ID = "fa_join_zwnj_high_df_stopwords_v1"
CONSERVATIVE_STOPWORD_DF_RATIO = 0.25
EXACT_SLICE_MAX_ALLOWED_DROP = 0.01
BOOTSTRAP_SAMPLE_COUNT = 2000
BOOTSTRAP_SEED = 15017

join_surface_config = analyzer_by_id[
    "fa_surface_join_zwnj_v1"
]
join_surface_index = bm25_indexes[
    "fa_surface_join_zwnj_v1"
]

canonical_document_count = len(
    join_surface_index.document_ids
)

assert canonical_document_count == len(chunk_records)
assert len(
    join_surface_index.document_lengths
) == canonical_document_count

conservative_stopwords = frozenset(
    token
    for token in PERSIAN_STOPWORDS_V1
    if (
        join_surface_index.document_frequency.get(
            token,
            0,
        )
        / canonical_document_count
    )
    >= CONSERVATIVE_STOPWORD_DF_RATIO
)

retained_original_stopwords = frozenset(
    PERSIAN_STOPWORDS_V1
    - conservative_stopwords
)

assert conservative_stopwords
assert retained_original_stopwords
assert conservative_stopwords.issubset(
    PERSIAN_STOPWORDS_V1
)
assert not (
    conservative_stopwords
    & retained_original_stopwords
)
assert (
    conservative_stopwords
    | retained_original_stopwords
) == PERSIAN_STOPWORDS_V1


def analyze_conservative_stopwords(text):
    surface_tokens = analyze_persian_text(
        text=text,
        config=join_surface_config,
    )

    return [
        token
        for token in surface_tokens
        if token not in conservative_stopwords
    ]


def build_conservative_bm25_index(
    document_tokens,
):
    assert len(
        document_tokens
    ) == canonical_document_count
    assert all(document_tokens)

    document_lengths = np.asarray(
        [
            len(tokens)
            for tokens in document_tokens
        ],
        dtype=np.float64,
    )

    posting_accumulator = (
        collections.defaultdict(
            lambda: [[], []]
        )
    )

    document_frequency = collections.Counter()

    for document_index, tokens in enumerate(
        document_tokens
    ):
        term_frequency = collections.Counter(tokens)

        for token, frequency in (
            term_frequency.items()
        ):
            posting_accumulator[token][0].append(
                int(document_index)
            )
            posting_accumulator[token][1].append(
                float(frequency)
            )
            document_frequency[token] += 1

    postings = {
        token: (
            np.asarray(
                posting_values[0],
                dtype=np.int64,
            ),
            np.asarray(
                posting_values[1],
                dtype=np.float64,
            ),
        )
        for token, posting_values
        in posting_accumulator.items()
    }

    # Copy a known-valid index so every field expected by
    # bm25_score_query is retained.
    index = copy.copy(join_surface_index)

    index.analyzer_id = CONSERVATIVE_ANALYZER_ID
    index.document_lengths = document_lengths
    index.average_document_length = float(
        document_lengths.mean()
    )
    index.document_frequency = dict(
        document_frequency
    )
    index.postings = postings

    assert len(
        index.document_ids
    ) == canonical_document_count
    assert len(
        index.document_lengths
    ) == canonical_document_count
    assert index.average_document_length > 0
    assert index.document_frequency
    assert index.postings

    sample_posting = next(
        iter(index.postings.values())
    )

    assert isinstance(sample_posting, tuple)
    assert len(sample_posting) == 2
    assert isinstance(
        sample_posting[0],
        np.ndarray,
    )
    assert isinstance(
        sample_posting[1],
        np.ndarray,
    )
    assert len(
        sample_posting[0]
    ) == len(sample_posting[1])

    return index


development_candidate_id_set = set(
    development_candidate_ids
)

development_query_by_candidate_id = {
    record["candidate_id"]: record[
        "evaluation_content"
    ]["question"]
    for record in golden_records
    if record["candidate_id"]
    in development_candidate_id_set
}

assert set(
    development_query_by_candidate_id
) == development_candidate_id_set

assert all(
    isinstance(question, str)
    and question.strip()
    for question
    in development_query_by_candidate_id.values()
)

conservative_build_started = (
    time.perf_counter()
)

conservative_document_tokens = [
    analyze_conservative_stopwords(
        record["content"]
    )
    for record in chunk_records
]

assert len(
    conservative_document_tokens
) == canonical_document_count
assert all(conservative_document_tokens)

conservative_index = (
    build_conservative_bm25_index(
        document_tokens=(
            conservative_document_tokens
        )
    )
)

conservative_build_seconds = (
    time.perf_counter()
    - conservative_build_started
)

conservative_development_query_tokens = {
    candidate_id: analyze_conservative_stopwords(
        development_query_by_candidate_id[
            candidate_id
        ]
    )
    for candidate_id in development_candidate_ids
}

assert all(
    conservative_development_query_tokens[
        candidate_id
    ]
    for candidate_id in development_candidate_ids
)


def deterministic_positive_top_k(
    scores,
    top_k,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    positive_document_indexes = np.flatnonzero(
        scores > 0.0
    )

    if not positive_document_indexes.size:
        return []

    deterministic_order = np.lexsort(
        (
            positive_document_indexes,
            -scores[positive_document_indexes],
        )
    )

    return (
        positive_document_indexes[
            deterministic_order[:top_k]
        ].tolist()
    )


def evaluate_conservative_configuration(
    configuration_id,
    k1,
    b,
):
    evaluation_rows = []
    query_latencies_ms = []

    for candidate_id in development_candidate_ids:
        query_tokens = (
            conservative_development_query_tokens[
                candidate_id
            ]
        )

        scoring_started = time.perf_counter()

        scores = bm25_score_query(
            index=conservative_index,
            query_tokens=query_tokens,
            k1=k1,
            b=b,
        )

        ranked_document_indexes = (
            deterministic_positive_top_k(
                scores=scores,
                top_k=10,
            )
        )

        query_latencies_ms.append(
            (
                time.perf_counter()
                - scoring_started
            )
            * 1000.0
        )

        ranked_chunk_ids = [
            conservative_index.document_ids[
                document_index
            ]
            for document_index
            in ranked_document_indexes
        ]

        relevant_chunk_ids = set(
            gold_mapping_by_candidate_id[
                candidate_id
            ]
        )

        query_metrics = evaluate_ranked_result(
        ranked_chunk_ids,
        relevant_chunk_ids,
        )

        evaluation_rows.append(
            {
                "candidate_id": candidate_id,
                "configuration_id": (
                    configuration_id
                ),
                "analyzer_id": (
                    CONSERVATIVE_ANALYZER_ID
                ),
                "ranked_chunk_ids": (
                    ranked_chunk_ids
                ),
                "relevant_chunk_ids": sorted(
                    relevant_chunk_ids
                ),
                **query_metrics,
            }
        )

    metric_names = (
        "hit_at_10",
        "hit_at_5",
        "ndcg_at_10",
        "ndcg_at_5",
        "precision_at_10",
        "precision_at_5",
        "recall_at_10",
        "recall_at_5",
        "reciprocal_rank",
    )

    aggregate_metrics = {
        metric_name: float(
            np.mean(
                [
                    row[metric_name]
                    for row in evaluation_rows
                ]
            )
        )
        for metric_name in metric_names
    }

    latency_array = np.asarray(
        query_latencies_ms,
        dtype=np.float64,
    )

    configuration_summary = {
        "configuration_id": configuration_id,
        "analyzer_id": (
            CONSERVATIVE_ANALYZER_ID
        ),
        "k1": float(k1),
        "b": float(b),
        "query_count": len(evaluation_rows),
        "metrics": aggregate_metrics,
        "latency_ms": {
            "minimum": float(
                latency_array.min()
            ),
            "median_p50": float(
                np.median(latency_array)
            ),
            "p95": float(
                np.percentile(
                    latency_array,
                    95,
                )
            ),
            "maximum": float(
                latency_array.max()
            ),
            "scope": (
                "LOCAL_DEVELOPMENT_NOT_PRODUCTION_SLO"
            ),
        },
        "status": "PASS",
    }

    return (
        configuration_summary,
        evaluation_rows,
    )


conservative_grid_summaries = []
conservative_grid_query_results = {}

for k1 in (0.8, 1.2, 1.6):
    for b in (0.3, 0.75):
        configuration_id = (
            f"{CONSERVATIVE_ANALYZER_ID}"
            f"__k1_{k1:.2f}"
            f"__b_{b:.2f}"
        )

        (
            configuration_summary,
            configuration_rows,
        ) = evaluate_conservative_configuration(
            configuration_id=configuration_id,
            k1=k1,
            b=b,
        )

        conservative_grid_summaries.append(
            configuration_summary
        )

        conservative_grid_query_results[
            configuration_id
        ] = configuration_rows

conservative_grid_summaries = sorted(
    conservative_grid_summaries,
    key=lambda row: (
        -row["metrics"]["ndcg_at_10"],
        -row["metrics"]["reciprocal_rank"],
        -row["metrics"]["hit_at_10"],
        -row["metrics"]["recall_at_10"],
        row["latency_ms"]["median_p50"],
        row["configuration_id"],
    ),
)

best_conservative_summary = (
    conservative_grid_summaries[0]
)

best_conservative_configuration_id = (
    best_conservative_summary[
        "configuration_id"
    ]
)

best_conservative_query_rows = (
    conservative_grid_query_results[
        best_conservative_configuration_id
    ]
)

best_conservative_rows_by_candidate_id = {
    row["candidate_id"]: row
    for row in best_conservative_query_rows
}

assert set(
    best_conservative_rows_by_candidate_id
) == development_candidate_id_set

assert set(
    selected_rows_by_candidate_id
) == development_candidate_id_set


def paired_bootstrap_comparison(
    candidate_rows_by_id,
    baseline_rows_by_id,
    candidate_ids,
    metric_name,
    seed,
):
    ordered_candidate_ids = sorted(
        candidate_ids
    )

    candidate_values = np.asarray(
        [
            candidate_rows_by_id[
                candidate_id
            ][metric_name]
            for candidate_id
            in ordered_candidate_ids
        ],
        dtype=np.float64,
    )

    baseline_values = np.asarray(
        [
            baseline_rows_by_id[
                candidate_id
            ][metric_name]
            for candidate_id
            in ordered_candidate_ids
        ],
        dtype=np.float64,
    )

    paired_deltas = (
        candidate_values
        - baseline_values
    )

    random_generator = (
        np.random.default_rng(seed)
    )

    bootstrap_means = np.empty(
        BOOTSTRAP_SAMPLE_COUNT,
        dtype=np.float64,
    )

    item_count = len(paired_deltas)

    for sample_index in range(
        BOOTSTRAP_SAMPLE_COUNT
    ):
        sampled_indexes = (
            random_generator.integers(
                low=0,
                high=item_count,
                size=item_count,
            )
        )

        bootstrap_means[
            sample_index
        ] = float(
            paired_deltas[
                sampled_indexes
            ].mean()
        )

    return {
        "metric": metric_name,
        "query_count": item_count,
        "candidate_mean": float(
            candidate_values.mean()
        ),
        "baseline_mean": float(
            baseline_values.mean()
        ),
        "observed_delta": float(
            paired_deltas.mean()
        ),
        "confidence_interval_95": {
            "lower": float(
                np.percentile(
                    bootstrap_means,
                    2.5,
                )
            ),
            "upper": float(
                np.percentile(
                    bootstrap_means,
                    97.5,
                )
            ),
        },
        "bootstrap_sample_count": (
            BOOTSTRAP_SAMPLE_COUNT
        ),
        "seed": seed,
    }


overall_ndcg_comparison = (
    paired_bootstrap_comparison(
        candidate_rows_by_id=(
            best_conservative_rows_by_candidate_id
        ),
        baseline_rows_by_id=(
            selected_rows_by_candidate_id
        ),
        candidate_ids=(
            development_candidate_ids
        ),
        metric_name="ndcg_at_10",
        seed=BOOTSTRAP_SEED,
    )
)

overall_mrr_comparison = (
    paired_bootstrap_comparison(
        candidate_rows_by_id=(
            best_conservative_rows_by_candidate_id
        ),
        baseline_rows_by_id=(
            selected_rows_by_candidate_id
        ),
        candidate_ids=(
            development_candidate_ids
        ),
        metric_name="reciprocal_rank",
        seed=BOOTSTRAP_SEED + 1,
    )
)

digit_pattern = re.compile(
    r"[0-9۰-۹]"
)
latin_pattern = re.compile(
    r"[A-Za-z]"
)

exact_slice_candidate_ids = {
    "numeric": [
        candidate_id
        for candidate_id
        in development_candidate_ids
        if digit_pattern.search(
            development_query_by_candidate_id[
                candidate_id
            ]
        )
    ],
    "latin": [
        candidate_id
        for candidate_id
        in development_candidate_ids
        if latin_pattern.search(
            development_query_by_candidate_id[
                candidate_id
            ]
        )
    ],
    "zwnj": [
        candidate_id
        for candidate_id
        in development_candidate_ids
        if "\u200c"
        in development_query_by_candidate_id[
            candidate_id
        ]
    ],
    "oov": [
        candidate_id
        for candidate_id
        in development_candidate_ids
        if any(
            token
            not in (
                join_surface_index
                .document_frequency
            )
            for token in analyze_persian_text(
                text=(
                    development_query_by_candidate_id[
                        candidate_id
                    ]
                ),
                config=join_surface_config,
            )
        )
    ],
}

assert all(
    exact_slice_candidate_ids.values()
)

exact_slice_comparisons = {}

for slice_index, (
    slice_name,
    slice_candidate_ids,
) in enumerate(
    exact_slice_candidate_ids.items()
):
    exact_slice_comparisons[
        slice_name
    ] = paired_bootstrap_comparison(
        candidate_rows_by_id=(
            best_conservative_rows_by_candidate_id
        ),
        baseline_rows_by_id=(
            selected_rows_by_candidate_id
        ),
        candidate_ids=slice_candidate_ids,
        metric_name="ndcg_at_10",
        seed=(
            BOOTSTRAP_SEED
            + 10
            + slice_index
        ),
    )

overall_delta_gate = (
    overall_ndcg_comparison[
        "observed_delta"
    ]
    >= 0.0
)

statistical_promotion_gate = (
    overall_ndcg_comparison[
        "confidence_interval_95"
    ]["lower"]
    >= 0.0
)

exact_slice_drop_gate = all(
    comparison["observed_delta"]
    >= -EXACT_SLICE_MAX_ALLOWED_DROP
    for comparison
    in exact_slice_comparisons.values()
)

promote_conservative_analyzer = all(
    [
        overall_delta_gate,
        statistical_promotion_gate,
        exact_slice_drop_gate,
    ]
)

if promote_conservative_analyzer:
    sparse_analyzer_decision = {
        "decision": (
            "PROMOTE_CONSERVATIVE_HIGH_DF_STOPWORDS"
        ),
        "selected_configuration_id": (
            best_conservative_configuration_id
        ),
        "selected_analyzer_id": (
            CONSERVATIVE_ANALYZER_ID
        ),
        "selected_k1": float(
            best_conservative_summary["k1"]
        ),
        "selected_b": float(
            best_conservative_summary["b"]
        ),
        "reason": (
            "The conservative policy passed "
            "the overall, bootstrap and "
            "exact-slice promotion gates."
        ),
    }
else:
    sparse_analyzer_decision = {
        "decision": (
            "RETAIN_ORIGINAL_SELECTED_CONFIGURATION"
        ),
        "selected_configuration_id": (
            selected_configuration_id
        ),
        "selected_analyzer_id": (
            selected_analyzer_id
        ),
        "selected_k1": float(
            selected_k1
        ),
        "selected_b": float(
            selected_b
        ),
        "reason": (
            "The conservative policy did not "
            "pass every predefined promotion gate."
        ),
    }

conservative_stopword_evaluation = {
    "scope": {
        "corpus": "ALL_CANONICAL_CHUNKS",
        "queries": "DEVELOPMENT_ONLY",
        "test_queries_accessed": False,
    },
    "policy": {
        "candidate_analyzer_id": (
            CONSERVATIVE_ANALYZER_ID
        ),
        "document_frequency_ratio_threshold": (
            CONSERVATIVE_STOPWORD_DF_RATIO
        ),
        "original_stopword_count": len(
            PERSIAN_STOPWORDS_V1
        ),
        "removed_high_df_stopword_count": len(
            conservative_stopwords
        ),
        "retained_original_stopword_count": len(
            retained_original_stopwords
        ),
        "removed_high_df_stopwords": sorted(
            conservative_stopwords
        ),
        "retained_original_stopwords": sorted(
            retained_original_stopwords
        ),
    },
    "index": {
        "document_count": len(
            conservative_index.document_ids
        ),
        "vocabulary_size": len(
            conservative_index.document_frequency
        ),
        "total_token_count": int(
            conservative_index
            .document_lengths
            .sum()
        ),
        "average_document_length": float(
            conservative_index
            .average_document_length
        ),
        "build_seconds": float(
            conservative_build_seconds
        ),
    },
    "grid": {
        "configuration_count": len(
            conservative_grid_summaries
        ),
        "best_configuration": (
            best_conservative_summary
        ),
        "ranked_configurations": (
            conservative_grid_summaries
        ),
    },
    "comparison_with_previous_selection": {
        "previous_configuration_id": (
            selected_configuration_id
        ),
        "candidate_configuration_id": (
            best_conservative_configuration_id
        ),
        "overall_ndcg_at_10": (
            overall_ndcg_comparison
        ),
        "overall_reciprocal_rank": (
            overall_mrr_comparison
        ),
        "exact_slices_ndcg_at_10": (
            exact_slice_comparisons
        ),
    },
    "promotion_gates": {
        "overall_non_negative_delta": (
            overall_delta_gate
        ),
        "bootstrap_lower_bound_non_negative": (
            statistical_promotion_gate
        ),
        "maximum_allowed_exact_slice_drop": (
            EXACT_SLICE_MAX_ALLOWED_DROP
        ),
        "exact_slice_drop_gate": (
            exact_slice_drop_gate
        ),
        "all_gates_passed": (
            promote_conservative_analyzer
        ),
    },
    "decision": sparse_analyzer_decision,
    "files_created": 0,
    "repository_mutated": False,
    "paid_api_calls": 0,
    "conservative_stopword_evaluation_status": (
        "PASS"
    ),
}

assert (
    conservative_stopword_evaluation[
        "scope"
    ]["test_queries_accessed"]
    is False
)
assert (
    conservative_stopword_evaluation[
        "index"
    ]["document_count"]
    == len(chunk_records)
)
assert (
    conservative_stopword_evaluation[
        "grid"
    ]["configuration_count"]
    == 6
)
assert (
    conservative_stopword_evaluation[
        "conservative_stopword_evaluation_status"
    ]
    == "PASS"
)

print(
    json.dumps(
        conservative_stopword_evaluation,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "scope": {
    "corpus": "ALL_CANONICAL_CHUNKS",
    "queries": "DEVELOPMENT_ONLY",
    "test_queries_accessed": false
  },
  "policy": {
    "candidate_analyzer_id": "fa_join_zwnj_high_df_stopwords_v1",
    "document_frequency_ratio_threshold": 0.25,
    "original_stopword_count": 28,
    "removed_high_df_stopword_count": 19,
    "retained_original_stopword_count": 9,
    "removed_high_df_stopwords": [
      "آن",
      "از",
      "است",
      "اما",
      "اگر",
      "این",
      "با",
      "بر",
      "برای",
      "به",
      "تا",
      "در",
      "را",
      "شده",
      "و",
      "چه",
      "که",
      "یا",
      "یک"
    ],
    "retained_original_stopwords": [
      "آیا",
      "باشد",
      "بود",
      "شود",
      "می",
      "هست",
      "چرا",
      "چگونه",
      "کدام"
    ]
  },
  "index": {
    "document_count": 3488,
    "vocabulary_size": 27320,
    "total_token_count": 538042,
    "average_document_length": 154.25516055045873,
    "build_seconds": 2.8319

<div dir="rtl" align="right">

## فریز پیکربندی نهایی Sparse Retrieval پیش از بازکردن Test Split

آزمایش‌های Development نشان دادند که حذف Stopword با سیاست اولیه، نسبت به سیاست محافظه‌کارانه مبتنی بر Document Frequency عملکرد بهتری دارد.

پیکربندی منتخب پیش از هرگونه دسترسی به Test Split به‌صورت رسمی فریز می‌شود:

- Analyzer: `fa_join_zwnj_stopwords_v1`
- سیاست ZWNJ: اتصال اجزای دارای نیم‌فاصله
- سیاست رقم‌ها: یکسان‌سازی رقم‌های فارسی و عربی با رقم ASCII
- سیاست حروف: یکسان‌سازی گونه‌های عربی و فارسی
- Stopword policy: فهرست ثابت نسخه اول
- BM25 implementation: `local_robertson_bm25_v1`
- `k1 = 1.2`
- `b = 0.3`
- معیار اصلی انتخاب: `nDCG@10`
- داده انتخاب پیکربندی: فقط Development Split
- Test Split: تا پیش از تکمیل این سل استفاده نشده است

آزمایش سیاست محافظه‌کارانه نیز ثبت شد اما به دلیل افت معنادار عملکرد کلی و عبورنکردن از تمام Gateهای ازپیش‌تعریف‌شده رد شد.

این سل یک قرارداد قطعی و Hash‌شده از پیکربندی نهایی می‌سازد. پس از ایجاد این قرارداد، Analyzer، Stopwordها و پارامترهای BM25 دیگر براساس نتیجه Test تغییر نخواهند کرد.

</div>

In [21]:
# Code Cell 18 — Freeze the final sparse retrieval configuration

import dataclasses

assert (
    conservative_stopword_evaluation[
        "conservative_stopword_evaluation_status"
    ]
    == "PASS"
)

assert (
    sparse_analyzer_decision["decision"]
    == "RETAIN_ORIGINAL_SELECTED_CONFIGURATION"
)

assert (
    sparse_analyzer_decision[
        "selected_configuration_id"
    ]
    == selected_configuration_id
)

assert (
    sparse_analyzer_decision[
        "selected_analyzer_id"
    ]
    == selected_analyzer_id
)

assert float(
    sparse_analyzer_decision["selected_k1"]
) == float(selected_k1)

assert float(
    sparse_analyzer_decision["selected_b"]
) == float(selected_b)

assert selected_analyzer_id == (
    "fa_join_zwnj_stopwords_v1"
)
assert float(selected_k1) == 1.2
assert float(selected_b) == 0.3

frozen_analyzer_config = analyzer_by_id[
    selected_analyzer_id
]

if dataclasses.is_dataclass(
    frozen_analyzer_config
):
    frozen_analyzer_definition = (
        dataclasses.asdict(
            frozen_analyzer_config
        )
    )
else:
    frozen_analyzer_definition = dict(
        vars(frozen_analyzer_config)
    )

frozen_analyzer_definition = {
    key: value
    for key, value
    in frozen_analyzer_definition.items()
    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
            type(None),
        ),
    )
}

assert (
    frozen_analyzer_definition[
        "analyzer_id"
    ]
    == selected_analyzer_id
)

frozen_stopwords = sorted(
    PERSIAN_STOPWORDS_V1
)

assert len(frozen_stopwords) == 28
assert len(frozen_stopwords) == len(
    set(frozen_stopwords)
)

selected_development_metrics = {
    metric_name: float(metric_value)
    for metric_name, metric_value
    in selected_configuration_row[
        "metrics"
    ].items()
}

assert (
    selected_development_metrics[
        "ndcg_at_10"
    ]
    == selected_configuration_row[
        "metrics"
    ]["ndcg_at_10"]
)

sparse_configuration_freeze = {
    "artifact_type": (
        "NB15_SPARSE_CONFIGURATION_FREEZE"
    ),
    "schema_version": "1.0",
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "selection_scope": {
        "corpus": "ALL_CANONICAL_CHUNKS",
        "selection_queries": (
            "DEVELOPMENT_ONLY"
        ),
        "development_query_count": len(
            development_candidate_ids
        ),
        "test_queries_accessed": False,
        "test_access_policy": (
            "LOCKED_DURING_SELECTION"
        ),
    },
    "canonical_inputs": {
        "chunk_path": (
            "data/phase2/chunks/"
            "nb11_benchmark_v1/"
            "baseline_phase1_char_safe_v1.jsonl"
        ),
        "chunk_sha256": (
            "3891c4ecad9332ba753cd605bbccd99da"
            "9d063f64934a30ee39e495349cffb50"
        ),
        "chunk_count": len(chunk_records),
        "golden_dataset_path": (
            "artifacts/phase2/golden_dataset_v2/"
            "golden_dataset_v2.jsonl"
        ),
        "golden_dataset_file_sha256": (
            "b9ca18e710686ddca48a25c637840b716"
            "adb306717cafd8ec7c96ce9e182e6f2"
        ),
        "golden_dataset_payload_sha256": (
            "539635b4c89e8f4eda41dde8106e04307"
            "c063f3fe0f5db402ef0440802233a03"
        ),
        "golden_query_count": len(
            golden_records
        ),
        "gold_mapping_sha256": (
            "42216f67665cfc50697789dffe4eb5508"
            "39ee22bcbfc0489accc407519dad4d1"
        ),
        "split_sha256": (
            "68df1196f683b6ab9a9df08c015f001b"
            "0caca056331eb80d0048aec783736d8f"
        ),
    },
    "analyzer": {
        "analyzer_id": (
            selected_analyzer_id
        ),
        "definition": (
            frozen_analyzer_definition
        ),
        "unicode_normalization": "NFC",
        "arabic_to_persian_character_mapping": (
            True
        ),
        "digit_policy": (
            "PERSIAN_AND_ARABIC_TO_ASCII"
        ),
        "zwnj_policy": "JOIN",
        "stopword_policy": (
            "FIXED_LIST_V1"
        ),
        "stopword_count": len(
            frozen_stopwords
        ),
        "stopwords": frozen_stopwords,
    },
    "bm25": {
        "implementation": (
            "local_robertson_bm25_v1"
        ),
        "k1": float(selected_k1),
        "b": float(selected_b),
        "top_k": 10,
        "positive_scores_only": True,
        "deterministic_tie_break": (
            "DOCUMENT_INDEX_ASCENDING"
        ),
    },
    "development_selection": {
        "configuration_id": (
            selected_configuration_id
        ),
        "primary_metric": "ndcg_at_10",
        "tie_break_order": [
            "reciprocal_rank",
            "hit_at_10",
            "recall_at_10",
            "median_latency_ms",
            "configuration_id",
        ],
        "metrics": (
            selected_development_metrics
        ),
    },
    "challenger_evaluation": {
        "challenger_analyzer_id": (
            CONSERVATIVE_ANALYZER_ID
        ),
        "challenger_configuration_id": (
            best_conservative_configuration_id
        ),
        "challenger_ndcg_at_10": float(
            best_conservative_summary[
                "metrics"
            ]["ndcg_at_10"]
        ),
        "selected_ndcg_at_10": float(
            selected_development_metrics[
                "ndcg_at_10"
            ]
        ),
        "ndcg_at_10_delta": float(
            overall_ndcg_comparison[
                "observed_delta"
            ]
        ),
        "ndcg_at_10_confidence_interval_95": (
            overall_ndcg_comparison[
                "confidence_interval_95"
            ]
        ),
        "promotion_gates_passed": False,
        "decision": (
            "REJECT_CHALLENGER"
        ),
    },
    "freeze_rules": {
        "analyzer_mutation_after_test": (
            False
        ),
        "stopword_mutation_after_test": (
            False
        ),
        "bm25_parameter_mutation_after_test": (
            False
        ),
        "test_result_may_change_configuration": (
            False
        ),
        "test_result_role": (
            "INDEPENDENT_FINAL_EVALUATION_ONLY"
        ),
    },
    "paid_api_calls": 0,
    "files_created": 0,
    "repository_mutated": False,
    "configuration_status": (
        "FROZEN_BEFORE_TEST_ACCESS"
    ),
}

sparse_configuration_freeze_payload = (
    json.dumps(
        sparse_configuration_freeze,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
)

sparse_configuration_sha256 = (
    hashlib.sha256(
        sparse_configuration_freeze_payload.encode(
            "utf-8"
        )
    ).hexdigest()
)

sparse_configuration_freeze[
    "configuration_sha256"
] = sparse_configuration_sha256

FROZEN_SPARSE_ANALYZER_ID = (
    selected_analyzer_id
)
FROZEN_SPARSE_K1 = float(selected_k1)
FROZEN_SPARSE_B = float(selected_b)
FROZEN_SPARSE_CONFIGURATION_ID = (
    selected_configuration_id
)
FROZEN_SPARSE_CONFIGURATION_SHA256 = (
    sparse_configuration_sha256
)
SPARSE_CONFIGURATION_FROZEN = True
TEST_SPLIT_ACCESS_AUTHORIZED = True

assert (
    sparse_configuration_freeze[
        "selection_scope"
    ]["test_queries_accessed"]
    is False
)
assert (
    sparse_configuration_freeze[
        "freeze_rules"
    ]["test_result_may_change_configuration"]
    is False
)
assert len(
    sparse_configuration_sha256
) == 64
assert SPARSE_CONFIGURATION_FROZEN
assert TEST_SPLIT_ACCESS_AUTHORIZED

print(
    json.dumps(
        sparse_configuration_freeze,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "artifact_type": "NB15_SPARSE_CONFIGURATION_FREEZE",
  "schema_version": "1.0",
  "release_id": "rag_finance_phase2_sparse_v1",
  "selection_scope": {
    "corpus": "ALL_CANONICAL_CHUNKS",
    "selection_queries": "DEVELOPMENT_ONLY",
    "development_query_count": 2664,
    "test_queries_accessed": false,
    "test_access_policy": "LOCKED_DURING_SELECTION"
  },
  "canonical_inputs": {
    "chunk_path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
    "chunk_sha256": "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
    "chunk_count": 3488,
    "golden_dataset_path": "artifacts/phase2/golden_dataset_v2/golden_dataset_v2.jsonl",
    "golden_dataset_file_sha256": "b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2",
    "golden_dataset_payload_sha256": "539635b4c89e8f4eda41dde8106e04307c063f3fe0f5db402ef0440802233a03",
    "golden_query_count": 3813,
    "gold_mapping_sha256": "42216f67665cfc50697789dffe4eb550839ee22bcbfc04

<div dir="rtl" align="right">

## ارزیابی مستقل و نهایی Sparse Retrieval روی Test Split

پیکربندی Sparse Retrieval پیش از مشاهده Test Split فریز و Hash شد. در این مرحله Test Split برای اولین بار باز می‌شود و هیچ انتخاب مدل، تغییر Analyzer، تغییر Stopword یا تنظیم مجدد پارامترهای BM25 براساس نتیجه آن مجاز نیست.

ارزیابی نهایی شامل موارد زیر است:

1. اجرای پیکربندی فریز‌شده روی تمام Queryهای Test؛
2. محاسبه Hit@5، Hit@10، Recall@5، Recall@10، Precision، MRR و nDCG؛
3. اندازه‌گیری Latency محلی؛
4. گزارش عملکرد به تفکیک نوع منبع؛
5. بررسی Queryهای عددی، لاتین، دارای نیم‌فاصله و OOV؛
6. طبقه‌بندی شکست‌ها به فقدان هم‌پوشانی واژگانی یا رتبه پایین‌تر از ۱۰؛
7. مقایسه Development و Test صرفاً برای گزارش پایداری، بدون تغییر پیکربندی.

نتیجه این سل فقط نقش ارزیابی مستقل نهایی را دارد و تحت هیچ شرایطی برای بازتنظیم پیکربندی فریز‌شده استفاده نمی‌شود.

</div>

In [24]:
# Code Cell 19 — Independent final evaluation on the locked test split

assert SPARSE_CONFIGURATION_FROZEN
assert TEST_SPLIT_ACCESS_AUTHORIZED

assert (
    FROZEN_SPARSE_CONFIGURATION_SHA256
    == sparse_configuration_freeze[
        "configuration_sha256"
    ]
)

assert (
    FROZEN_SPARSE_ANALYZER_ID
    == "fa_join_zwnj_stopwords_v1"
)
assert FROZEN_SPARSE_K1 == 1.2
assert FROZEN_SPARSE_B == 0.3

assert len(test_candidate_ids) == 1149
assert not (
    set(development_candidate_ids)
    & set(test_candidate_ids)
)

golden_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in golden_records
}

assert set(test_candidate_ids).issubset(
    golden_record_by_candidate_id
)

test_query_by_candidate_id = {
    candidate_id: (
        golden_record_by_candidate_id[
            candidate_id
        ]["evaluation_content"]["question"]
    )
    for candidate_id in test_candidate_ids
}

assert all(
    isinstance(question, str)
    and question.strip()
    for question in test_query_by_candidate_id.values()
)

frozen_analyzer_config = analyzer_by_id[
    FROZEN_SPARSE_ANALYZER_ID
]
frozen_sparse_index = bm25_indexes[
    FROZEN_SPARSE_ANALYZER_ID
]

assert tuple(
    frozen_sparse_index.document_ids
) == tuple(
    record["chunk_id"]
    for record in chunk_records
)

test_query_tokens_by_candidate_id = {
    candidate_id: analyze_persian_text(
        text=test_query_by_candidate_id[
            candidate_id
        ],
        config=frozen_analyzer_config,
    )
    for candidate_id in test_candidate_ids
}

empty_test_query_ids = [
    candidate_id
    for candidate_id, tokens
    in test_query_tokens_by_candidate_id.items()
    if not tokens
]

assert not empty_test_query_ids

test_evaluation_rows = []
test_query_latencies_ms = []

for candidate_id in test_candidate_ids:
    query_tokens = (
        test_query_tokens_by_candidate_id[
            candidate_id
        ]
    )

    scoring_started = time.perf_counter()

    scores = bm25_score_query(
        index=frozen_sparse_index,
        query_tokens=query_tokens,
        k1=FROZEN_SPARSE_K1,
        b=FROZEN_SPARSE_B,
    )

    ranked_document_indexes = (
        deterministic_positive_top_k(
            scores=scores,
            top_k=10,
        )
    )

    query_latency_ms = (
        time.perf_counter()
        - scoring_started
    ) * 1000.0

    test_query_latencies_ms.append(
        query_latency_ms
    )

    ranked_chunk_ids = [
        frozen_sparse_index.document_ids[
            document_index
        ]
        for document_index
        in ranked_document_indexes
    ]

    relevant_chunk_ids = set(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    query_metrics = evaluate_ranked_result(
        ranked_chunk_ids,
        relevant_chunk_ids,
    )

    source_types = sorted(
        set(
            golden_record_by_candidate_id[
                candidate_id
            ].get("source_types", [])
        )
    )

    test_evaluation_rows.append(
        {
            "candidate_id": candidate_id,
            "source_types": source_types,
            "query_tokens": query_tokens,
            "ranked_chunk_ids": ranked_chunk_ids,
            "relevant_chunk_ids": sorted(
                relevant_chunk_ids
            ),
            "query_latency_ms": float(
                query_latency_ms
            ),
            **query_metrics,
        }
    )

assert len(test_evaluation_rows) == len(
    test_candidate_ids
)

test_rows_by_candidate_id = {
    row["candidate_id"]: row
    for row in test_evaluation_rows
}

assert set(
    test_rows_by_candidate_id
) == set(test_candidate_ids)

metric_names = (
    "hit_at_10",
    "hit_at_5",
    "ndcg_at_10",
    "ndcg_at_5",
    "precision_at_10",
    "precision_at_5",
    "recall_at_10",
    "recall_at_5",
    "reciprocal_rank",
)

test_metrics = {
    metric_name: float(
        np.mean(
            [
                row[metric_name]
                for row in test_evaluation_rows
            ]
        )
    )
    for metric_name in metric_names
}

test_latency_array = np.asarray(
    test_query_latencies_ms,
    dtype=np.float64,
)

test_latency_summary = {
    "minimum": float(
        test_latency_array.min()
    ),
    "median_p50": float(
        np.median(test_latency_array)
    ),
    "p95": float(
        np.percentile(
            test_latency_array,
            95,
        )
    ),
    "maximum": float(
        test_latency_array.max()
    ),
    "scope": (
        "LOCAL_INDEPENDENT_TEST_"
        "NOT_PRODUCTION_SLO"
    ),
}

test_source_type_metrics = {}

for source_type in (
    "book",
    "transcript",
    "website",
):
    source_rows = [
        row
        for row in test_evaluation_rows
        if source_type in row["source_types"]
    ]

    assert source_rows

    test_source_type_metrics[
        source_type
    ] = {
        "query_count": len(source_rows),
        "metrics": {
            metric_name: float(
                np.mean(
                    [
                        row[metric_name]
                        for row in source_rows
                    ]
                )
            )
            for metric_name in metric_names
        },
    }

digit_pattern = re.compile(
    r"[0-9۰-۹]"
)
latin_pattern = re.compile(
    r"[A-Za-z]"
)

test_slice_candidate_ids = {
    "numeric": [
        candidate_id
        for candidate_id in test_candidate_ids
        if digit_pattern.search(
            test_query_by_candidate_id[
                candidate_id
            ]
        )
    ],
    "latin": [
        candidate_id
        for candidate_id in test_candidate_ids
        if latin_pattern.search(
            test_query_by_candidate_id[
                candidate_id
            ]
        )
    ],
    "zwnj": [
        candidate_id
        for candidate_id in test_candidate_ids
        if "\u200c"
        in test_query_by_candidate_id[
            candidate_id
        ]
    ],
    "oov": [
        candidate_id
        for candidate_id in test_candidate_ids
        if any(
            token
            not in (
                frozen_sparse_index
                .document_frequency
            )
            for token in (
                test_query_tokens_by_candidate_id[
                    candidate_id
                ]
            )
        )
    ],
}

test_slice_metrics = {}

for slice_name, slice_candidate_ids in (
    test_slice_candidate_ids.items()
):
    slice_rows = [
        test_rows_by_candidate_id[
            candidate_id
        ]
        for candidate_id
        in slice_candidate_ids
    ]

    if slice_rows:
        test_slice_metrics[
            slice_name
        ] = {
            "query_count": len(slice_rows),
            "metrics": {
                metric_name: float(
                    np.mean(
                        [
                            row[metric_name]
                            for row in slice_rows
                        ]
                    )
                )
                for metric_name in metric_names
            },
        }
    else:
        test_slice_metrics[
            slice_name
        ] = {
            "query_count": 0,
            "metrics": None,
        }

document_index_by_id = {
    document_id: document_index
    for document_index, document_id
    in enumerate(
        frozen_sparse_index.document_ids
    )
}

frozen_document_token_sets = [
    set(tokens)
    for tokens in selected_document_tokens
]

assert len(
    frozen_document_token_sets
) == len(chunk_records)

test_failure_taxonomy = (
    collections.Counter()
)

test_failure_examples = (
    collections.defaultdict(list)
)

for row in test_evaluation_rows:
    candidate_id = row["candidate_id"]

    if row["hit_at_10"] == 1.0:
        failure_type = "SUCCESS"
    else:
        query_token_set = set(
            row["query_tokens"]
        )

        relevant_document_indexes = [
            document_index_by_id[
                chunk_id
            ]
            for chunk_id
            in row["relevant_chunk_ids"]
        ]

        relevant_token_union = set()

        for document_index in (
            relevant_document_indexes
        ):
            relevant_token_union.update(
                frozen_document_token_sets[
                    document_index
                ]
            )

        if not (
            query_token_set
            & relevant_token_union
        ):
            failure_type = (
                "NO_QUERY_GOLD_"
                "LEXICAL_OVERLAP"
            )
        else:
            failure_type = (
                "LEXICAL_OVERLAP_"
                "RANK_BELOW_10"
            )

    test_failure_taxonomy[
        failure_type
    ] += 1

    if (
        failure_type != "SUCCESS"
        and len(
            test_failure_examples[
                failure_type
            ]
        )
        < 10
    ):
        test_failure_examples[
            failure_type
        ].append(
            {
                "candidate_id": candidate_id,
                "question": (
                    test_query_by_candidate_id[
                        candidate_id
                    ]
                ),
                "source_types": (
                    row["source_types"]
                ),
                "relevant_chunk_count": len(
                    row["relevant_chunk_ids"]
                ),
                "retrieved_chunk_count": len(
                    row["ranked_chunk_ids"]
                ),
            }
        )

development_test_metric_delta = {
    metric_name: float(
        test_metrics[metric_name]
        - selected_development_metrics[
            metric_name
        ]
    )
    for metric_name in metric_names
}

independent_sparse_test_evaluation = {
    "artifact_type": (
        "NB15_INDEPENDENT_SPARSE_TEST_EVALUATION"
    ),
    "schema_version": "1.0",
    "configuration": {
        "configuration_id": (
            FROZEN_SPARSE_CONFIGURATION_ID
        ),
        "configuration_sha256": (
            FROZEN_SPARSE_CONFIGURATION_SHA256
        ),
        "analyzer_id": (
            FROZEN_SPARSE_ANALYZER_ID
        ),
        "bm25": {
            "implementation": (
                "local_robertson_bm25_v1"
            ),
            "k1": FROZEN_SPARSE_K1,
            "b": FROZEN_SPARSE_B,
        },
        "mutated_after_freeze": False,
    },
    "evaluation_scope": {
        "split": "TEST",
        "query_count": len(
            test_candidate_ids
        ),
        "corpus_document_count": len(
            frozen_sparse_index.document_ids
        ),
        "test_accessed_after_freeze": True,
        "configuration_tuning_on_test": False,
    },
    "metrics": test_metrics,
    "latency_ms": test_latency_summary,
    "source_type_metrics": (
        test_source_type_metrics
    ),
    "exact_term_slices": (
        test_slice_metrics
    ),
    "failure_taxonomy": {
        "counts": dict(
            sorted(
                test_failure_taxonomy.items()
            )
        ),
        "examples": {
            key: value
            for key, value
            in sorted(
                test_failure_examples.items()
            )
        },
    },
    "development_comparison": {
        "development_metrics": (
            selected_development_metrics
        ),
        "test_minus_development": (
            development_test_metric_delta
        ),
        "role": (
            "REPORT_ONLY_NO_RETUNING"
        ),
    },
    "paid_api_calls": 0,
    "files_created": 0,
    "repository_mutated": False,
    "independent_test_status": "PASS",
}

assert (
    independent_sparse_test_evaluation[
        "configuration"
    ]["mutated_after_freeze"]
    is False
)
assert (
    independent_sparse_test_evaluation[
        "evaluation_scope"
    ]["configuration_tuning_on_test"]
    is False
)
assert sum(
    test_failure_taxonomy.values()
) == len(test_candidate_ids)
assert (
    independent_sparse_test_evaluation[
        "independent_test_status"
    ]
    == "PASS"
)

print(
    json.dumps(
        independent_sparse_test_evaluation,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "artifact_type": "NB15_INDEPENDENT_SPARSE_TEST_EVALUATION",
  "schema_version": "1.0",
  "configuration": {
    "configuration_id": "fa_join_zwnj_stopwords_v1__k1_1.20__b_0.30",
    "configuration_sha256": "96d66676d20bacd764ec9a163995dd48266c90c2acea1c8a5f77d95d9efb6ecc",
    "analyzer_id": "fa_join_zwnj_stopwords_v1",
    "bm25": {
      "implementation": "local_robertson_bm25_v1",
      "k1": 1.2,
      "b": 0.3
    },
    "mutated_after_freeze": false
  },
  "evaluation_scope": {
    "split": "TEST",
    "query_count": 1149,
    "corpus_document_count": 3488,
    "test_accessed_after_freeze": true,
    "configuration_tuning_on_test": false
  },
  "metrics": {
    "hit_at_10": 0.7075718015665796,
    "hit_at_5": 0.6518711923411662,
    "ndcg_at_10": 0.49020522695816876,
    "ndcg_at_5": 0.46879829553108543,
    "precision_at_10": 0.09451697127937336,
    "precision_at_5": 0.16675369886858138,
    "recall_at_10": 0.5535776451572797,
    "recall_at_5": 0.49356686974180447,
    "re

<div dir="rtl" align="right">

## ساخت Artifact نسخه‌دار Sparse Index V1

پیکربندی نهایی Sparse Retrieval پیش از ارزیابی Test فریز شد و ارزیابی مستقل نیز بدون بازتنظیم پیکربندی با موفقیت انجام شد.

در این مرحله Sparse Index از State موقت حافظه خارج و به مجموعه‌ای از Artifactهای قطعی، نسخه‌دار و قابل اعتبارسنجی تبدیل می‌شود:

1. جدول نسخه‌دار اسناد و طول Token هر سند؛
2. Inverted Postings شامل شناسه سند و Term Frequency؛
3. پیکربندی فریز‌شده Analyzer و BM25؛
4. گزارش مستقل Test؛
5. Inventory شامل SHA-256 تمام فایل‌ها؛
6. Logical Index Hash برای اتصال قطعی Document Table، Postings و Configuration.

فرمت Postings به‌صورت JSONL قطعی ذخیره می‌شود تا به Pickle، Kernel جاری یا ساختار حافظه‌ای Python وابسته نباشد. Tokenها به‌ترتیب Unicode مرتب می‌شوند و تمام JSONها با Canonical Serialization نوشته می‌شوند.

این مرحله هیچ Embedding جدیدی محاسبه نمی‌کند، هیچ API پولی فراخوانی نمی‌کند و Dense Index فریز‌شده را تغییر نمی‌دهد.

</div>

In [25]:
# Code Cell 20 — Persist the versioned Sparse Index V1 artifacts

import os
from pathlib import Path

SPARSE_INDEX_ROOT = Path(
    "data/phase2/index/sparse_index_v1"
)
SPARSE_REPORT_ROOT = Path(
    "reports/phase2"
)

SPARSE_DOCUMENTS_PATH = (
    SPARSE_INDEX_ROOT
    / "sparse_documents_v1.jsonl"
)
SPARSE_POSTINGS_PATH = (
    SPARSE_INDEX_ROOT
    / "sparse_postings_v1.jsonl"
)
SPARSE_CONFIGURATION_PATH = (
    SPARSE_INDEX_ROOT
    / "sparse_configuration_v1.json"
)
SPARSE_TEST_EVALUATION_PATH = (
    SPARSE_REPORT_ROOT
    / "sparse_test_evaluation_v1.json"
)
SPARSE_INVENTORY_PATH = (
    SPARSE_INDEX_ROOT
    / "sparse_index_inventory_v1.json"
)

assert SPARSE_CONFIGURATION_FROZEN
assert (
    independent_sparse_test_evaluation[
        "independent_test_status"
    ]
    == "PASS"
)
assert (
    sparse_configuration_freeze[
        "configuration_sha256"
    ]
    == FROZEN_SPARSE_CONFIGURATION_SHA256
)
assert len(
    frozen_sparse_index.document_ids
) == 3488
assert len(
    frozen_sparse_index.document_lengths
) == 3488
assert frozen_sparse_index.postings

SPARSE_INDEX_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
SPARSE_REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def canonical_json_text(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def pretty_json_text(value):
    return (
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        )
        + "\n"
    )


def atomic_write_text(
    target_path,
    text,
):
    target_path = Path(target_path)

    temporary_path = target_path.with_name(
        target_path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as file_handle:
        file_handle.write(text)
        file_handle.flush()
        os.fsync(file_handle.fileno())

    os.replace(
        temporary_path,
        target_path,
    )


def sha256_file(file_path):
    digest = hashlib.sha256()

    with Path(file_path).open(
        "rb"
    ) as file_handle:
        for block in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


document_rows = []

for document_index, (
    document_id,
    document_length,
) in enumerate(
    zip(
        frozen_sparse_index.document_ids,
        frozen_sparse_index.document_lengths,
    )
):
    expected_chunk_id = chunk_records[
        document_index
    ]["chunk_id"]

    assert document_id == expected_chunk_id

    document_rows.append(
        {
            "document_index": int(
                document_index
            ),
            "document_id": str(
                document_id
            ),
            "document_length": int(
                document_length
            ),
            "source_type": str(
                chunk_records[
                    document_index
                ]["source_type"]
            ),
        }
    )

assert len(document_rows) == len(
    chunk_records
)
assert len(
    {
        row["document_id"]
        for row in document_rows
    }
) == len(document_rows)

documents_jsonl_text = "".join(
    canonical_json_text(row) + "\n"
    for row in document_rows
)

atomic_write_text(
    SPARSE_DOCUMENTS_PATH,
    documents_jsonl_text,
)

sorted_vocabulary = sorted(
    frozen_sparse_index.postings
)

assert len(sorted_vocabulary) == len(
    frozen_sparse_index.document_frequency
)
assert len(sorted_vocabulary) > 0

posting_line_count = 0
posting_entry_count = 0
postings_temporary_path = (
    SPARSE_POSTINGS_PATH.with_name(
        SPARSE_POSTINGS_PATH.name + ".tmp"
    )
)

with postings_temporary_path.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as file_handle:
    for token in sorted_vocabulary:
        (
            document_indexes,
            term_frequencies,
        ) = frozen_sparse_index.postings[
            token
        ]

        document_indexes = np.asarray(
            document_indexes,
            dtype=np.int64,
        )
        term_frequencies = np.asarray(
            term_frequencies,
            dtype=np.float64,
        )

        assert document_indexes.ndim == 1
        assert term_frequencies.ndim == 1
        assert len(
            document_indexes
        ) == len(term_frequencies)
        assert len(document_indexes) > 0
        assert np.all(
            document_indexes[:-1]
            < document_indexes[1:]
        )
        assert np.all(
            document_indexes >= 0
        )
        assert np.all(
            document_indexes
            < canonical_document_count
        )
        assert np.all(
            term_frequencies > 0
        )
        assert np.all(
            term_frequencies
            == np.floor(
                term_frequencies
            )
        )

        expected_document_frequency = (
            frozen_sparse_index
            .document_frequency[token]
        )

        assert len(
            document_indexes
        ) == expected_document_frequency

        posting_row = {
            "token": token,
            "document_frequency": int(
                expected_document_frequency
            ),
            "document_indexes": [
                int(value)
                for value
                in document_indexes.tolist()
            ],
            "term_frequencies": [
                int(value)
                for value
                in term_frequencies.tolist()
            ],
        }

        file_handle.write(
            canonical_json_text(
                posting_row
            )
            + "\n"
        )

        posting_line_count += 1
        posting_entry_count += len(
            document_indexes
        )

    file_handle.flush()
    os.fsync(file_handle.fileno())

os.replace(
    postings_temporary_path,
    SPARSE_POSTINGS_PATH,
)

assert posting_line_count == len(
    sorted_vocabulary
)
assert posting_entry_count > 0

persisted_sparse_configuration = {
    **sparse_configuration_freeze,
    "artifact_paths": {
        "documents": str(
            SPARSE_DOCUMENTS_PATH
        ),
        "postings": str(
            SPARSE_POSTINGS_PATH
        ),
        "test_evaluation": str(
            SPARSE_TEST_EVALUATION_PATH
        ),
    },
    "index_statistics": {
        "document_count": len(
            frozen_sparse_index.document_ids
        ),
        "vocabulary_size": len(
            sorted_vocabulary
        ),
        "posting_entry_count": int(
            posting_entry_count
        ),
        "total_token_count": int(
            np.sum(
                frozen_sparse_index
                .document_lengths
            )
        ),
        "average_document_length": float(
            frozen_sparse_index
            .average_document_length
        ),
    },
    "serialization": {
        "documents_format": (
            "CANONICAL_JSONL_UTF8"
        ),
        "postings_format": (
            "CANONICAL_JSONL_UTF8"
        ),
        "token_order": (
            "UNICODE_CODEPOINT_ASCENDING"
        ),
        "posting_document_order": (
            "DOCUMENT_INDEX_ASCENDING"
        ),
        "pickle_used": False,
    },
}

atomic_write_text(
    SPARSE_CONFIGURATION_PATH,
    pretty_json_text(
        persisted_sparse_configuration
    ),
)

atomic_write_text(
    SPARSE_TEST_EVALUATION_PATH,
    pretty_json_text(
        independent_sparse_test_evaluation
    ),
)

documents_sha256 = sha256_file(
    SPARSE_DOCUMENTS_PATH
)
postings_sha256 = sha256_file(
    SPARSE_POSTINGS_PATH
)
configuration_file_sha256 = sha256_file(
    SPARSE_CONFIGURATION_PATH
)
test_evaluation_sha256 = sha256_file(
    SPARSE_TEST_EVALUATION_PATH
)

logical_index_contract = {
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "configuration_sha256": (
        FROZEN_SPARSE_CONFIGURATION_SHA256
    ),
    "documents_sha256": (
        documents_sha256
    ),
    "postings_sha256": postings_sha256,
    "document_count": len(
        document_rows
    ),
    "vocabulary_size": len(
        sorted_vocabulary
    ),
    "posting_entry_count": int(
        posting_entry_count
    ),
}

sparse_logical_index_sha256 = (
    hashlib.sha256(
        canonical_json_text(
            logical_index_contract
        ).encode("utf-8")
    ).hexdigest()
)

sparse_index_inventory = {
    "artifact_type": (
        "NB15_SPARSE_INDEX_INVENTORY"
    ),
    "schema_version": "1.0",
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "files": {
        "documents": {
            "path": str(
                SPARSE_DOCUMENTS_PATH
            ),
            "sha256": (
                documents_sha256
            ),
            "record_count": len(
                document_rows
            ),
        },
        "postings": {
            "path": str(
                SPARSE_POSTINGS_PATH
            ),
            "sha256": postings_sha256,
            "record_count": int(
                posting_line_count
            ),
            "posting_entry_count": int(
                posting_entry_count
            ),
        },
        "configuration": {
            "path": str(
                SPARSE_CONFIGURATION_PATH
            ),
            "sha256": (
                configuration_file_sha256
            ),
            "configuration_sha256": (
                FROZEN_SPARSE_CONFIGURATION_SHA256
            ),
        },
        "test_evaluation": {
            "path": str(
                SPARSE_TEST_EVALUATION_PATH
            ),
            "sha256": (
                test_evaluation_sha256
            ),
            "query_count": len(
                test_candidate_ids
            ),
        },
    },
    "logical_index_contract": (
        logical_index_contract
    ),
    "logical_index_sha256": (
        sparse_logical_index_sha256
    ),
    "canonical_chunk_count": len(
        chunk_records
    ),
    "dense_index_mutated": False,
    "embedding_recomputation": False,
    "paid_api_calls": 0,
    "inventory_status": (
        "WRITTEN_PENDING_RELOAD_VALIDATION"
    ),
}

atomic_write_text(
    SPARSE_INVENTORY_PATH,
    pretty_json_text(
        sparse_index_inventory
    ),
)

inventory_file_sha256 = sha256_file(
    SPARSE_INVENTORY_PATH
)

assert all(
    path.exists()
    for path in (
        SPARSE_DOCUMENTS_PATH,
        SPARSE_POSTINGS_PATH,
        SPARSE_CONFIGURATION_PATH,
        SPARSE_TEST_EVALUATION_PATH,
        SPARSE_INVENTORY_PATH,
    )
)
assert len(
    sparse_logical_index_sha256
) == 64
assert len(
    inventory_file_sha256
) == 64

sparse_artifact_write_result = {
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "document_count": len(
        document_rows
    ),
    "vocabulary_size": len(
        sorted_vocabulary
    ),
    "posting_entry_count": int(
        posting_entry_count
    ),
    "configuration_sha256": (
        FROZEN_SPARSE_CONFIGURATION_SHA256
    ),
    "logical_index_sha256": (
        sparse_logical_index_sha256
    ),
    "inventory": {
        "path": str(
            SPARSE_INVENTORY_PATH
        ),
        "sha256": (
            inventory_file_sha256
        ),
    },
    "files_created": 5,
    "dense_index_mutated": False,
    "embedding_recomputation": False,
    "paid_api_calls": 0,
    "artifact_write_status": (
        "PASS_PENDING_RELOAD_VALIDATION"
    ),
}

print(
    json.dumps(
        sparse_artifact_write_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "release_id": "rag_finance_phase2_sparse_v1",
  "document_count": 3488,
  "vocabulary_size": 27311,
  "posting_entry_count": 383637,
  "configuration_sha256": "96d66676d20bacd764ec9a163995dd48266c90c2acea1c8a5f77d95d9efb6ecc",
  "logical_index_sha256": "ce09bb4749da03c9dfa1a849de18fc8c490ce6b09eb34a30a96b9fedc478a8eb",
  "inventory": {
    "path": "data/phase2/index/sparse_index_v1/sparse_index_inventory_v1.json",
    "sha256": "6e2f5f7e055d56c9234f3f9cb6f7d6bcfee618e85485cdd13bcdc9f3ce0fd810"
  },
  "files_created": 5,
  "dense_index_mutated": false,
  "embedding_recomputation": false,
  "paid_api_calls": 0,
  "artifact_write_status": "PASS_PENDING_RELOAD_VALIDATION"
}


<div dir="rtl" align="right">

## Reload Validation و اثبات استقلال Sparse Index از Kernel

صرفاً نوشته‌شدن فایل‌های Sparse Index برای پذیرش نهایی کافی نیست. Artifact باید بدون استفاده از ساختارهای موقت حافظه، از روی فایل‌های نسخه‌دار دوباره بارگذاری و به یک Index قابل جست‌وجو تبدیل شود.

در این مرحله موارد زیر بررسی می‌شوند:

1. برابری SHA-256 تمام فایل‌ها با Inventory؛
2. بارگذاری مجدد ۳۴۸۸ سند و ۲۷۳۱۱ Posting؛
3. صحت ترتیب و یکتایی Document IDها؛
4. صحت Document Frequency و Term Frequency؛
5. بازسازی کامل Index از فایل‌های JSONL؛
6. اجرای مجدد تمام ۱۱۴۹ Query مستقل Test؛
7. برابری دقیق Top-10 نتایج Reload شده با اجرای مرجع؛
8. برابری Metricهای Test پس از Reload؛
9. ثبت Evidence مستقل برای تحویل به NB16.

عبور از این سل اثبات می‌کند که Sparse Index به Kernel جاری یا Objectهای موقت Notebook وابسته نیست و از Artifactهای ذخیره‌شده قابل بازسازی است.

</div>

In [27]:
# Code Cell 21 — Reload and validate the persisted Sparse Index V1

import copy

SPARSE_RELOAD_EVIDENCE_PATH = (
    SPARSE_INDEX_ROOT
    / "sparse_reload_validation_v1.json"
)

SCORE_RTOL = 1e-6
SCORE_ATOL = 1e-8

persisted_inventory = json.loads(
    SPARSE_INVENTORY_PATH.read_text(
        encoding="utf-8"
    )
)

persisted_configuration = json.loads(
    SPARSE_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)

persisted_test_evaluation = json.loads(
    SPARSE_TEST_EVALUATION_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    persisted_inventory["release_id"]
    == "rag_finance_phase2_sparse_v1"
)
assert (
    persisted_inventory["inventory_status"]
    == "WRITTEN_PENDING_RELOAD_VALIDATION"
)

expected_file_hashes = {
    "documents": persisted_inventory[
        "files"
    ]["documents"]["sha256"],
    "postings": persisted_inventory[
        "files"
    ]["postings"]["sha256"],
    "configuration": persisted_inventory[
        "files"
    ]["configuration"]["sha256"],
    "test_evaluation": persisted_inventory[
        "files"
    ]["test_evaluation"]["sha256"],
}

actual_file_hashes = {
    "documents": sha256_file(
        SPARSE_DOCUMENTS_PATH
    ),
    "postings": sha256_file(
        SPARSE_POSTINGS_PATH
    ),
    "configuration": sha256_file(
        SPARSE_CONFIGURATION_PATH
    ),
    "test_evaluation": sha256_file(
        SPARSE_TEST_EVALUATION_PATH
    ),
}

assert actual_file_hashes == expected_file_hashes

assert (
    persisted_configuration[
        "configuration_sha256"
    ]
    == FROZEN_SPARSE_CONFIGURATION_SHA256
)

assert (
    persisted_test_evaluation[
        "configuration"
    ]["configuration_sha256"]
    == FROZEN_SPARSE_CONFIGURATION_SHA256
)

reloaded_document_rows = []

with SPARSE_DOCUMENTS_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:
    for line_number, line in enumerate(
        file_handle,
        start=1,
    ):
        stripped_line = line.strip()

        assert stripped_line, (
            f"Empty document line: {line_number}"
        )

        reloaded_document_rows.append(
            json.loads(stripped_line)
        )

assert len(reloaded_document_rows) == 3488

assert len(
    reloaded_document_rows
) == persisted_inventory[
    "files"
]["documents"]["record_count"]

assert [
    row["document_index"]
    for row in reloaded_document_rows
] == list(
    range(len(reloaded_document_rows))
)

reloaded_document_ids = tuple(
    row["document_id"]
    for row in reloaded_document_rows
)

reloaded_document_lengths = np.asarray(
    [
        row["document_length"]
        for row in reloaded_document_rows
    ],
    dtype=np.float64,
)

assert len(
    set(reloaded_document_ids)
) == len(reloaded_document_ids)

assert reloaded_document_ids == tuple(
    record["chunk_id"]
    for record in chunk_records
)

assert np.array_equal(
    reloaded_document_lengths,
    np.asarray(
        frozen_sparse_index.document_lengths,
        dtype=np.float64,
    ),
)

reloaded_postings = {}
reloaded_document_frequency = {}
reloaded_posting_entry_count = 0
previous_token = None

with SPARSE_POSTINGS_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:
    for line_number, line in enumerate(
        file_handle,
        start=1,
    ):
        stripped_line = line.strip()

        assert stripped_line, (
            f"Empty posting line: {line_number}"
        )

        posting_row = json.loads(
            stripped_line
        )

        token = posting_row["token"]

        if previous_token is not None:
            assert previous_token < token

        previous_token = token

        document_indexes = np.asarray(
            posting_row["document_indexes"],
            dtype=np.int64,
        )

        term_frequencies = np.asarray(
            posting_row["term_frequencies"],
            dtype=np.float64,
        )

        document_frequency = int(
            posting_row["document_frequency"]
        )

        assert token not in reloaded_postings
        assert len(document_indexes) > 0
        assert len(
            document_indexes
        ) == len(term_frequencies)
        assert len(
            document_indexes
        ) == document_frequency
        assert np.all(document_indexes >= 0)
        assert np.all(
            document_indexes
            < len(reloaded_document_ids)
        )
        assert np.all(
            document_indexes[:-1]
            < document_indexes[1:]
        )
        assert np.all(term_frequencies > 0)
        assert np.all(
            term_frequencies
            == np.floor(term_frequencies)
        )

        reloaded_postings[token] = (
            document_indexes,
            term_frequencies,
        )

        reloaded_document_frequency[
            token
        ] = document_frequency

        reloaded_posting_entry_count += len(
            document_indexes
        )

assert len(
    reloaded_postings
) == persisted_inventory[
    "files"
]["postings"]["record_count"]

assert (
    reloaded_posting_entry_count
    == persisted_inventory[
        "files"
    ]["postings"]["posting_entry_count"]
)

assert set(
    reloaded_postings
) == set(
    frozen_sparse_index.postings
)

for token in reloaded_postings:
    (
        reloaded_document_indexes,
        reloaded_term_frequencies,
    ) = reloaded_postings[token]

    (
        reference_document_indexes,
        reference_term_frequencies,
    ) = frozen_sparse_index.postings[token]

    assert np.array_equal(
        reloaded_document_indexes,
        np.asarray(
            reference_document_indexes,
            dtype=np.int64,
        ),
    )

    assert np.array_equal(
        reloaded_term_frequencies,
        np.asarray(
            reference_term_frequencies,
            dtype=np.float64,
        ),
    )

    assert (
        reloaded_document_frequency[token]
        == frozen_sparse_index
        .document_frequency[token]
    )

reloaded_sparse_index = copy.copy(
    frozen_sparse_index
)

reloaded_sparse_index.analyzer_id = (
    persisted_configuration[
        "analyzer"
    ]["analyzer_id"]
)

reloaded_sparse_index.document_ids = (
    reloaded_document_ids
)

reloaded_sparse_index.document_lengths = (
    reloaded_document_lengths
)

reloaded_sparse_index.average_document_length = float(
    persisted_configuration[
        "index_statistics"
    ]["average_document_length"]
)

reloaded_sparse_index.document_frequency = (
    reloaded_document_frequency
)

reloaded_sparse_index.postings = (
    reloaded_postings
)

assert (
    reloaded_sparse_index.analyzer_id
    == FROZEN_SPARSE_ANALYZER_ID
)
assert len(
    reloaded_sparse_index.document_ids
) == 3488
assert len(
    reloaded_sparse_index.postings
) == 27311
assert (
    reloaded_sparse_index
    .average_document_length
    == frozen_sparse_index
    .average_document_length
)

reloaded_test_rows = []
ranking_mismatch_ids = []
score_mismatch_ids = []
maximum_absolute_score_delta = 0.0

for candidate_id in test_candidate_ids:
    query_tokens = (
        test_query_tokens_by_candidate_id[
            candidate_id
        ]
    )

    reloaded_scores = bm25_score_query(
        index=reloaded_sparse_index,
        query_tokens=query_tokens,
        k1=FROZEN_SPARSE_K1,
        b=FROZEN_SPARSE_B,
    )

    reference_scores = bm25_score_query(
        index=frozen_sparse_index,
        query_tokens=query_tokens,
        k1=FROZEN_SPARSE_K1,
        b=FROZEN_SPARSE_B,
    )

    scores_are_equivalent = np.allclose(
        reloaded_scores,
        reference_scores,
        rtol=SCORE_RTOL,
        atol=SCORE_ATOL,
    )

    if not scores_are_equivalent:
        score_mismatch_ids.append(
            candidate_id
        )

    score_delta = float(
        np.max(
            np.abs(
                reloaded_scores
                - reference_scores
            )
        )
    )

    maximum_absolute_score_delta = max(
        maximum_absolute_score_delta,
        score_delta,
    )

    reloaded_ranked_indexes = (
        deterministic_positive_top_k(
            scores=reloaded_scores,
            top_k=10,
        )
    )

    reloaded_ranked_chunk_ids = [
        reloaded_sparse_index.document_ids[
            document_index
        ]
        for document_index
        in reloaded_ranked_indexes
    ]

    reference_ranked_chunk_ids = (
        test_rows_by_candidate_id[
            candidate_id
        ]["ranked_chunk_ids"]
    )

    if (
        reloaded_ranked_chunk_ids
        != reference_ranked_chunk_ids
    ):
        ranking_mismatch_ids.append(
            candidate_id
        )

    relevant_chunk_ids = set(
        gold_mapping_by_candidate_id[
            candidate_id
        ]
    )

    reloaded_metrics = evaluate_ranked_result(
        reloaded_ranked_chunk_ids,
        relevant_chunk_ids,
    )

    reloaded_test_rows.append(
        {
            "candidate_id": candidate_id,
            **reloaded_metrics,
        }
    )

assert not score_mismatch_ids
assert not ranking_mismatch_ids
assert np.isfinite(
    maximum_absolute_score_delta
)

reloaded_test_metrics = {
    metric_name: float(
        np.mean(
            [
                row[metric_name]
                for row in reloaded_test_rows
            ]
        )
    )
    for metric_name in metric_names
}

metric_absolute_deltas = {
    metric_name: abs(
        reloaded_test_metrics[metric_name]
        - test_metrics[metric_name]
    )
    for metric_name in metric_names
}

maximum_metric_absolute_delta = float(
    max(metric_absolute_deltas.values())
)

assert maximum_metric_absolute_delta <= 1e-15

reloaded_logical_contract = {
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "configuration_sha256": (
        persisted_configuration[
            "configuration_sha256"
        ]
    ),
    "documents_sha256": (
        actual_file_hashes["documents"]
    ),
    "postings_sha256": (
        actual_file_hashes["postings"]
    ),
    "document_count": len(
        reloaded_document_ids
    ),
    "vocabulary_size": len(
        reloaded_postings
    ),
    "posting_entry_count": int(
        reloaded_posting_entry_count
    ),
}

reloaded_logical_index_sha256 = (
    hashlib.sha256(
        canonical_json_text(
            reloaded_logical_contract
        ).encode("utf-8")
    ).hexdigest()
)

assert (
    reloaded_logical_contract
    == persisted_inventory[
        "logical_index_contract"
    ]
)

assert (
    reloaded_logical_index_sha256
    == persisted_inventory[
        "logical_index_sha256"
    ]
)

assert (
    reloaded_logical_index_sha256
    == sparse_logical_index_sha256
)

sparse_reload_validation = {
    "artifact_type": (
        "NB15_SPARSE_INDEX_RELOAD_VALIDATION"
    ),
    "schema_version": "1.0",
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "configuration_sha256": (
        FROZEN_SPARSE_CONFIGURATION_SHA256
    ),
    "logical_index_sha256": (
        reloaded_logical_index_sha256
    ),
    "file_hash_validation": "PASS",
    "document_validation": {
        "document_count": len(
            reloaded_document_ids
        ),
        "id_order_parity": "PASS",
        "length_parity": "PASS",
    },
    "posting_validation": {
        "vocabulary_size": len(
            reloaded_postings
        ),
        "posting_entry_count": int(
            reloaded_posting_entry_count
        ),
        "document_frequency_parity": (
            "PASS"
        ),
        "posting_array_parity": "PASS",
    },
    "retrieval_validation": {
        "query_count": len(
            test_candidate_ids
        ),
        "score_mismatch_count": len(
            score_mismatch_ids
        ),
        "score_parity": "PASS",
        "score_comparison_method": (
            "numpy.allclose"
        ),
        "score_rtol": SCORE_RTOL,
        "score_atol": SCORE_ATOL,
        "maximum_absolute_score_delta": (
            maximum_absolute_score_delta
        ),
        "ranking_mismatch_count": len(
            ranking_mismatch_ids
        ),
        "top_10_ranking_parity": "PASS",
        "metric_parity": "PASS",
        "maximum_metric_absolute_delta": (
            maximum_metric_absolute_delta
        ),
        "reason_for_score_tolerance": (
            "Reloaded numeric arrays may use "
            "a different floating-point dtype "
            "while preserving equivalent scores, "
            "identical Top-10 rankings and "
            "identical evaluation metrics."
        ),
    },
    "kernel_state_dependency": False,
    "dense_index_mutated": False,
    "embedding_recomputation": False,
    "paid_api_calls": 0,
    "reload_validation_status": "PASS",
}

atomic_write_text(
    SPARSE_RELOAD_EVIDENCE_PATH,
    pretty_json_text(
        sparse_reload_validation
    ),
)

sparse_reload_evidence_sha256 = (
    sha256_file(
        SPARSE_RELOAD_EVIDENCE_PATH
    )
)

assert len(
    sparse_reload_evidence_sha256
) == 64

sparse_reload_validation_result = {
    **sparse_reload_validation,
    "evidence": {
        "path": str(
            SPARSE_RELOAD_EVIDENCE_PATH
        ),
        "sha256": (
            sparse_reload_evidence_sha256
        ),
    },
    "files_created": 1,
}

print(
    json.dumps(
        sparse_reload_validation_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "artifact_type": "NB15_SPARSE_INDEX_RELOAD_VALIDATION",
  "schema_version": "1.0",
  "release_id": "rag_finance_phase2_sparse_v1",
  "configuration_sha256": "96d66676d20bacd764ec9a163995dd48266c90c2acea1c8a5f77d95d9efb6ecc",
  "logical_index_sha256": "ce09bb4749da03c9dfa1a849de18fc8c490ce6b09eb34a30a96b9fedc478a8eb",
  "file_hash_validation": "PASS",
  "document_validation": {
    "document_count": 3488,
    "id_order_parity": "PASS",
    "length_parity": "PASS"
  },
  "posting_validation": {
    "vocabulary_size": 27311,
    "posting_entry_count": 383637,
    "document_frequency_parity": "PASS",
    "posting_array_parity": "PASS"
  },
  "retrieval_validation": {
    "query_count": 1149,
    "score_mismatch_count": 0,
    "score_parity": "PASS",
    "score_comparison_method": "numpy.allclose",
    "score_rtol": 1e-06,
    "score_atol": 1e-08,
    "maximum_absolute_score_delta": 4.840906775882559e-06,
    "ranking_mismatch_count": 0,
    "top_10_ranking_parity": "PASS",
    "metric_

<div dir="rtl" align="right">

## فریز نهایی Sparse Index و ایجاد قرارداد تحویل به NB16

Sparse Index نسخه اول با موفقیت ساخته، ذخیره و از روی Artifactهای مستقل Reload شد. برابری Documentها، Postingها، رتبه‌بندی Top-10 و Metricهای Test تأیید شده است.

در این مرحله فایل‌های نهایی زیر ساخته می‌شوند:

1. Manifest نهایی Sparse Index؛
2. Release نسخه‌دار و Frozen؛
3. قرارداد Handoff برای NB16؛
4. گزارش Acceptance نهایی NB15.

Handoff شامل شناسه Release، Hash منطقی Index، پیکربندی Analyzer و BM25، مسیر تمام فایل‌های لازم، Metricهای مستقل Test و سازگاری با Dense Index فریز‌شده NB14 است.

NB16 باید Sparse Index را فقط از طریق این Handoff و فایل‌های نسخه‌دار Load کند و نباید به State حافظه NB15 وابسته باشد. همچنین NB16 مجاز به تغییر Analyzer، Stopwordها یا پارامترهای BM25 فریز‌شده نیست.

</div>

In [31]:
# Code Cell 22 — Finalize Sparse Index manifest, release and NB16 handoff

SPARSE_RELEASE_ROOT = Path(
    "artifacts/phase2/sparse_index"
)

SPARSE_MANIFEST_PATH = (
    SPARSE_INDEX_ROOT
    / "sparse_index_manifest.json"
)
SPARSE_RELEASE_PATH = (
    SPARSE_RELEASE_ROOT
    / "sparse_index_release_v1.json"
)
NB16_SPARSE_HANDOFF_PATH = (
    SPARSE_RELEASE_ROOT
    / "nb16_sparse_index_handoff_v1.json"
)
SPARSE_ACCEPTANCE_PATH = (
    SPARSE_REPORT_ROOT
    / "sparse_index_acceptance_v1.json"
)

DENSE_RELEASE_PATH = Path(
    "artifacts/phase2/dense_index/"
    "dense_index_release_v1.json"
)
EXPECTED_DENSE_RELEASE_SHA256 = (
    "706c020caa2bda77c767fda8cd1b5749"
    "8818afdbdbbbdd2a72f898b55ca6ca50"
)
EXPECTED_DENSE_LOGICAL_INDEX_SHA256 = (
    "2fc5e50ebbd1e1e66d59c703eb48f136"
    "7b2d3deadfed62efb9d5a5e6959f7ef4"
)

assert (
    sparse_reload_validation[
        "reload_validation_status"
    ]
    == "PASS"
)
assert (
    sparse_reload_validation[
        "retrieval_validation"
    ]["top_10_ranking_parity"]
    == "PASS"
)
assert (
    sparse_reload_validation[
        "retrieval_validation"
    ]["metric_parity"]
    == "PASS"
)

DENSE_RELEASE_PATH = next(candidate for base in (Path.cwd(), *Path.cwd().parents) for candidate in (base / "artifacts/phase2/dense_index/dense_index_release_v1.json",) if candidate.is_file() and sha256_file(candidate) == EXPECTED_DENSE_RELEASE_SHA256)
assert DENSE_RELEASE_PATH.exists()
assert (
    sha256_file(DENSE_RELEASE_PATH)
    == EXPECTED_DENSE_RELEASE_SHA256
)

dense_release = json.loads(
    DENSE_RELEASE_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    dense_release["status"]
    == "FROZEN_FOR_DOWNSTREAM_RETRIEVAL"
)
assert (
    dense_release["final_status"]
    == "PASS_COMPLETE_FROZEN"
)
assert (
    dense_release["logical_index_sha256"]
    == EXPECTED_DENSE_LOGICAL_INDEX_SHA256
)
assert dense_release["point_count"] == 3488

SPARSE_RELEASE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

current_artifact_hashes = {
    "documents": sha256_file(
        SPARSE_DOCUMENTS_PATH
    ),
    "postings": sha256_file(
        SPARSE_POSTINGS_PATH
    ),
    "configuration": sha256_file(
        SPARSE_CONFIGURATION_PATH
    ),
    "test_evaluation": sha256_file(
        SPARSE_TEST_EVALUATION_PATH
    ),
    "inventory": sha256_file(
        SPARSE_INVENTORY_PATH
    ),
    "reload_validation": sha256_file(
        SPARSE_RELOAD_EVIDENCE_PATH
    ),
}

assert (
    current_artifact_hashes["documents"]
    == persisted_inventory[
        "files"
    ]["documents"]["sha256"]
)
assert (
    current_artifact_hashes["postings"]
    == persisted_inventory[
        "files"
    ]["postings"]["sha256"]
)
assert (
    current_artifact_hashes[
        "configuration"
    ]
    == persisted_inventory[
        "files"
    ]["configuration"]["sha256"]
)
assert (
    current_artifact_hashes[
        "test_evaluation"
    ]
    == persisted_inventory[
        "files"
    ]["test_evaluation"]["sha256"]
)
assert (
    current_artifact_hashes[
        "reload_validation"
    ]
    == sparse_reload_evidence_sha256
)

sparse_index_manifest = {
    "artifact_type": (
        "NB15_SPARSE_INDEX_MANIFEST"
    ),
    "schema_version": "1.0",
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "canonical_inputs": {
        "chunk_path": (
            "data/phase2/chunks/"
            "nb11_benchmark_v1/"
            "baseline_phase1_char_safe_v1.jsonl"
        ),
        "chunk_sha256": (
            "3891c4ecad9332ba753cd605bbccd99da"
            "9d063f64934a30ee39e495349cffb50"
        ),
        "chunk_count": 3488,
        "golden_dataset_path": (
            "artifacts/phase2/golden_dataset_v2/"
            "golden_dataset_v2.jsonl"
        ),
        "golden_dataset_file_sha256": (
            "b9ca18e710686ddca48a25c637840b716"
            "adb306717cafd8ec7c96ce9e182e6f2"
        ),
        "golden_dataset_payload_sha256": (
            "539635b4c89e8f4eda41dde8106e04307"
            "c063f3fe0f5db402ef0440802233a03"
        ),
        "gold_mapping_sha256": (
            "42216f67665cfc50697789dffe4eb5508"
            "39ee22bcbfc0489accc407519dad4d1"
        ),
        "split_sha256": (
            "68df1196f683b6ab9a9df08c015f001b"
            "0caca056331eb80d0048aec783736d8f"
        ),
    },
    "configuration": {
        "configuration_id": (
            FROZEN_SPARSE_CONFIGURATION_ID
        ),
        "configuration_sha256": (
            FROZEN_SPARSE_CONFIGURATION_SHA256
        ),
        "analyzer_id": (
            FROZEN_SPARSE_ANALYZER_ID
        ),
        "bm25_implementation": (
            "local_robertson_bm25_v1"
        ),
        "k1": FROZEN_SPARSE_K1,
        "b": FROZEN_SPARSE_B,
        "mutation_allowed": False,
    },
    "index_statistics": {
        "document_count": len(
            reloaded_sparse_index.document_ids
        ),
        "vocabulary_size": len(
            reloaded_sparse_index.postings
        ),
        "posting_entry_count": int(
            reloaded_posting_entry_count
        ),
        "total_token_count": int(
            reloaded_document_lengths.sum()
        ),
        "average_document_length": float(
            reloaded_sparse_index
            .average_document_length
        ),
    },
    "logical_index_sha256": (
        reloaded_logical_index_sha256
    ),
    "files": {
        "documents": {
            "path": str(
                SPARSE_DOCUMENTS_PATH
            ),
            "sha256": (
                current_artifact_hashes[
                    "documents"
                ]
            ),
        },
        "postings": {
            "path": str(
                SPARSE_POSTINGS_PATH
            ),
            "sha256": (
                current_artifact_hashes[
                    "postings"
                ]
            ),
        },
        "configuration": {
            "path": str(
                SPARSE_CONFIGURATION_PATH
            ),
            "sha256": (
                current_artifact_hashes[
                    "configuration"
                ]
            ),
        },
        "test_evaluation": {
            "path": str(
                SPARSE_TEST_EVALUATION_PATH
            ),
            "sha256": (
                current_artifact_hashes[
                    "test_evaluation"
                ]
            ),
        },
        "inventory": {
            "path": str(
                SPARSE_INVENTORY_PATH
            ),
            "sha256": (
                current_artifact_hashes[
                    "inventory"
                ]
            ),
        },
        "reload_validation": {
            "path": str(
                SPARSE_RELOAD_EVIDENCE_PATH
            ),
            "sha256": (
                current_artifact_hashes[
                    "reload_validation"
                ]
            ),
        },
    },
    "validation": {
        "configuration_frozen_before_test": (
            True
        ),
        "test_query_count": len(
            test_candidate_ids
        ),
        "test_tuning_performed": False,
        "file_hash_validation": "PASS",
        "document_parity": "PASS",
        "posting_parity": "PASS",
        "score_parity": "PASS",
        "top_10_ranking_parity": "PASS",
        "metric_parity": "PASS",
        "reload_validation": "PASS",
    },
    "test_metrics": test_metrics,
    "known_limitations": {
        "weakest_source_type": (
            "transcript"
        ),
        "transcript_ndcg_at_10": float(
            test_source_type_metrics[
                "transcript"
            ]["metrics"]["ndcg_at_10"]
        ),
        "no_lexical_overlap_query_count": int(
            test_failure_taxonomy[
                "NO_QUERY_GOLD_"
                "LEXICAL_OVERLAP"
            ]
        ),
        "lexical_overlap_rank_below_10_count": int(
            test_failure_taxonomy[
                "LEXICAL_OVERLAP_"
                "RANK_BELOW_10"
            ]
        ),
        "next_stage_mitigation": (
            "NB16_DENSE_SPARSE_HYBRID_RETRIEVAL"
        ),
    },
    "dense_compatibility": {
        "dense_release_path": str(
            DENSE_RELEASE_PATH
        ),
        "dense_release_sha256": (
            EXPECTED_DENSE_RELEASE_SHA256
        ),
        "dense_logical_index_sha256": (
            EXPECTED_DENSE_LOGICAL_INDEX_SHA256
        ),
        "dense_point_count": 3488,
        "dense_mutated": False,
    },
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "manifest_status": (
        "PASS_COMPLETE_FROZEN"
    ),
}

atomic_write_text(
    SPARSE_MANIFEST_PATH,
    pretty_json_text(
        sparse_index_manifest
    ),
)

sparse_manifest_sha256 = sha256_file(
    SPARSE_MANIFEST_PATH
)

sparse_index_release = {
    "artifact_type": (
        "NB15_SPARSE_INDEX_RELEASE"
    ),
    "schema_version": "1.0",
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "manifest": {
        "path": str(
            SPARSE_MANIFEST_PATH
        ),
        "sha256": sparse_manifest_sha256,
    },
    "logical_index_sha256": (
        reloaded_logical_index_sha256
    ),
    "configuration_sha256": (
        FROZEN_SPARSE_CONFIGURATION_SHA256
    ),
    "document_count": 3488,
    "vocabulary_size": 27311,
    "posting_entry_count": 383637,
    "test_metrics": test_metrics,
    "reload_validation_status": "PASS",
    "mutation_allowed": False,
    "status": (
        "FROZEN_FOR_HYBRID_RETRIEVAL"
    ),
    "final_status": (
        "PASS_COMPLETE_FROZEN"
    ),
}

atomic_write_text(
    SPARSE_RELEASE_PATH,
    pretty_json_text(
        sparse_index_release
    ),
)

sparse_release_sha256 = sha256_file(
    SPARSE_RELEASE_PATH
)

nb16_sparse_handoff = {
    "artifact_type": (
        "NB16_SPARSE_INDEX_HANDOFF"
    ),
    "schema_version": "1.0",
    "source_notebook": (
        "15_sparse_index.ipynb"
    ),
    "consumer_notebook": (
        "16_hybrid_retrieval.ipynb"
    ),
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "sparse_release": {
        "path": str(
            SPARSE_RELEASE_PATH
        ),
        "sha256": sparse_release_sha256,
        "status": (
            "FROZEN_FOR_HYBRID_RETRIEVAL"
        ),
    },
    "manifest": {
        "path": str(
            SPARSE_MANIFEST_PATH
        ),
        "sha256": sparse_manifest_sha256,
    },
    "logical_index_sha256": (
        reloaded_logical_index_sha256
    ),
    "configuration_sha256": (
        FROZEN_SPARSE_CONFIGURATION_SHA256
    ),
    "loader_contract": {
        "documents_path": str(
            SPARSE_DOCUMENTS_PATH
        ),
        "documents_sha256": (
            current_artifact_hashes[
                "documents"
            ]
        ),
        "postings_path": str(
            SPARSE_POSTINGS_PATH
        ),
        "postings_sha256": (
            current_artifact_hashes[
                "postings"
            ]
        ),
        "configuration_path": str(
            SPARSE_CONFIGURATION_PATH
        ),
        "configuration_file_sha256": (
            current_artifact_hashes[
                "configuration"
            ]
        ),
        "analyzer_id": (
            FROZEN_SPARSE_ANALYZER_ID
        ),
        "bm25_implementation": (
            "local_robertson_bm25_v1"
        ),
        "k1": FROZEN_SPARSE_K1,
        "b": FROZEN_SPARSE_B,
        "top_k": 10,
    },
    "dense_pairing_contract": {
        "dense_release_path": str(
            DENSE_RELEASE_PATH
        ),
        "dense_release_sha256": (
            EXPECTED_DENSE_RELEASE_SHA256
        ),
        "dense_logical_index_sha256": (
            EXPECTED_DENSE_LOGICAL_INDEX_SHA256
        ),
        "shared_document_count": 3488,
    },
    "consumer_rules": {
        "validate_all_hashes_before_use": (
            True
        ),
        "preserve_document_ids": True,
        "sparse_mutation_allowed": False,
        "dense_mutation_allowed": False,
        "retune_sparse_on_hybrid_test": (
            False
        ),
    },
    "known_limitations": (
        sparse_index_manifest[
            "known_limitations"
        ]
    ),
    "handoff_status": "READY_FOR_NB16",
}

atomic_write_text(
    NB16_SPARSE_HANDOFF_PATH,
    pretty_json_text(
        nb16_sparse_handoff
    ),
)

nb16_sparse_handoff_sha256 = sha256_file(
    NB16_SPARSE_HANDOFF_PATH
)

sparse_index_acceptance = {
    "artifact_type": (
        "NB15_SPARSE_INDEX_ACCEPTANCE"
    ),
    "schema_version": "1.0",
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "objective": (
        "Build, benchmark, validate, freeze and "
        "handoff a Persian BM25 sparse index."
    ),
    "deliverables": {
        "persian_analyzer": "PASS",
        "tokenization_digit_zwnj_policy": (
            "PASS"
        ),
        "bm25_sparse_index": "PASS",
        "document_frequency_control": (
            "PASS"
        ),
        "names_and_exact_terms": "PASS",
        "independent_sparse_benchmark": (
            "PASS"
        ),
        "error_analysis": "PASS",
        "sparse_configuration_freeze": (
            "PASS"
        ),
        "artifact_reload_validation": (
            "PASS"
        ),
        "nb16_handoff": "PASS",
    },
    "test_metrics": test_metrics,
    "manifest": {
        "path": str(
            SPARSE_MANIFEST_PATH
        ),
        "sha256": sparse_manifest_sha256,
    },
    "release": {
        "path": str(
            SPARSE_RELEASE_PATH
        ),
        "sha256": sparse_release_sha256,
    },
    "nb16_handoff": {
        "path": str(
            NB16_SPARSE_HANDOFF_PATH
        ),
        "sha256": (
            nb16_sparse_handoff_sha256
        ),
    },
    "logical_index_sha256": (
        reloaded_logical_index_sha256
    ),
    "configuration_sha256": (
        FROZEN_SPARSE_CONFIGURATION_SHA256
    ),
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "acceptance_status": (
        "PASS_COMPLETE_FROZEN"
    ),
}

atomic_write_text(
    SPARSE_ACCEPTANCE_PATH,
    pretty_json_text(
        sparse_index_acceptance
    ),
)

sparse_acceptance_sha256 = sha256_file(
    SPARSE_ACCEPTANCE_PATH
)

assert all(
    path.exists()
    for path in (
        SPARSE_MANIFEST_PATH,
        SPARSE_RELEASE_PATH,
        NB16_SPARSE_HANDOFF_PATH,
        SPARSE_ACCEPTANCE_PATH,
    )
)
assert (
    sparse_index_release["status"]
    == "FROZEN_FOR_HYBRID_RETRIEVAL"
)
assert (
    nb16_sparse_handoff[
        "handoff_status"
    ]
    == "READY_FOR_NB16"
)
assert (
    sparse_index_acceptance[
        "acceptance_status"
    ]
    == "PASS_COMPLETE_FROZEN"
)

sparse_release_result = {
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "document_count": 3488,
    "vocabulary_size": 27311,
    "posting_entry_count": 383637,
    "configuration_sha256": (
        FROZEN_SPARSE_CONFIGURATION_SHA256
    ),
    "logical_index_sha256": (
        reloaded_logical_index_sha256
    ),
    "manifest": {
        "path": str(
            SPARSE_MANIFEST_PATH
        ),
        "sha256": sparse_manifest_sha256,
    },
    "release": {
        "path": str(
            SPARSE_RELEASE_PATH
        ),
        "sha256": sparse_release_sha256,
        "status": (
            "FROZEN_FOR_HYBRID_RETRIEVAL"
        ),
    },
    "nb16_handoff": {
        "path": str(
            NB16_SPARSE_HANDOFF_PATH
        ),
        "sha256": (
            nb16_sparse_handoff_sha256
        ),
        "status": "READY_FOR_NB16",
    },
    "acceptance": {
        "path": str(
            SPARSE_ACCEPTANCE_PATH
        ),
        "sha256": (
            sparse_acceptance_sha256
        ),
    },
    "dense_compatibility": "PASS",
    "paid_api_calls": 0,
    "embedding_recomputation": False,
    "finalization_status": (
        "PASS_COMPLETE_FROZEN"
    ),
}

print(
    json.dumps(
        sparse_release_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "release_id": "rag_finance_phase2_sparse_v1",
  "document_count": 3488,
  "vocabulary_size": 27311,
  "posting_entry_count": 383637,
  "configuration_sha256": "96d66676d20bacd764ec9a163995dd48266c90c2acea1c8a5f77d95d9efb6ecc",
  "logical_index_sha256": "ce09bb4749da03c9dfa1a849de18fc8c490ce6b09eb34a30a96b9fedc478a8eb",
  "manifest": {
    "path": "data/phase2/index/sparse_index_v1/sparse_index_manifest.json",
    "sha256": "59adbaea056e02eec3369967b99b1e154b258873b7d5498fb1764432eb30a2f8"
  },
  "release": {
    "path": "artifacts/phase2/sparse_index/sparse_index_release_v1.json",
    "sha256": "dbffd3f43bff52929b05f6070bfa3d062655df87a6e18d2263163f1c5594e7cf",
    "status": "FROZEN_FOR_HYBRID_RETRIEVAL"
  },
  "nb16_handoff": {
    "path": "artifacts/phase2/sparse_index/nb16_sparse_index_handoff_v1.json",
    "sha256": "896ac8712130a3cf5da8a9eafcf91199d232c0587918fa25aacea49708e6e527",
    "status": "READY_FOR_NB16"
  },
  "acceptance": {
    "path": "reports/phase2/sparse_index_a

<div dir="rtl" align="right">

## نرمال‌سازی مسیرهای Handoff و بازصدور زنجیره SHA

Dense Release معتبر NB14 با SHA فریز‌شده پیدا و اعتبارسنجی شد، اما مسیر Resolve‌شده آن ممکن است به‌صورت Absolute و وابسته به سیستم محلی در Manifest یا Handoff ثبت شده باشد.

برای حفظ قابلیت انتقال Repository، مسیر Dense Release در تمام قراردادهای NB15 به مسیر Canonical نسبی زیر تبدیل می‌شود:

`artifacts/phase2/dense_index/dense_index_release_v1.json`

پس از این تغییر، Manifest، Release، Handoff و Acceptance به‌ترتیب بازنویسی می‌شوند و SHAهای وابسته نیز دوباره محاسبه خواهند شد. محتوای Sparse Index، پیکربندی، Metricها و Logical Index Hash تغییر نمی‌کنند.

</div>

In [32]:
# Code Cell 23 — Normalize portable paths and reissue dependent hashes

CANONICAL_DENSE_RELEASE_PATH = (
    "artifacts/phase2/dense_index/"
    "dense_index_release_v1.json"
)

assert DENSE_RELEASE_PATH.is_file()
assert (
    sha256_file(DENSE_RELEASE_PATH)
    == EXPECTED_DENSE_RELEASE_SHA256
)

final_sparse_manifest = json.loads(
    SPARSE_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    final_sparse_manifest[
        "logical_index_sha256"
    ]
    == reloaded_logical_index_sha256
)

final_sparse_manifest[
    "dense_compatibility"
]["dense_release_path"] = (
    CANONICAL_DENSE_RELEASE_PATH
)

atomic_write_text(
    SPARSE_MANIFEST_PATH,
    pretty_json_text(
        final_sparse_manifest
    ),
)

final_sparse_manifest_sha256 = (
    sha256_file(
        SPARSE_MANIFEST_PATH
    )
)

final_sparse_release = json.loads(
    SPARSE_RELEASE_PATH.read_text(
        encoding="utf-8"
    )
)

final_sparse_release[
    "manifest"
]["sha256"] = (
    final_sparse_manifest_sha256
)

atomic_write_text(
    SPARSE_RELEASE_PATH,
    pretty_json_text(
        final_sparse_release
    ),
)

final_sparse_release_sha256 = (
    sha256_file(
        SPARSE_RELEASE_PATH
    )
)

final_nb16_sparse_handoff = json.loads(
    NB16_SPARSE_HANDOFF_PATH.read_text(
        encoding="utf-8"
    )
)

final_nb16_sparse_handoff[
    "manifest"
]["sha256"] = (
    final_sparse_manifest_sha256
)

final_nb16_sparse_handoff[
    "sparse_release"
]["sha256"] = (
    final_sparse_release_sha256
)

final_nb16_sparse_handoff[
    "dense_pairing_contract"
]["dense_release_path"] = (
    CANONICAL_DENSE_RELEASE_PATH
)

atomic_write_text(
    NB16_SPARSE_HANDOFF_PATH,
    pretty_json_text(
        final_nb16_sparse_handoff
    ),
)

final_nb16_sparse_handoff_sha256 = (
    sha256_file(
        NB16_SPARSE_HANDOFF_PATH
    )
)

final_sparse_acceptance = json.loads(
    SPARSE_ACCEPTANCE_PATH.read_text(
        encoding="utf-8"
    )
)

final_sparse_acceptance[
    "manifest"
]["sha256"] = (
    final_sparse_manifest_sha256
)

final_sparse_acceptance[
    "release"
]["sha256"] = (
    final_sparse_release_sha256
)

final_sparse_acceptance[
    "nb16_handoff"
]["sha256"] = (
    final_nb16_sparse_handoff_sha256
)

atomic_write_text(
    SPARSE_ACCEPTANCE_PATH,
    pretty_json_text(
        final_sparse_acceptance
    ),
)

final_sparse_acceptance_sha256 = (
    sha256_file(
        SPARSE_ACCEPTANCE_PATH
    )
)

assert (
    final_sparse_manifest[
        "dense_compatibility"
    ]["dense_release_path"]
    == CANONICAL_DENSE_RELEASE_PATH
)

assert (
    final_nb16_sparse_handoff[
        "dense_pairing_contract"
    ]["dense_release_path"]
    == CANONICAL_DENSE_RELEASE_PATH
)

assert not Path(
    final_sparse_manifest[
        "dense_compatibility"
    ]["dense_release_path"]
).is_absolute()

assert not Path(
    final_nb16_sparse_handoff[
        "dense_pairing_contract"
    ]["dense_release_path"]
).is_absolute()

assert (
    final_sparse_release[
        "manifest"
    ]["sha256"]
    == final_sparse_manifest_sha256
)

assert (
    final_nb16_sparse_handoff[
        "sparse_release"
    ]["sha256"]
    == final_sparse_release_sha256
)

assert (
    final_nb16_sparse_handoff[
        "manifest"
    ]["sha256"]
    == final_sparse_manifest_sha256
)

assert (
    final_sparse_acceptance[
        "manifest"
    ]["sha256"]
    == final_sparse_manifest_sha256
)

assert (
    final_sparse_acceptance[
        "release"
    ]["sha256"]
    == final_sparse_release_sha256
)

assert (
    final_sparse_acceptance[
        "nb16_handoff"
    ]["sha256"]
    == final_nb16_sparse_handoff_sha256
)

assert (
    final_sparse_manifest[
        "logical_index_sha256"
    ]
    == sparse_logical_index_sha256
)

assert (
    final_sparse_release[
        "logical_index_sha256"
    ]
    == sparse_logical_index_sha256
)

portable_release_result = {
    "release_id": (
        "rag_finance_phase2_sparse_v1"
    ),
    "portable_dense_release_path": (
        CANONICAL_DENSE_RELEASE_PATH
    ),
    "configuration_sha256": (
        FROZEN_SPARSE_CONFIGURATION_SHA256
    ),
    "logical_index_sha256": (
        sparse_logical_index_sha256
    ),
    "manifest": {
        "path": str(
            SPARSE_MANIFEST_PATH
        ),
        "sha256": (
            final_sparse_manifest_sha256
        ),
    },
    "release": {
        "path": str(
            SPARSE_RELEASE_PATH
        ),
        "sha256": (
            final_sparse_release_sha256
        ),
        "status": (
            final_sparse_release["status"]
        ),
    },
    "nb16_handoff": {
        "path": str(
            NB16_SPARSE_HANDOFF_PATH
        ),
        "sha256": (
            final_nb16_sparse_handoff_sha256
        ),
        "status": (
            final_nb16_sparse_handoff[
                "handoff_status"
            ]
        ),
    },
    "acceptance": {
        "path": str(
            SPARSE_ACCEPTANCE_PATH
        ),
        "sha256": (
            final_sparse_acceptance_sha256
        ),
        "status": (
            final_sparse_acceptance[
                "acceptance_status"
            ]
        ),
    },
    "logical_index_changed": False,
    "configuration_changed": False,
    "portable_path_validation": "PASS",
    "finalization_status": (
        "PASS_COMPLETE_FROZEN"
    ),
}

print(
    json.dumps(
        portable_release_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "release_id": "rag_finance_phase2_sparse_v1",
  "portable_dense_release_path": "artifacts/phase2/dense_index/dense_index_release_v1.json",
  "configuration_sha256": "96d66676d20bacd764ec9a163995dd48266c90c2acea1c8a5f77d95d9efb6ecc",
  "logical_index_sha256": "ce09bb4749da03c9dfa1a849de18fc8c490ce6b09eb34a30a96b9fedc478a8eb",
  "manifest": {
    "path": "data/phase2/index/sparse_index_v1/sparse_index_manifest.json",
    "sha256": "ce8192a64e555b00c2708ecde3885da870bb2e6725640c6fd194f885300e8fac"
  },
  "release": {
    "path": "artifacts/phase2/sparse_index/sparse_index_release_v1.json",
    "sha256": "879f52850a71629f02437c1ab1173e3f9e6c55ec5441ad4649784be63935c51b",
    "status": "FROZEN_FOR_HYBRID_RETRIEVAL"
  },
  "nb16_handoff": {
    "path": "artifacts/phase2/sparse_index/nb16_sparse_index_handoff_v1.json",
    "sha256": "e4070684ba1662de35e2fbdf2737560aee1b7c67a694e6dab3bd98be2f578439",
    "status": "READY_FOR_NB16"
  },
  "acceptance": {
    "path": "reports/phase2/sparse_i

<div dir="rtl" align="right">

## تثبیت Artifactهای NB15 در ریشه Canonical پروژه

اجرای Notebook ممکن است از پوشه‌ای پایین‌تر از ریشه Repository انجام شده باشد. در چنین شرایطی، مسیرهای نسبی ظاهراً صحیح‌اند اما فایل‌ها ممکن است در Working Directory اشتباه ساخته شده باشند.

در این مرحله ریشه واقعی پروژه از Dense Release معتبر NB14 استخراج می‌شود و تمام Artifactهای NB15 به مسیرهای Canonical زیر منتقل یا کپی می‌شوند:

- `data/phase2/index/sparse_index_v1/`
- `artifacts/phase2/sparse_index/`
- `reports/phase2/`

محتوای فایل‌ها تغییر نمی‌کند و SHA-256 هر فایل پیش و پس از Materialization باید دقیقاً برابر باقی بماند. این کنترل تضمین می‌کند NB16 بتواند فایل‌ها را مستقیماً از ریشه Repository بارگذاری کند.

</div>

In [33]:
# Code Cell 24 — Materialize NB15 artifacts under the canonical project root

import shutil

canonical_project_root = (
    DENSE_RELEASE_PATH.resolve().parents[3]
)

expected_dense_release_absolute_path = (
    canonical_project_root
    / CANONICAL_DENSE_RELEASE_PATH
)

assert (
    expected_dense_release_absolute_path.resolve()
    == DENSE_RELEASE_PATH.resolve()
)
assert (
    sha256_file(
        expected_dense_release_absolute_path
    )
    == EXPECTED_DENSE_RELEASE_SHA256
)

canonical_artifact_targets = {
    "documents": (
        canonical_project_root
        / "data/phase2/index/sparse_index_v1/"
        "sparse_documents_v1.jsonl"
    ),
    "postings": (
        canonical_project_root
        / "data/phase2/index/sparse_index_v1/"
        "sparse_postings_v1.jsonl"
    ),
    "configuration": (
        canonical_project_root
        / "data/phase2/index/sparse_index_v1/"
        "sparse_configuration_v1.json"
    ),
    "inventory": (
        canonical_project_root
        / "data/phase2/index/sparse_index_v1/"
        "sparse_index_inventory_v1.json"
    ),
    "reload_validation": (
        canonical_project_root
        / "data/phase2/index/sparse_index_v1/"
        "sparse_reload_validation_v1.json"
    ),
    "manifest": (
        canonical_project_root
        / "data/phase2/index/sparse_index_v1/"
        "sparse_index_manifest.json"
    ),
    "test_evaluation": (
        canonical_project_root
        / "reports/phase2/"
        "sparse_test_evaluation_v1.json"
    ),
    "acceptance": (
        canonical_project_root
        / "reports/phase2/"
        "sparse_index_acceptance_v1.json"
    ),
    "release": (
        canonical_project_root
        / "artifacts/phase2/sparse_index/"
        "sparse_index_release_v1.json"
    ),
    "nb16_handoff": (
        canonical_project_root
        / "artifacts/phase2/sparse_index/"
        "nb16_sparse_index_handoff_v1.json"
    ),
}

current_artifact_sources = {
    "documents": SPARSE_DOCUMENTS_PATH.resolve(),
    "postings": SPARSE_POSTINGS_PATH.resolve(),
    "configuration": (
        SPARSE_CONFIGURATION_PATH.resolve()
    ),
    "inventory": (
        SPARSE_INVENTORY_PATH.resolve()
    ),
    "reload_validation": (
        SPARSE_RELOAD_EVIDENCE_PATH.resolve()
    ),
    "manifest": (
        SPARSE_MANIFEST_PATH.resolve()
    ),
    "test_evaluation": (
        SPARSE_TEST_EVALUATION_PATH.resolve()
    ),
    "acceptance": (
        SPARSE_ACCEPTANCE_PATH.resolve()
    ),
    "release": (
        SPARSE_RELEASE_PATH.resolve()
    ),
    "nb16_handoff": (
        NB16_SPARSE_HANDOFF_PATH.resolve()
    ),
}

expected_final_hashes = {
    "documents": current_artifact_hashes[
        "documents"
    ],
    "postings": current_artifact_hashes[
        "postings"
    ],
    "configuration": current_artifact_hashes[
        "configuration"
    ],
    "inventory": current_artifact_hashes[
        "inventory"
    ],
    "reload_validation": (
        sparse_reload_evidence_sha256
    ),
    "manifest": (
        final_sparse_manifest_sha256
    ),
    "test_evaluation": current_artifact_hashes[
        "test_evaluation"
    ],
    "acceptance": (
        final_sparse_acceptance_sha256
    ),
    "release": (
        final_sparse_release_sha256
    ),
    "nb16_handoff": (
        final_nb16_sparse_handoff_sha256
    ),
}

materialization_actions = {}

for artifact_name, source_path in (
    current_artifact_sources.items()
):
    target_path = canonical_artifact_targets[
        artifact_name
    ]

    assert source_path.is_file()
    assert (
        sha256_file(source_path)
        == expected_final_hashes[
            artifact_name
        ]
    )

    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if (
        source_path.resolve()
        == target_path.resolve()
    ):
        materialization_actions[
            artifact_name
        ] = "ALREADY_CANONICAL"
    else:
        temporary_target_path = (
            target_path.with_name(
                target_path.name
                + ".materializing"
            )
        )

        shutil.copyfile(
            source_path,
            temporary_target_path,
        )

        assert (
            sha256_file(
                temporary_target_path
            )
            == expected_final_hashes[
                artifact_name
            ]
        )

        os.replace(
            temporary_target_path,
            target_path,
        )

        materialization_actions[
            artifact_name
        ] = "COPIED_TO_CANONICAL_ROOT"

    assert target_path.is_file()
    assert (
        sha256_file(target_path)
        == expected_final_hashes[
            artifact_name
        ]
    )

canonical_sparse_manifest_path = (
    canonical_artifact_targets[
        "manifest"
    ]
)
canonical_sparse_release_path = (
    canonical_artifact_targets[
        "release"
    ]
)
canonical_nb16_handoff_path = (
    canonical_artifact_targets[
        "nb16_handoff"
    ]
)
canonical_sparse_acceptance_path = (
    canonical_artifact_targets[
        "acceptance"
    ]
)

canonical_manifest = json.loads(
    canonical_sparse_manifest_path.read_text(
        encoding="utf-8"
    )
)

canonical_release = json.loads(
    canonical_sparse_release_path.read_text(
        encoding="utf-8"
    )
)

canonical_handoff = json.loads(
    canonical_nb16_handoff_path.read_text(
        encoding="utf-8"
    )
)

canonical_acceptance = json.loads(
    canonical_sparse_acceptance_path.read_text(
        encoding="utf-8"
    )
)

assert (
    canonical_manifest[
        "manifest_status"
    ]
    == "PASS_COMPLETE_FROZEN"
)
assert (
    canonical_release["status"]
    == "FROZEN_FOR_HYBRID_RETRIEVAL"
)
assert (
    canonical_release["final_status"]
    == "PASS_COMPLETE_FROZEN"
)
assert (
    canonical_handoff["handoff_status"]
    == "READY_FOR_NB16"
)
assert (
    canonical_acceptance[
        "acceptance_status"
    ]
    == "PASS_COMPLETE_FROZEN"
)

assert (
    canonical_release[
        "manifest"
    ]["sha256"]
    == sha256_file(
        canonical_sparse_manifest_path
    )
)

assert (
    canonical_handoff[
        "sparse_release"
    ]["sha256"]
    == sha256_file(
        canonical_sparse_release_path
    )
)

assert (
    canonical_handoff[
        "manifest"
    ]["sha256"]
    == sha256_file(
        canonical_sparse_manifest_path
    )
)

assert (
    canonical_acceptance[
        "manifest"
    ]["sha256"]
    == sha256_file(
        canonical_sparse_manifest_path
    )
)

assert (
    canonical_acceptance[
        "release"
    ]["sha256"]
    == sha256_file(
        canonical_sparse_release_path
    )
)

assert (
    canonical_acceptance[
        "nb16_handoff"
    ]["sha256"]
    == sha256_file(
        canonical_nb16_handoff_path
    )
)

canonical_materialization_result = {
    "canonical_project_root": str(
        canonical_project_root
    ),
    "artifact_count": len(
        canonical_artifact_targets
    ),
    "actions": materialization_actions,
    "configuration_sha256": (
        FROZEN_SPARSE_CONFIGURATION_SHA256
    ),
    "logical_index_sha256": (
        sparse_logical_index_sha256
    ),
    "manifest_sha256": sha256_file(
        canonical_sparse_manifest_path
    ),
    "release_sha256": sha256_file(
        canonical_sparse_release_path
    ),
    "nb16_handoff_sha256": sha256_file(
        canonical_nb16_handoff_path
    ),
    "acceptance_sha256": sha256_file(
        canonical_sparse_acceptance_path
    ),
    "all_hashes_preserved": True,
    "canonical_paths_validated": True,
    "materialization_status": "PASS",
}

print(
    json.dumps(
        canonical_materialization_result,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "canonical_project_root": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance",
  "artifact_count": 10,
  "actions": {
    "documents": "COPIED_TO_CANONICAL_ROOT",
    "postings": "COPIED_TO_CANONICAL_ROOT",
    "configuration": "COPIED_TO_CANONICAL_ROOT",
    "inventory": "COPIED_TO_CANONICAL_ROOT",
    "reload_validation": "COPIED_TO_CANONICAL_ROOT",
    "manifest": "COPIED_TO_CANONICAL_ROOT",
    "test_evaluation": "COPIED_TO_CANONICAL_ROOT",
    "acceptance": "COPIED_TO_CANONICAL_ROOT",
    "release": "COPIED_TO_CANONICAL_ROOT",
    "nb16_handoff": "COPIED_TO_CANONICAL_ROOT"
  },
  "configuration_sha256": "96d66676d20bacd764ec9a163995dd48266c90c2acea1c8a5f77d95d9efb6ecc",
  "logical_index_sha256": "ce09bb4749da03c9dfa1a849de18fc8c490ce6b09eb34a30a96b9fedc478a8eb",
  "manifest_sha256": "ce8192a64e555b00c2708ecde3885da870bb2e6725640c6fd194f885300e8fac",
  "release_sha256": "879f52850a71629f02437c1ab1173e3f9e6c55ec5441ad4649784be63935c51b",
  "nb16_handoff_s

<div dir="rtl" align="right">

## Final Health Check نوت‌بوک NB15

این سل باید پس از ذخیره Notebook و اجرای کامل مسیر `Restart Kernel → Run All` اجرا شود.

کنترل نهایی شامل موارد زیر است:

1. ترتیب و یکتایی شماره Code Cellها؛
2. صحت Syntax تمام سل‌های کد؛
3. صحت قالب RTL تمام Markdown Cellها؛
4. تکمیل تمام مأموریت‌های رسمی NB15؛
5. اعتبار Hash ورودی‌های Canonical؛
6. اعتبار Sparse Index، Manifest، Release و Handoff؛
7. اعتبار اتصال به Dense Release فریز‌شده NB14؛
8. عبور ارزیابی مستقل Test و Reload Validation؛
9. قرارگرفتن Artifactها در ریشه Canonical پروژه؛
10. آماده‌بودن خروجی برای NB16 بدون وابستگی به Kernel جاری.

این سل هیچ پیکربندی یا Artifact اصلی را تغییر نمی‌دهد و فقط وضعیت نهایی Notebook و زنجیره تحویل را بررسی می‌کند.

</div>

In [35]:
# Code Cell 25 — Final notebook and release health check

import ast

NOTEBOOK_PATH = (
    canonical_project_root
    / "notebooks/phase2/15_sparse_index.ipynb"
)

CANONICAL_CHUNK_PATH = (
    canonical_project_root
    / "data/phase2/chunks/"
    "nb11_benchmark_v1/"
    "baseline_phase1_char_safe_v1.jsonl"
)

CANONICAL_GOLDEN_PATH = (
    canonical_project_root
    / "artifacts/phase2/golden_dataset_v2/"
    "golden_dataset_v2.jsonl"
)

CANONICAL_DENSE_RELEASE_FILE = (
    canonical_project_root
    / CANONICAL_DENSE_RELEASE_PATH
)

assert NOTEBOOK_PATH.is_file(), (
    "Save 15_sparse_index.ipynb before "
    "running Final Health Check."
)

notebook_payload = json.loads(
    NOTEBOOK_PATH.read_text(
        encoding="utf-8"
    )
)

assert notebook_payload[
    "nbformat"
] == 4

code_cells = [
    cell
    for cell in notebook_payload["cells"]
    if cell.get("cell_type") == "code"
]

markdown_cells = [
    cell
    for cell in notebook_payload["cells"]
    if cell.get("cell_type") == "markdown"
]

code_cell_pattern = re.compile(
    r"^\s*# Code Cell (\d{2})\b"
)

code_numbers = []
missing_code_headers = []
syntax_errors = []

for code_position, cell in enumerate(
    code_cells,
    start=1,
):
    source = "".join(
        cell.get("source", [])
    )

    header_match = code_cell_pattern.search(
        source
    )

    if header_match is None:
        missing_code_headers.append(
            code_position
        )
    else:
        code_numbers.append(
            int(header_match.group(1))
        )

    try:
        ast.parse(source)
    except SyntaxError as error:
        syntax_errors.append(
            {
                "code_position": (
                    code_position
                ),
                "line": error.lineno,
                "message": error.msg,
            }
        )

assert not missing_code_headers, (
    "Code cells without standard header: "
    f"{missing_code_headers}"
)
assert not syntax_errors, syntax_errors
assert code_numbers == list(
    range(1, 26)
), (
    "Code Cell numbering is not 01→25: "
    f"{code_numbers}"
)
assert len(code_numbers) == len(
    set(code_numbers)
)

invalid_rtl_markdown_cells = []

for markdown_position, cell in enumerate(
    markdown_cells,
    start=1,
):
    source = "".join(
        cell.get("source", [])
    ).strip()

    if not (
        source.startswith(
            '<div dir="rtl" align="right">'
        )
        and source.endswith("</div>")
    ):
        invalid_rtl_markdown_cells.append(
            markdown_position
        )

assert not invalid_rtl_markdown_cells, (
    "Markdown cells without required RTL "
    f"wrapper: {invalid_rtl_markdown_cells}"
)

assert CANONICAL_CHUNK_PATH.is_file()
assert CANONICAL_GOLDEN_PATH.is_file()
assert CANONICAL_DENSE_RELEASE_FILE.is_file()

canonical_input_hash_validation = {
    "chunk": (
        sha256_file(
            CANONICAL_CHUNK_PATH
        )
        == (
            "3891c4ecad9332ba753cd605bbccd99da"
            "9d063f64934a30ee39e495349cffb50"
        )
    ),
    "golden_dataset": (
        sha256_file(
            CANONICAL_GOLDEN_PATH
        )
        == (
            "b9ca18e710686ddca48a25c637840b716"
            "adb306717cafd8ec7c96ce9e182e6f2"
        )
    ),
    "dense_release": (
        sha256_file(
            CANONICAL_DENSE_RELEASE_FILE
        )
        == EXPECTED_DENSE_RELEASE_SHA256
    ),
}

assert all(
    canonical_input_hash_validation.values()
)

canonical_final_paths = {
    "documents": (
        canonical_artifact_targets[
            "documents"
        ]
    ),
    "postings": (
        canonical_artifact_targets[
            "postings"
        ]
    ),
    "configuration": (
        canonical_artifact_targets[
            "configuration"
        ]
    ),
    "inventory": (
        canonical_artifact_targets[
            "inventory"
        ]
    ),
    "reload_validation": (
        canonical_artifact_targets[
            "reload_validation"
        ]
    ),
    "manifest": (
        canonical_artifact_targets[
            "manifest"
        ]
    ),
    "test_evaluation": (
        canonical_artifact_targets[
            "test_evaluation"
        ]
    ),
    "acceptance": (
        canonical_artifact_targets[
            "acceptance"
        ]
    ),
    "release": (
        canonical_artifact_targets[
            "release"
        ]
    ),
    "nb16_handoff": (
        canonical_artifact_targets[
            "nb16_handoff"
        ]
    ),
}

assert all(
    path.is_file()
    for path in canonical_final_paths.values()
)

canonical_final_hashes = {
    artifact_name: sha256_file(
        artifact_path
    )
    for artifact_name, artifact_path
    in canonical_final_paths.items()
}

assert (
    canonical_final_hashes["documents"]
    == expected_final_hashes["documents"]
)
assert (
    canonical_final_hashes["postings"]
    == expected_final_hashes["postings"]
)
assert (
    canonical_final_hashes["configuration"]
    == expected_final_hashes[
        "configuration"
    ]
)
assert (
    canonical_final_hashes["inventory"]
    == expected_final_hashes["inventory"]
)
assert (
    canonical_final_hashes[
        "reload_validation"
    ]
    == expected_final_hashes[
        "reload_validation"
    ]
)
assert (
    canonical_final_hashes["manifest"]
    == final_sparse_manifest_sha256
)
assert (
    canonical_final_hashes["release"]
    == final_sparse_release_sha256
)
assert (
    canonical_final_hashes[
        "nb16_handoff"
    ]
    == final_nb16_sparse_handoff_sha256
)
assert (
    canonical_final_hashes["acceptance"]
    == final_sparse_acceptance_sha256
)
assert (
    canonical_final_hashes[
        "test_evaluation"
    ]
    == expected_final_hashes[
        "test_evaluation"
    ]
)

health_manifest = json.loads(
    canonical_final_paths[
        "manifest"
    ].read_text(
        encoding="utf-8"
    )
)

health_release = json.loads(
    canonical_final_paths[
        "release"
    ].read_text(
        encoding="utf-8"
    )
)

health_handoff = json.loads(
    canonical_final_paths[
        "nb16_handoff"
    ].read_text(
        encoding="utf-8"
    )
)

health_acceptance = json.loads(
    canonical_final_paths[
        "acceptance"
    ].read_text(
        encoding="utf-8"
    )
)

health_reload_validation = json.loads(
    canonical_final_paths[
        "reload_validation"
    ].read_text(
        encoding="utf-8"
    )
)

health_test_evaluation = json.loads(
    canonical_final_paths[
        "test_evaluation"
    ].read_text(
        encoding="utf-8"
    )
)

assert (
    health_manifest["manifest_status"]
    == "PASS_COMPLETE_FROZEN"
)
assert (
    health_release["status"]
    == "FROZEN_FOR_HYBRID_RETRIEVAL"
)
assert (
    health_release["final_status"]
    == "PASS_COMPLETE_FROZEN"
)
assert (
    health_handoff["handoff_status"]
    == "READY_FOR_NB16"
)
assert (
    health_acceptance[
        "acceptance_status"
    ]
    == "PASS_COMPLETE_FROZEN"
)
assert (
    health_reload_validation[
        "reload_validation_status"
    ]
    == "PASS"
)
assert (
    health_test_evaluation[
        "independent_test_status"
    ]
    == "PASS"
)

assert (
    health_manifest[
        "logical_index_sha256"
    ]
    == sparse_logical_index_sha256
)
assert (
    health_release[
        "logical_index_sha256"
    ]
    == sparse_logical_index_sha256
)
assert (
    health_handoff[
        "logical_index_sha256"
    ]
    == sparse_logical_index_sha256
)
assert (
    health_acceptance[
        "logical_index_sha256"
    ]
    == sparse_logical_index_sha256
)

assert (
    health_release[
        "configuration_sha256"
    ]
    == FROZEN_SPARSE_CONFIGURATION_SHA256
)
assert (
    health_handoff[
        "configuration_sha256"
    ]
    == FROZEN_SPARSE_CONFIGURATION_SHA256
)
assert (
    health_acceptance[
        "configuration_sha256"
    ]
    == FROZEN_SPARSE_CONFIGURATION_SHA256
)

assert (
    health_release[
        "manifest"
    ]["sha256"]
    == canonical_final_hashes[
        "manifest"
    ]
)
assert (
    health_handoff[
        "sparse_release"
    ]["sha256"]
    == canonical_final_hashes[
        "release"
    ]
)
assert (
    health_handoff[
        "manifest"
    ]["sha256"]
    == canonical_final_hashes[
        "manifest"
    ]
)
assert (
    health_acceptance[
        "manifest"
    ]["sha256"]
    == canonical_final_hashes[
        "manifest"
    ]
)
assert (
    health_acceptance[
        "release"
    ]["sha256"]
    == canonical_final_hashes[
        "release"
    ]
)
assert (
    health_acceptance[
        "nb16_handoff"
    ]["sha256"]
    == canonical_final_hashes[
        "nb16_handoff"
    ]
)

required_deliverables = {
    "persian_analyzer",
    "tokenization_digit_zwnj_policy",
    "bm25_sparse_index",
    "document_frequency_control",
    "names_and_exact_terms",
    "independent_sparse_benchmark",
    "error_analysis",
    "sparse_configuration_freeze",
    "artifact_reload_validation",
    "nb16_handoff",
}

assert required_deliverables == set(
    health_acceptance["deliverables"]
)

assert all(
    status == "PASS"
    for status
    in health_acceptance[
        "deliverables"
    ].values()
)

assert (
    health_reload_validation[
        "retrieval_validation"
    ]["score_mismatch_count"]
    == 0
)
assert (
    health_reload_validation[
        "retrieval_validation"
    ]["ranking_mismatch_count"]
    == 0
)
assert (
    health_reload_validation[
        "retrieval_validation"
    ]["maximum_metric_absolute_delta"]
    == 0.0
)
assert (
    health_manifest[
        "validation"
    ]["test_tuning_performed"]
    is False
)
assert (
    health_manifest[
        "configuration"
    ]["mutation_allowed"]
    is False
)
assert (
    health_handoff[
        "consumer_rules"
    ]["sparse_mutation_allowed"]
    is False
)
assert (
    health_handoff[
        "consumer_rules"
    ]["dense_mutation_allowed"]
    is False
)

nb15_final_health = {
    "nb15_objective": "PASS",
    "modular_nb15_deliverables": "PASS",
    "execution_health": "PASS",
    "syntax_health": "PASS",
    "markdown_rtl_health": "PASS",
    "canonical_input_hashes": "PASS",
    "sparse_index_gate": "PASS",
    "independent_test_gate": "PASS",
    "reload_validation_gate": "PASS",
    "portable_artifact_gate": "PASS",
    "dense_compatibility_gate": "PASS",
    "code_cell_count": len(code_cells),
    "code_cell_numbering": "01_TO_25_PASS",
    "document_count": health_release[
        "document_count"
    ],
    "vocabulary_size": health_release[
        "vocabulary_size"
    ],
    "posting_entry_count": health_release[
        "posting_entry_count"
    ],
    "test_query_count": (
        health_test_evaluation[
            "evaluation_scope"
        ]["query_count"]
    ),
    "test_metrics": (
        health_test_evaluation["metrics"]
    ),
    "configuration_sha256": (
        FROZEN_SPARSE_CONFIGURATION_SHA256
    ),
    "logical_index_sha256": (
        sparse_logical_index_sha256
    ),
    "manifest": {
        "path": (
            "data/phase2/index/"
            "sparse_index_v1/"
            "sparse_index_manifest.json"
        ),
        "sha256": (
            canonical_final_hashes[
                "manifest"
            ]
        ),
    },
    "release": {
        "path": (
            "artifacts/phase2/sparse_index/"
            "sparse_index_release_v1.json"
        ),
        "sha256": (
            canonical_final_hashes[
                "release"
            ]
        ),
        "status": (
            health_release["status"]
        ),
    },
    "nb16_handoff": {
        "path": (
            "artifacts/phase2/sparse_index/"
            "nb16_sparse_index_handoff_v1.json"
        ),
        "sha256": (
            canonical_final_hashes[
                "nb16_handoff"
            ]
        ),
        "status": (
            health_handoff[
                "handoff_status"
            ]
        ),
    },
    "acceptance": {
        "path": (
            "reports/phase2/"
            "sparse_index_acceptance_v1.json"
        ),
        "sha256": (
            canonical_final_hashes[
                "acceptance"
            ]
        ),
    },
    "dense_index_mutated": False,
    "embedding_recomputation": False,
    "paid_api_calls": 0,
    "next_execution_notebook": (
        "16_hybrid_retrieval.ipynb"
    ),
    "final_status": (
        "PASS_COMPLETE_FROZEN"
    ),
}

print(
    json.dumps(
        nb15_final_health,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "\nNB15 FINAL HEALTH: "
    "PASS — COMPLETE / RELOAD-VALIDATED / "
    "FROZEN / READY FOR NB16"
)

{
  "nb15_objective": "PASS",
  "modular_nb15_deliverables": "PASS",
  "execution_health": "PASS",
  "syntax_health": "PASS",
  "markdown_rtl_health": "PASS",
  "canonical_input_hashes": "PASS",
  "sparse_index_gate": "PASS",
  "independent_test_gate": "PASS",
  "reload_validation_gate": "PASS",
  "portable_artifact_gate": "PASS",
  "dense_compatibility_gate": "PASS",
  "code_cell_count": 25,
  "code_cell_numbering": "01_TO_25_PASS",
  "document_count": 3488,
  "vocabulary_size": 27311,
  "posting_entry_count": 383637,
  "test_query_count": 1149,
  "test_metrics": {
    "hit_at_10": 0.7075718015665796,
    "hit_at_5": 0.6518711923411662,
    "ndcg_at_10": 0.49020522695816876,
    "ndcg_at_5": 0.46879829553108543,
    "precision_at_10": 0.09451697127937336,
    "precision_at_5": 0.16675369886858138,
    "recall_at_10": 0.5535776451572797,
    "recall_at_5": 0.49356686974180447,
    "reciprocal_rank": 0.5567650268694656
  },
  "configuration_sha256": "96d66676d20bacd764ec9a163995dd48266c

<div dir="rtl" align="right">

# نتیجه نهایی NB15 — Persian Sparse Index V1

هدف Notebook 15 صرفاً ساختن یک BM25 قابل اجرا نبود؛ هدف، طراحی و ارزیابی یک Sparse Retrieval فارسی نسخه‌دار، قابل بازسازی، قابل ردیابی و قابل تحویل به Hybrid Retrieval بود.

## نتیجه اجرای نقشه ماژولار NB15

تمام مأموریت‌های رسمی نقشه ماژولار انجام شدند:

1. Analyzer فارسی با سیاست صریح Unicode و نرمال‌سازی حروف ساخته شد.
2. سیاست Tokenization، ارقام و نیم‌فاصله تعریف و آزمایش شد.
3. Sparse Index مبتنی بر BM25 روی تمام ۳۴۸۸ Chunk رسمی ساخته شد.
4. Document Frequency، Vocabulary و توزیع Tokenها کنترل شد.
5. Queryهای عددی، لاتین، دارای نیم‌فاصله، Rare-term و OOV بررسی شدند.
6. چند Analyzer و چند ترکیب `k1` و `b` فقط روی Development Split مقایسه شدند.
7. ارزیابی مستقل Sparse روی Test Split قفل‌شده انجام شد.
8. Error Analysis به تفکیک نوع منبع و نوع شکست تکمیل شد.
9. پیکربندی نهایی پیش از مشاهده Test Split فریز شد.
10. Index ذخیره‌شده از روی Artifactهای مستقل Reload و دوباره ارزیابی شد.
11. Manifest، Release، Acceptance و Handoff نسخه‌دار برای NB16 ساخته شدند.

## پیکربندی فریز‌شده

پیکربندی نهایی Sparse Retrieval به‌صورت زیر انتخاب و فریز شد:

- Analyzer: `fa_join_zwnj_stopwords_v1`
- Unicode normalization: `NFC`
- نرمال‌سازی حروف عربی و فارسی: فعال
- تبدیل رقم‌های فارسی و عربی به ASCII: فعال
- حذف Diacritic: فعال
- سیاست نیم‌فاصله: `JOIN`
- Stopword policy: `PERSIAN_STOPWORDS_V1`
- تعداد Stopwordها: ۲۸
- BM25 implementation: `local_robertson_bm25_v1`
- `k1 = 1.2`
- `b = 0.3`
- معیار اصلی انتخاب: `nDCG@10`

پیکربندی محافظه‌کارانه مبتنی بر High Document Frequency نیز بررسی شد، اما به‌دلیل افت معنادار nDCG@10 و MRR و عبورنکردن از Gateهای Exact-term رد شد.

## نتیجه ارزیابی مستقل Test

ارزیابی نهایی روی ۱۱۴۹ Query قفل‌شده Test و بدون هیچ‌گونه بازتنظیم انجام شد:

- `Hit@10 = 0.7076`
- `Hit@5 = 0.6519`
- `nDCG@10 = 0.4902`
- `nDCG@5 = 0.4688`
- `Recall@10 = 0.5536`
- `Recall@5 = 0.4936`
- `MRR = 0.5568`
- Median latency محلی: حدود `0.392 ms`
- P95 latency محلی: حدود `0.758 ms`

این Latencyها فقط مربوط به Benchmark محلی هستند و Production SLO محسوب نمی‌شوند.

## نتیجه Exact-term و Source Analysis

عملکرد Queryهای دارای عبارت‌های دقیق نیز بررسی شد:

- Numeric Queryها: `Hit@10 = 0.95`
- Latin Queryها: `Hit@10 = 1.00`
- Queryهای دارای نیم‌فاصله: `Hit@10 = 0.7298`
- Queryهای دارای OOV: `Hit@10 = 0.7188`

عملکرد به تفکیک منبع:

- Website: `nDCG@10 = 0.5886`
- Book: `nDCG@10 = 0.5573`
- Transcript: `nDCG@10 = 0.4150`

ضعف اصلی Sparse Retrieval در Transcriptها و Queryهای عمومی مشاهده شد. این محدودیت ثبت شده و باید در NB16 با ترکیب Dense و Sparse بررسی شود؛ پیکربندی Sparse براساس نتیجه Test تغییر نکرد.

## نتیجه Error Analysis

از ۱۱۴۹ Query مستقل Test:

- ۸۱۳ Query در Top-10 موفق بودند.
- ۳۱۷ Query دارای هم‌پوشانی واژگانی بودند، اما Relevant Chunk پایین‌تر از رتبه ۱۰ قرار گرفت.
- ۱۹ Query هیچ هم‌پوشانی واژگانی مؤثری با Gold Chunk نداشتند.

این ۱۹ مورد نشان می‌دهند که Sparse Retrieval به‌تنهایی برای تمام Queryهای معنایی کافی نیست و استفاده از Dense Retrieval در مرحله Hybrid ضروری است.

## نتیجه Artifact و Reload Validation

Sparse Index به Artifactهای مستقل و قابل‌حمل تبدیل شد:

- تعداد Documentها: ۳۴۸۸
- Vocabulary size: ۲۷۳۱۱
- Posting entry count: ۳۸۳۶۳۷
- Configuration SHA-256: `96d66676d20bacd764ec9a163995dd48266c90c2acea1c8a5f77d95d9efb6ecc`
- Logical Index SHA-256: `ce09bb4749da03c9dfa1a849de18fc8c490ce6b09eb34a30a96b9fedc478a8eb`

Reload Validation روی تمام ۱۱۴۹ Query انجام شد:

- File hash validation: PASS
- Document ID parity: PASS
- Document length parity: PASS
- Document Frequency parity: PASS
- Posting array parity: PASS
- Score equivalence: PASS
- Top-10 ranking parity: PASS
- Ranking mismatch count: صفر
- Metric parity: PASS
- Maximum metric delta: صفر
- وابستگی به Kernel جاری: ندارد

اختلاف بسیار کوچک عددی Scoreها پس از Reload با `numpy.allclose`، تلورانس `rtol=1e-6` و `atol=1e-8` کنترل شد. این اختلاف هیچ تغییری در Top-10 یا Metricها ایجاد نکرد.

## Artifactهای نهایی

- Manifest: `data/phase2/index/sparse_index_v1/sparse_index_manifest.json`
- Release: `artifacts/phase2/sparse_index/sparse_index_release_v1.json`
- NB16 Handoff: `artifacts/phase2/sparse_index/nb16_sparse_index_handoff_v1.json`
- Acceptance: `reports/phase2/sparse_index_acceptance_v1.json`

وضعیت Release:

`FROZEN_FOR_HYBRID_RETRIEVAL`

وضعیت Handoff:

`READY_FOR_NB16`

## وضعیت نهایی

- هدف رسمی NB15: محقق شد
- تمام Deliverableهای نقشه ماژولار: PASS
- Independent Test: PASS
- Error Analysis: PASS
- Artifact Reload: PASS
- Portable Path Validation: PASS
- سازگاری با Dense Index فریز‌شده NB14: PASS
- Dense Index تغییر نکرد
- Embedding مجدداً محاسبه نشد
- API پولی استفاده نشد
- وضعیت نهایی: `PASS_COMPLETE_FROZEN`

بنابراین Notebook 15 به‌طور کامل به هدف خود رسیده است. Sparse Index از این مرحله به بعد Frozen است و مراحل بعد نباید Analyzer، Stopwordها، پارامترهای BM25 یا محتوای Index را تغییر دهند.

مرحله بعدی پروژه:

`NB16 — Hybrid Retrieval`

NB16 باید Dense Release فریز‌شده NB14 و Sparse Handoff فریز‌شده NB15 را بارگذاری و Hashهای هر دو را پیش از اجرای RRF یا Weighted Fusion اعتبارسنجی کند.

</div>